In [ ]:
import asyncio
import os
from google import genai
from google.genai import types
# 1. Define a tool the model can call
def get_current_time():
    """Returns the current system time."""
    from datetime import datetime
    return {"time": datetime.now().strftime("%H:%M:%S")}

# 2. Setup Client
# Ensure you use 'v1alpha' for preview features
client = genai.Client(
    api_key="YOUR_API_KEY", 
    http_options={'api_version': 'v1alpha'}
)

MODEL_ID = "gemini-2.5-flash-native-audio-preview-12-2025"
# types.LiveServerContent()
async def main():
    # 3. Configure the session for audio + text + tools
    config = types.LiveConnectConfig(
        tools=[get_current_time],
        response_modalities=["AUDIO"], # Returns native audio
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        # This returns the text transcript of the model's audio response
        output_audio_transcription=types.AudioTranscriptionConfig()
    )

    async with client.aio.live.connect(model=MODEL_ID, config=config) as session:
        # await session.send_tool_response()
        # Send text input or audio bytes (e.g., from a mic stream)
        # To send audio, replace the string with audio bytes:
        # await session.send(audio_bytes, end_of_turn=True)
        await session.send_tool_response(function_responses=[types.FunctionResponse(name="get_current_time", id="123", response="12:34:56")])
        await session.send_realtime_input()
        await session.send_client_content(turns= types.Content(role="user", parts=[types.Part(text="What time is it?")]), turn_complete=True ,)

        async for message in session.receive():
            # A. HANDLE FUNCTION CALLS
            # message.tool_call.function_calls
            # message.server_content.model_turn.parts
            # message.server_content.output_transcription.


            if message.tool_call:
                if message.tool_call.function_calls:
                    for call in message.tool_call.function_calls:
                        print(f"Executing: {call.name}")
                        await session.send(
                            types.LiveClientToolResponse(
                                function_responses=[
                                    types.FunctionResponse(name=call.name, id=call.id, response=result)
                                ]
                            )
                        )

            # B. HANDLE TEXT & AUDIO OUTPUT
            if message.server_content and message.server_content.model_turn:
                for part in message.server_content.model_turn.parts:
                    # Capture Text Transcript
                    if part.text:
                        print(f"Model Transcript: {part.text}")
                    
                    # Capture Audio Data
                    if part.inline_data:
                        # part.inline_data.data contains the base64 audio bytes
                        print("Received audio chunk...")

if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
PRODUCTS = [{ "id": 1751, "sku": "WT08", "name": "Antonia Racer Tank", "short": "This is a variable product called a Antonia Racer Tank", "desc": "You won't know what you like best about the Antonia Racer Tank: soft, stretchy, lightweight fabric? Super-cute colorblocked details? Whatever it is, this piece is sure to quickly move to the top of your workout rotation. &bull; Machine wash.&bull; Line dry.", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 34.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/t/wt08-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/t/wt08-black_alt1.jpg", "stock": 138, "colors": [], "sizes": [], "type": "variable" }, { "id": 1639, "sku": "WT01", "name": "Bella Tank", "short": "This is a variable product called a Bella Tank", "desc": "Style, performance and comfort mix it up in the Bella Tank. With striking color block contrast, fitted form and a built-in bra, you'll be supported and stylish at the same time. &bull; Navy blue tank top - cotton.&bull; Feminine scoop neckline.&bull; Power mesh lining in shelf bra for superior suppo", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/t/wt01-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/t/wt01-blue_back.jpg", "stock": 39, "colors": [], "sizes": [], "type": "variable" }, { "id": 1767, "sku": "WT09", "name": "Breathe-Easy Tank", "short": "This is a variable product called a Breathe-Easy Tank", "desc": "The Breathe Easy Tank is so soft, lightweight, and comfortable, you won't even know it's there -- until its high-tech Cocona&reg; fabric starts wicking sweat away from your body to help you stay dry and focused. Layer it over your favorite sports bra and get moving. &bull; Machine wash/dry.&bull; Co", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 34.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/t/wt09-white_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/t/wt09-white_back.jpg", "stock": 161, "colors": [], "sizes": [], "type": "variable" }, { "id": 1591, "sku": "WB03", "name": "Celeste Sports Bra", "short": "This is a variable product called a Celeste Sports Bra", "desc": "Whatever your goals for the day's workout, the Celeste Sports Bra lets you do it in comfort and coolness, plus enhanced support and shaping. A power mesh back zone and moisture-wicking fabric ensure you stay dry. &bull; Mint bra top. &bull; Seam-free interior molded cups&bull; Odor control. &bull; U", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 39.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/b/wb03-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/b/wb03-green_alt1.jpg", "stock": 140, "colors": [], "sizes": [], "type": "variable" }, { "id": 1719, "sku": "WT06", "name": "Chloe Compete Tank", "short": "This is a variable product called a Chloe Compete Tank", "desc": "You've earned your figure, so stay cool and let it do all the talking in the form-fitted, sleek racerback Chloe Compete Tank. Designed for total range of motion and performance, the Nona is made with highly breathable mesh fabric. &bull; Royal blue tank top - nylon/spandex.&bull; Flatlock stitching.", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 39.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/t/wt06-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/t/wt06-blue_back.jpg", "stock": 157, "colors": [], "sizes": [], "type": "variable" }, { "id": 1559, "sku": "WB01", "name": "Electra Bra Top", "short": "This is a variable product called a Electra Bra Top", "desc": "A heavenly soft and stylish eco garment, the Electra Bra Top is perfect for wearing on its own as a hot yoga top or layered under a tank. &bull; Gray rouched bra top.&bull; Attractive back straps feature contrasting motif fabric.&bull; Interior bra top is lined with breathable mesh.&bull; Elastic un", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 39.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/b/wb01-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/b/wb01-gray_alt1.jpg", "stock": 56, "colors": [], "sizes": [], "type": "variable" }, { "id": 1575, "sku": "WB02", "name": "Erica Evercool Sports Bra", "short": "This is a variable product called a Erica Evercool Sports Bra", "desc": "Perfect for medium-impact workouts, our Erica EverCool&trade; Sports Bra brings a brilliant combo of comfort and style. Moisture-wicking technology keeps you dry, and the flattering hybrid racerback promises an unbeatable range of motion. &bull; Honeycomb light blue bra top.&bull; Elastic hem.&bull;", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 39.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/b/wb02-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/b/wb02-blue_alt1.jpg", "stock": 203, "colors": [], "sizes": [], "type": "variable" }, { "id": 1703, "sku": "WT05", "name": "Leah Yoga Top", "short": "This is a variable product called a Leah Yoga Top", "desc": "The Leah Yoga Top offers a practical, comfortable upper that will not compromise your style. Body hugging fit and interior shelf bra make it suitable for active or leisure pursuits. &bull; Blue heather rouched tank top.&bull; Camisole tank top.&bull; Banding and shirring details.&bull; Body hugging ", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 39.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/t/wt05-purple_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/t/wt05-purple_alt1.jpg", "stock": 137, "colors": [], "sizes": [], "type": "variable" }, { "id": 1623, "sku": "WB05", "name": "Lucia Cross-Fit Bra", "short": "This is a variable product called a Lucia Cross-Fit Bra", "desc": "Being cool is a big part of being comfy, which is why the Lucia Cross-Fit Bra features moisture-wicking technology as well as soft, lightweight fabric. &bull; Black/white bra top. &bull; Criss-cross back design. &bull; Machine wash/dry.", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 39.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/b/wb05-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/b/wb05-black_back.jpg", "stock": 43, "colors": [], "sizes": [], "type": "variable" }, { "id": 1735, "sku": "WT07", "name": "Maya Tunic", "short": "This is a variable product called a Maya Tunic", "desc": "With abutted seams and moisture wicking capacity, the Maya Tunic lets you work out in complete comfort. &bull; Mint green heather tunic-style tank. &bull; Wrapped back with cut out detail.&bull; Drawcord detail at end.&bull; Abutted seams.", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/t/wt07-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/t/wt07-green_alt1.jpg", "stock": 64, "colors": [], "sizes": [], "type": "variable" }, { "id": 1687, "sku": "WT04", "name": "Nona Fitness Tank", "short": "This is a variable product called a Nona Fitness Tank", "desc": "It doesn't matter if your goal is 5 miles or an hour of Vinyasa, because our Nona Fitness Tank does it all. You don't have to sacrifice either -- this v-neck top features smooth, chafe-free seams and a breathable mesh back insert. Cute, too. &bull; Blue/white striped mesh tank.&bull; Relaxed fit. &b", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 39.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/t/wt04-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/t/wt04-blue_back.jpg", "stock": 58, "colors": [], "sizes": [], "type": "variable" }, { "id": 1671, "sku": "WT03", "name": "Nora Practice Tank", "short": "This is a variable product called a Nora Practice Tank", "desc": "A closet go-to, the Nora Practice Tank can be worn at or after workouts or beneath a baggy tee for a laid-back style. Perfect fit and very comfortable! &bull; Pink stripped tank with side rouching.&bull; Pre-shrunk.&bull; Garment dyed.&bull; 92% Organic Cotton/8% Lycra.", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 39.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/t/wt03-red_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/t/wt03-red_alt1.jpg", "stock": 77, "colors": [], "sizes": [], "type": "variable" }, { "id": 1607, "sku": "WB04", "name": "Prima Compete Bra Top", "short": "This is a variable product called a Prima Compete Bra Top", "desc": "Pull on the Prima Compete Bra Top and you'll love the fabric: soft, stretchy, and ultra lightweight. But you'll also love the racerback cut, for freer movement through all your athletic feats. `&bull; Colorblocked details.&bull; Machine wash/line dry.", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 24.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/b/wb04-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/b/wb04-blue-0.jpg", "stock": 9, "colors": [], "sizes": [], "type": "variable" }, { "id": 1655, "sku": "WT02", "name": "Zoe Tank", "short": "This is a variable product called a Zoe Tank", "desc": "The Zoe Tank leads with eye-catching fitness-pop looks, keeping your studio style uptrend. The inner shelf bra and soft, breathable fabric create a tank top that looks, feels and fits great. &bull; Salmon heather tank top.&bull; 1\" elastic band on inner bra.&bull; Mesh lining on shelf bra for suppor", "category": "Bras & Tanks", "top": "Clothing", "gender": "women", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/t/wt02-orange_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/t/wt02-orange_back.jpg", "stock": 152, "colors": [], "sizes": [], "type": "variable" }, { "id": 145, "sku": "MH09", "name": "Abominable Hoodie", "short": "This is a variable product called a Abominable Hoodie", "desc": "It took CoolTech&trade; weather apparel know-how and lots of wind-resistant fabric to get the Abominable Hoodie just right. It's aggressively warm when it needs to be, while maintaining your comfort in milder climes. &bull; Blue heather hoodie.&bull; Relaxed fit. &bull; Moisture-wicking. &bull; Mach", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 69.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh09-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh09-blue_alt1.jpg", "stock": 189, "colors": [], "sizes": [], "type": "variable" }, { "id": 193, "sku": "MH12", "name": "Ajax Full-Zip Sweatshirt", "short": "This is a variable product called a Ajax Full-Zip Sweatshirt", "desc": "The Ajax Full-Zip Sweatshirt makes the optimal layering or outer piece for archers, golfers, hikers and virtually any other sportsmen. Not only does it have top-notch moisture-wicking abilities, but the tight-weave fabric also prevents pilling from repeated wash-and-wear cycles. &bull; Mint striped ", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 69.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh12-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh12-green_alt1.jpg", "stock": 139, "colors": [], "sizes": [], "type": "variable" }, { "id": 1117, "sku": "WH09", "name": "Ariel Roll Sleeve Sweatshirt", "short": "This is a variable product called a Ariel Roll Sleeve Sweatshirt", "desc": "Soft, sleek and subtle, the Ariel Roll Sleeve Sweatshirt is a nuanced fitness garment for all occasions. It works equally well as a workout piece or in a casual social setting. &bull; Purple two-tone lightweight hoodie. &bull; 100% cotton.&bull; Adjustable roll sleeves for Long-Sleeve or 3/4 sleeve.", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "women", "price": 39.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/h/wh09-purple_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/h/wh09-purple_alt1.jpg", "stock": 209, "colors": [], "sizes": [], "type": "variable" }, { "id": 1031, "sku": "WH03", "name": "Autumn Pullie", "short": "This is a variable product called a Autumn Pullie", "desc": "With ultra-soft fleece fabric and an athletic design, our short-sleeve Autumn Pullie delivers a comfortable fit that makes it an everyday essential. A luxurious roll neck protects you from elements. &bull; Cayenne Short-Sleeve roll neck sweatshirt. &bull; Relaxed fit. &bull; Short-Sleeves. &bull; Ma", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "women", "price": 57.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/h/wh03-red_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/h/wh03-red_alt1.jpg", "stock": 139, "colors": [], "sizes": [], "type": "variable" }, { "id": 49, "sku": "MH03", "name": "Bruno Compete Hoodie", "short": "This is a variable product called a Bruno Compete Hoodie", "desc": "Stay comfortable and stay in the race no matter what the weather's up to. The Bruno Compete Hoodie's water-repellent exterior shields you from the elements, while advanced fabric technology inside wicks moisture to keep you dry. &bull; Full zip black hoodie pullover. &bull; Adjustable drawstring hoo", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 63.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh03-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh03-black_alt1.jpg", "stock": 167, "colors": [], "sizes": [], "type": "variable" }, { "id": 1101, "sku": "WH08", "name": "Cassia Funnel Sweatshirt", "short": "This is a variable product called a Cassia Funnel Sweatshirt", "desc": "The casual Cassia Funnel Sweatshirt sports front angled pockets and a drawstring stretch funnel hoodie. A cowl front neck pairs with a tagless label at back neck. &bull; White full zip hoodie with gray detail.&bull; 65% Cotton/28% Nylon/7% Spandex.&bull; Front slash pockets.&bull; Tagless label at b", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "women", "price": 48.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/h/wh08-white_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/h/wh08-white_alt1.jpg", "stock": 5, "colors": [], "sizes": [], "type": "variable" }, { "id": 17, "sku": "MH01", "name": "Chaz Kangeroo Hoodie", "short": "This is a variable product called a Chaz Kangeroo Hoodie", "desc": "Ideal for cold-weather training or work outdoors, the Chaz Hoodie promises superior warmth with every wear. Thick material blocks out the wind as ribbed cuffs and bottom band seal in body heat. &bull; Two-tone gray heather hoodie.&bull; Drawstring-adjustable hood. &bull; Machine wash/dry.", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 52.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh01-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh01-gray_alt1.jpg", "stock": 149, "colors": [], "sizes": [], "type": "variable" }, { "id": 1165, "sku": "WH12", "name": "Circe Hooded Ice Fleece", "short": "This is a variable product called a Circe Hooded Ice Fleece", "desc": "Keep shivers at bay with the Circe Hooded Ice Fleece. Ultra-thick, high-pile fleece traps your body heat and keeps cold outside. The drawstring hood and ribbed details add extra coziness and touches of classic style. Its relaxed fit is loose enough to layer comfortably over your shirt. Full-zip fron", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "women", "price": 68.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/h/wh12-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/h/wh12-gray_back.jpg", "stock": 110, "colors": [], "sizes": [], "type": "variable" }, { "id": 1069, "sku": "WH06", "name": "Daphne Full-Zip Hoodie", "short": "This is a variable product called a Daphne Full-Zip Hoodie", "desc": "The Daphne Hoodie is an attractive yet rugged layering or outer piece, especially for chilly days en route the studio, doing yardwork or enjoying outdoor recreation. &bull; Purple full zip hoodie with pink accents. &bull; Heather texture.&bull; 4-way stretch.&bull; Pre-shrunk.&bull; Hood lined in ve", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "women", "price": 59.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/h/wh06-purple_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/h/wh06-purple_alt1.jpg", "stock": 9, "colors": [], "sizes": [], "type": "variable" }, { "id": 1149, "sku": "WH11", "name": "Eos V-Neck Hoodie", "short": "This is a variable product called a Eos V-Neck Hoodie", "desc": "Getting chilly before class starts? Wear the Eos on your way to and from yoga for a cute and cozy warmup piece. Reach for its reliable comfort and enjoy a super-soft blend of fabrics finished in sporty style that includes a hidden kangaroo pocket. Semi-fitted. Long-Sleeve. Machine wash/line dry.", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "women", "price": 54.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/h/wh11-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/h/wh11-blue_back.jpg", "stock": 169, "colors": [], "sizes": [], "type": "variable" }, { "id": 65, "sku": "MH04", "name": "Frankie  Sweatshirt", "short": "This is a variable product called a Frankie Sweatshirt", "desc": "The Frankie Sweatshirt is your best friend at long afternoon stadium stints or winter trailside campsites. The soft fleece fabric keeps you toasty as moisture-wicking technology kicks in when the sun comes out. &bull; Light green crewneck sweatshirt.&bull; Hand pockets.&bull; Relaxed fit.&bull; Mach", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 60.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh04-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh04-green_alt1.jpg", "stock": 42, "colors": [], "sizes": [], "type": "variable" }, { "id": 177, "sku": "MH11", "name": "Grayson Crewneck Sweatshirt", "short": "This is a variable product called a Grayson Crewneck Sweatshirt", "desc": "The Grayson Crewneck Sweatshirt gives you that ageless, classic look &ndash; a style that comes back around nearly every season. What's more, its performance and temp-control values are always in vogue. &bull; Cream crewneck sweatshirt with black accents.&bull; 80% cotton/20% polyester fleece. &bull", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 64.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh11-white_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh11-white_alt1.jpg", "stock": 214, "colors": [], "sizes": [], "type": "variable" }, { "id": 1133, "sku": "WH10", "name": "Helena Hooded Fleece", "short": "This is a variable product called a Helena Hooded Fleece", "desc": "Wear this hoodie after the gym or before a chilly weather bike ride. Either way, this versatile sweatshirt offers an effortlessly appealing silhouette and a super-comfy fit. Smooth, stretchy fabric creates flattering shape, while the full-zip placket and hood help block gusty winds. Full zip. Banded", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "women", "price": 55.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/h/wh10-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/h/wh10-gray_alt1.jpg", "stock": 190, "colors": [], "sizes": [], "type": "variable" }, { "id": 1015, "sku": "WH02", "name": "Hera Pullover Hoodie", "short": "This is a variable product called a Hera Pullover Hoodie", "desc": "Get ready to rule the studio and dominate the yoga mat in the Hera Pullover Hoodie, a cozy yet classy look for any level of yogi. &bull; Teal with purple stiching.&bull; Hoodie pullover.&bull; Snug fit.", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "women", "price": 48.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/h/wh02-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/h/wh02-blue_alt1.jpg", "stock": 81, "colors": [], "sizes": [], "type": "variable" }, { "id": 113, "sku": "MH07", "name": "Hero Hoodie", "short": "This is a variable product called a Hero Hoodie", "desc": "Gray and black color blocking sets you apart as the Hero Hoodie keeps you warm on the bus, campus or cold mean streets. Slanted outsize front pockets keep your style real . . . convenient. &bull; Full-zip gray and black hoodie.&bull; Ribbed hem.&bull; Standard fit.&bull; Drawcord hood cinch.&bull; W", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 54.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh07-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh07-gray_alt1.jpg", "stock": 162, "colors": [], "sizes": [], "type": "variable" }, { "id": 81, "sku": "MH05", "name": "Hollister Backyard Sweatshirt", "short": "This is a variable product called a Hollister Backyard Sweatshirt", "desc": "Kick off your weekend in the Hollister Backyard Sweatshirt. Whether you're raking leaves or flipping burgers, this comfy layer blocks the bite of the crisp autumn air. Puffy thick from hood to hem, it traps heat against your core. &bull; Cream crewneck sweatshirt with navy sleeves/trim.&bull; Relaxe", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 52.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh05-white_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh05-white_alt1.jpg", "stock": 75, "colors": [], "sizes": [], "type": "variable" }, { "id": 161, "sku": "MH10", "name": "Mach Street Sweatshirt", "short": "This is a variable product called a Mach Street Sweatshirt", "desc": "From hard streets to asphalt track, the Mach Street Sweatshirt holds up to wear and wind and rain. An infusion of performance and stylish comfort, with moisture-wicking LumaTech&trade; fabric, it's bound to become an everyday part of your active lifestyle &bull; Navy heather crewneck sweatshirt.&bul", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 62.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh10-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh10-blue_alt1.jpg", "stock": 145, "colors": [], "sizes": [], "type": "variable" }, { "id": 209, "sku": "MH13", "name": "Marco Lightweight Active Hoodie", "short": "This is a variable product called a Marco Lightweight Active Hoodie", "desc": "For cold-weather training or post-game layering, you need something more than a basic fleece. Our Marco Lightweight Active Hoodie brings both style and performance to the plate, court, or touchline. The smooth-faced, brushed-back fabric blocks wind and traps body heat, while integrated Cocona&reg; f", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 74.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh13-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh13-blue_alt1.jpg", "stock": 108, "colors": [], "sizes": [], "type": "variable" }, { "id": 1047, "sku": "WH04", "name": "Miko Pullover Hoodie", "short": "This is a variable product called a Miko Pullover Hoodie", "desc": "After quality gym time, put on the Miko Pullover Hoody and keep your body warm. You'll find it's fashionable enough for the streets, but comfy enough to relax in at home. &bull; Teal two-tone hoodie.&bull; Low scoop neckline.&bull; Adjustable hood drawstrings.&bull; Longer rounded hemline for extra ", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "women", "price": 69.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/h/wh04-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/h/wh04-blue_alt1.jpg", "stock": 24, "colors": [], "sizes": [], "type": "variable" }, { "id": 999, "sku": "WH01", "name": "Mona Pullover Hoodlie", "short": "This is a variable product called a Mona Pullover Hoodlie", "desc": "Whether you're after energizing activity or eye-catching apparel, the Mona Pullover is what you want. You'll stay warm and look fashionable, wherever you are. &bull; Light green heathered hoodie. &bull; Long-Sleeve, pullover. &bull; Long elliptical hem for extra coverage. &bull; Deep button placket ", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "women", "price": 57.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/h/wh01-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/h/wh01-green_alt1.jpg", "stock": 29, "colors": [], "sizes": [], "type": "variable" }, { "id": 129, "sku": "MH08", "name": "Oslo Trek Hoodie", "short": "This is a variable product called a Oslo Trek Hoodie", "desc": "Chilly weather is just an excuse to throw on your toasty, handsome new Oslo Trek Hoodie. It features an adjustable drawstring hood and a kangaroo pocket for extra hand warmth. The ultra-soft, cozy lining will have you wishing for more brisk days. &bull; Brown hoodie with black detail.&bull; Pullover", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 42.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh08-brown_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh08-brown_alt1.jpg", "stock": 144, "colors": [], "sizes": [], "type": "variable" }, { "id": 1085, "sku": "WH07", "name": "Phoebe Zipper Sweatshirt", "short": "This is a variable product called a Phoebe Zipper Sweatshirt", "desc": "A sophisticated layer of warmth awaits you in our full-zip sweatshirt jacket. You'll reach for this one in any season to enjoy its sturdy exterior and plush interior. &bull; Gray full zip hoodie with yellow detail.&bull; Hand-warmer pockets. &bull; Zip MP3 pocket with outlet for earphones wire. &bul", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "women", "price": 59.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/h/wh07-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/h/wh07-gray_alt1.jpg", "stock": 181, "colors": [], "sizes": [], "type": "variable" }, { "id": 1063, "sku": "WH05", "name": "Selene Yoga Hoodie", "short": "This is a variable product called a Selene Yoga Hoodie", "desc": "The Selene Yoga Hoodie gets you to and from the studio in semi form-fitted comfort. Snug, sleek contours add fashion to function in this free-moving yoga warm up sweatshirt. &bull; Ivory heather full zip 3/4 sleeve hoodie.&bull; Zip pocket at arm for convenient storage.&bull; 24.0\" body length.&bull", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "women", "price": 42.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/h/wh05-white_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/h/wh05-white_back.jpg", "stock": 74, "colors": [], "sizes": [], "type": "variable" }, { "id": 97, "sku": "MH06", "name": "Stark Fundamental Hoodie", "short": "This is a variable product called a Stark Fundamental Hoodie", "desc": "You don't need bells and whistles when performance speaks for itself. The full-zip Stark Fundamental Hoodie give just what you need. Hood and fleece lining keep you warm, while breathable fabric and wicking technology won't let you overheat. &bull; Navy specked full zip hoodie.&bull; Ribbed cuffs, b", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 42.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh06-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh06-blue_alt1.jpg", "stock": 146, "colors": [], "sizes": [], "type": "variable" }, { "id": 33, "sku": "MH02", "name": "Teton Pullover Hoodie", "short": "This is a variable product called a Teton Pullover Hoodie", "desc": "This Teton Pullover Hoodie gives you more than laid-back style. It's equipped with moisture-wicking fabric to keep light and dry inside, especially in chilly-weather workouts. An elasticized hem lets you move about freely. &bull; Black pullover hoodie.&bull; Soft, brushed interior. &bull; Front hand", "category": "Hoodies & Sweatshirts", "top": "Clothing", "gender": "men", "price": 70.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/h/mh02-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/h/mh02-black_alt1.jpg", "stock": 208, "colors": [], "sizes": [], "type": "variable" }, { "id": 1271, "sku": "WJ08", "name": "Adrienne Trek Jacket", "short": "This is a variable product called a Adrienne Trek Jacket", "desc": "You're ready for a cross-country jog or a coffee on the patio in the Adrienne Trek Jacket. Its style is unique with stand collar and drawstrings, and it fits like a jacket should. &bull; gray 1/4 zip pullover.&bull; Comfortable, relaxed fit.&bull; Front zip for venting.&bull; Spacious, kangaroo pock", "category": "Jackets", "top": "Clothing", "gender": "women", "price": 57.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/j/wj08-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/j/wj08-gray_alt1.jpg", "stock": 174, "colors": [], "sizes": [], "type": "variable" }, { "id": 1207, "sku": "WJ03", "name": "Augusta Pullover Jacket", "short": "This is a variable product called a Augusta Pullover Jacket", "desc": "It's hard to be uncomfortable in the Augusta Pullover Jacket with &frac14; zip. With an incredibly soft fleece lining and textured outer fabric, it offers reliable protection from the elements and a cozy fit as well. &bull; Pink half-zip pullover. &bull; Front pouch pockets. &bull; Fold-down collar.", "category": "Jackets", "top": "Clothing", "gender": "women", "price": 57.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/j/wj03-red_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/j/wj03-red_alt1.jpg", "stock": 182, "colors": [], "sizes": [], "type": "variable" }, { "id": 225, "sku": "MJ01", "name": "Beaumont Summit Kit", "short": "This is a variable product called a Beaumont Summit Kit", "desc": "The smooth nylon shell around the Beaumont Summit Kit combats wind, reinforced with a cold-fighting brushed fleece layer. The jacket is reversible, giving you a new look for the return trek. Ample pocket space rounds out this hiker's paradise package. &bull; Yellow full zip rain jacket.&bull; Full-z", "category": "Jackets", "top": "Clothing", "gender": "men", "price": 42.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/j/mj01-yellow_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/j/mj01-yellow_alt1.jpg", "stock": 61, "colors": [], "sizes": [], "type": "variable" }, { "id": 241, "sku": "MJ02", "name": "Hyperion Elements Jacket", "short": "This is a variable product called a Hyperion Elements Jacket", "desc": "Boldly face high winds, frigid temps and stormy weather the whole winter through in the Hyperion Elements Jacket. LumaTech&trade; insulating technology helps maintain your core temperature and wick sweat. The smooth shell is water repellent and quilted to retain body heat. &bull; Lime 1/4 zip pullov", "category": "Jackets", "top": "Clothing", "gender": "men", "price": 51.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/j/mj02-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/j/mj02-green_alt1.jpg", "stock": 202, "colors": [], "sizes": [], "type": "variable" }, { "id": 1255, "sku": "WJ07", "name": "Inez Full Zip Jacket", "short": "This is a variable product called a Inez Full Zip Jacket", "desc": "The Inez Full Zip jacket is more than a cute layer. It's a full-on tech wonder you'll use for outdoor, pre and post workout. You'll love how the highlighted zipper and sleeve safety trim protect and catch looks. &bull; Purple heather inset full zip jacket.&bull; Full zip hoodie.&bull; Contrast bindi", "category": "Jackets", "top": "Clothing", "gender": "women", "price": 59.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/j/wj07-purple_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/j/wj07-purple_alt1.jpg", "stock": 244, "colors": [], "sizes": [], "type": "variable" }, { "id": 1223, "sku": "WJ04", "name": "Ingrid Running Jacket", "short": "This is a variable product called a Ingrid Running Jacket", "desc": "The Ingrid Running Jacket combines sleek design and high performance with slim, contoured fit and moisture-wicking fabric. It features a full-zip construction and a collared neck to keep the elements out and body heat in. &bull; Slim fit. &bull; Moisture-wicking fabric. &bull; Two side pockets. &bul", "category": "Jackets", "top": "Clothing", "gender": "women", "price": 84.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/j/wj04-white_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/j/wj04-white_alt1.jpg", "stock": 28, "colors": [], "sizes": [], "type": "variable" }, { "id": 1287, "sku": "WJ09", "name": "Jade Yoga Jacket", "short": "This is a variable product called a Jade Yoga Jacket", "desc": "If only all your other jackets were as comfortable as the relaxed-fit Jade Yoga Jacket, perfect for use during stretching, biking to and from studio or strolling on breezy fall days. &bull; Seafoam 1/4 zip pullover with purple stitching.&bull; Lightweight, quick-drying, water-resistant construction.", "category": "Jackets", "top": "Clothing", "gender": "women", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/j/wj09-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/j/wj09-green_alt1.jpg", "stock": 143, "colors": [], "sizes": [], "type": "variable" }, { "id": 1191, "sku": "WJ02", "name": "Josie Yoga Jacket", "short": "This is a variable product called a Josie Yoga Jacket", "desc": "When your near future includes yoga, the cozy comfort of the Josie Yoga Jacket gets your mind and body ready. Stretchy CoolTech&trade; fabric with zipper pockets makes this jacket the right gear for studio time or teatime after. &bull; Slate rouched neck pullover.&bull; Moisture-wicking fabric.&bull", "category": "Jackets", "top": "Clothing", "gender": "women", "price": 56.25, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/j/wj02-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/j/wj02-gray_alt1.jpg", "stock": 177, "colors": [], "sizes": [], "type": "variable" }, { "id": 1335, "sku": "WJ06", "name": "Juno Jacket", "short": "This is a variable product called a Juno Jacket", "desc": "On colder-than-comfortable mornings, you'll love warming up in the Juno All-Ways Performanc Jacket, designed to compete with wind and chill. Built-in Cocona&reg; technology aids evaporation, while a special zip placket and stand-up collar keep your neck protected. &bull; Adjustable hood.&bull; Fleec", "category": "Jackets", "top": "Clothing", "gender": "women", "price": 77.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/j/wj06-purple_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/j/wj06-purple_alt1.jpg", "stock": 39, "colors": [], "sizes": [], "type": "variable" }, { "id": 353, "sku": "MJ06", "name": "Jupiter All-Weather Trainer", "short": "This is a variable product called a Jupiter All-Weather Trainer", "desc": "Inclement climate be damned. With your breathable, water-resistant Jupiter All-Weather Trainer, you can focus on fuel and form and leave protection and comfort to us. The fabric is a special light fleece that wicks moisture and insulates at once. &bull; Relaxed fit. &bull; Hand pockets. &bull; Machi", "category": "Jackets", "top": "Clothing", "gender": "men", "price": 56.99, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/j/mj06-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/j/mj06-blue_alt1.jpg", "stock": 91, "colors": [], "sizes": [], "type": "variable" }, { "id": 257, "sku": "MJ04", "name": "Kenobi Trail Jacket", "short": "This is a variable product called a Kenobi Trail Jacket", "desc": "Aside from sealed seams to keep moisture out and body heat in, the Kenobi Trail Vest is all about media convenience. Use your phone and music player without ever taking it out of its cozy compartment: an attached remote on the left chest lets you change songs, pause and raise/lower volume. &bull; Bl", "category": "Jackets", "top": "Clothing", "gender": "men", "price": 47.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/j/mj04-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/j/mj04-black_alt1.jpg", "stock": 201, "colors": [], "sizes": [], "type": "variable" }, { "id": 289, "sku": "MJ08", "name": "Lando Gym Jacket", "short": "This is a variable product called a Lando Gym Jacket", "desc": "The Lando Gym Jacket offers strategic ventilation at perspiration-prone areas, while moisture-wicking technology helps you stay dry. Side-seam pockets house your favorite media device, so you're plugged in when working out. &bull; Gray polyester/spandex full zip jacket with orange lining. &bull; Rig", "category": "Jackets", "top": "Clothing", "gender": "men", "price": 99.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/j/mj08-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/j/mj08-gray_alt1.jpg", "stock": 220, "colors": [], "sizes": [], "type": "variable" }, { "id": 321, "sku": "MJ10", "name": "Mars HeatTech&trade; Pullover", "short": "This is a variable product called a Mars HeatTech&trade; Pullover", "desc": "The Mars HeatTech&trade; Jacket defense against winter climes so you can play offense in the park, on the trail or in the deck chair. Great for the ski lodge or stadium, it features a wind- and water-resistant outer shell with a draw-cord hood and deep pockets for storage. &bull; Red 1/4 zip pullove", "category": "Jackets", "top": "Clothing", "gender": "men", "price": 66.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/j/mj10-red_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/j/mj10-red_alt1.jpg", "stock": 113, "colors": [], "sizes": [], "type": "variable" }, { "id": 369, "sku": "MJ03", "name": "Montana Wind Jacket", "short": "This is a variable product called a Montana Wind Jacket", "desc": "Light-as-a-feather wind protection for runners, walkers and outdoor fitness buffs, the Montana Wind Jacket can be stuffed into your pocket for portable protection. Its stylish, move-with-you design makes it especially versatile. Adjustable hood. Split pocket. Thumb holes. Machine wash/hang to dry.", "category": "Jackets", "top": "Clothing", "gender": "men", "price": 49.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/j/mj03-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/j/mj03-black_alt1.jpg", "stock": 44, "colors": [], "sizes": [], "type": "variable" }, { "id": 1303, "sku": "WJ10", "name": "Nadia Elements Shell", "short": "This is a variable product called a Nadia Elements Shell", "desc": "Protect yourself from wind and rain in the stylish Nadia Elements Shell. It repels water using hydro-resistant fabric, with fleece lining that adds a touch of warmth. It's finished with bold contrast zippers, adjustable cuffs and a hood. &bull; Zippered front. &bull; Zippered side pockets. &bull; Dr", "category": "Jackets", "top": "Clothing", "gender": "women", "price": 69.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/j/wj10-yellow_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/j/wj10-yellow_alt1.jpg", "stock": 201, "colors": [], "sizes": [], "type": "variable" }, { "id": 1319, "sku": "WJ11", "name": "Neve Studio Dance Jacket", "short": "This is a variable product called a Neve Studio Dance Jacket", "desc": "If you're constantly on the move, the Neve Studio Dance Jacket is for you. It's not just for dance, either, with a tight fit that works as a mid-layer. The reversible design makes it even more versatile. &bull; Bright blue 1/4 zip pullover.&bull; CoolTech&trade; liner is sweat-wicking.&bull; Sleeve ", "category": "Jackets", "top": "Clothing", "gender": "women", "price": 69.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/j/wj11-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/j/wj11-blue_alt1.jpg", "stock": 94, "colors": [], "sizes": [], "type": "variable" }, { "id": 1351, "sku": "WJ12", "name": "Olivia 1/4 Zip Light Jacket", "short": "This is a variable product called a Olivia 1/4 Zip Light Jacket", "desc": "Running errands or headed to the gym, you just want to be comfortable. The Olivia Light Jacket promises that, plus a laid-back look. This zip-up is designed with shoulder stripes for an athletic touch, and banded waist and contoured seams for a flattering silhouette. &bull; Loose fit.&bull; Reflecti", "category": "Jackets", "top": "Clothing", "gender": "women", "price": 77.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/j/wj12-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/j/wj12-blue_alt1.jpg", "stock": 191, "colors": [], "sizes": [], "type": "variable" }, { "id": 273, "sku": "MJ07", "name": "Orion Two-Tone Fitted Jacket", "short": "This is a variable product called a Orion Two-Tone Fitted Jacket", "desc": "While you're getting fit, you need a fitted jacket to match. Striking color blocking patterns on hood, shoulders and arms are hallmarks of the Orion Two-Tone Fitted Jacket. They provide eye-catching contrast against the rich, torso tones of this 100% polyester, moisture-wicking essential. &bull; Red", "category": "Jackets", "top": "Clothing", "gender": "men", "price": 72.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/j/mj07-red_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/j/mj07-red_alt1.jpg", "stock": 123, "colors": [], "sizes": [], "type": "variable" }, { "id": 385, "sku": "MJ12", "name": "Proteus Fitness Jackshirt", "short": "This is a variable product called a Proteus Fitness Jackshirt", "desc": "Part jacket, part shirt, the Proteus Fitness Jackshirt makes an ideal companion for outdoor training, camping or loafing on crisp days. Natural Cocona&reg; technology brings breathable comfort and increased dryness along with UV protection and odor management. The drop-tail hem provides extra covera", "category": "Jackets", "top": "Clothing", "gender": "men", "price": 45.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/j/mj12-orange_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/j/mj12-orange_alt1.jpg", "stock": 248, "colors": [], "sizes": [], "type": "variable" }, { "id": 1239, "sku": "WJ05", "name": "Riona Full Zip Jacket", "short": "This is a variable product called a Riona Full Zip Jacket", "desc": "The Riona Basic Zip Jacket makes the perfect extra layer for cold-weather workouts. It features amazing breathability and moisture management, but full-length zipper lets you moderate your core temperature even more. &bull; Brown heather full zip rouched jacket.&bull; Side hand pockets for extra sto", "category": "Jackets", "top": "Clothing", "gender": "women", "price": 60.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/j/wj05-brown_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/j/wj05-brown_alt1.jpg", "stock": 203, "colors": [], "sizes": [], "type": "variable" }, { "id": 1175, "sku": "WJ01", "name": "Stellar Solar Jacket", "short": "This is a variable product called a Stellar Solar Jacket", "desc": "Beat the heat and protect yourself from sunrays with the Stellar Solar Jacket. It's loaded with all the engineered features you need for an intense, safe outdoor workout: 100% UV protection, a breathable perforated construction, and advanced moisture-wicking technology. &bull; Loose fit. &bull; Refl", "category": "Jackets", "top": "Clothing", "gender": "women", "price": 75.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/j/wj01-red_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/j/wj01-red_alt1.jpg", "stock": 189, "colors": [], "sizes": [], "type": "variable" }, { "id": 305, "sku": "MJ09", "name": "Taurus Elements Shell", "short": "This is a variable product called a Taurus Elements Shell", "desc": "What's a little rain or snow when you're inside the Taurus Elements Shell? This specially engineered Cocona&reg; jacket lets you enjoy the great outdoors and brave the elements, thanks to the all-waterproof, breathable exterior. &bull; Yellow 1/4 zip pullover. &bull; Two chest pockets.&bull; Standar", "category": "Jackets", "top": "Clothing", "gender": "men", "price": 65.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/j/mj09-yellow_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/j/mj09-yellow_alt1.jpg", "stock": 181, "colors": [], "sizes": [], "type": "variable" }, { "id": 337, "sku": "MJ11", "name": "Typhon Performance Fleece-lined Jacket", "short": "This is a variable product called a Typhon Performance Fleece-lined Jacket", "desc": "Ironmen and couch warriors both reach for the Typhon Performance Fleece-lined Jacket. After all, no man can resist ultra-soft microfleece lining. Flatlock seams make it ideal for wearing over everything from tanks and tees to high-tech base layers. &bull; Black full-zip flight jacket.&bull; Cocona&r", "category": "Jackets", "top": "Clothing", "gender": "men", "price": 60.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/j/mj11-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/j/mj11-black_alt1.jpg", "stock": 28, "colors": [], "sizes": [], "type": "variable" }, { "id": 1816, "sku": "WP07", "name": "Aeon Capri", "short": "This is a variable product called a Aeon Capri", "desc": "Reach for the stars and beyond in these Aeon Capri pant. With a soft, comfortable feel and moisture wicking fabric, these duo-tone leggings are easy to wear -- and wear attractively. &bull; Black capris with teal accents.&bull; Thick, 3\" flattering waistband.&bull; Media pocket on inner waistband.&b", "category": "Pants", "top": "Clothing", "gender": "women", "price": 48.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp07-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp07-black_alt1.jpg", "stock": 157, "colors": [], "sizes": [], "type": "variable" }, { "id": 822, "sku": "MP11", "name": "Aether Gym Pant", "short": "This is a variable product called a Aether Gym Pant", "desc": "The Aether Gym Pant is built for the studio, but adapts perfectly well to outdoor and gym environs too. With lightweight stretch fabric and water-repellent exterior, the Aether is ready for all comers. &bull; Pants/shorts convertible.&bull; Lightweight moisture wicking. &bull; Water repellent.&bull;", "category": "Pants", "top": "Clothing", "gender": "men", "price": 74.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/p/mp11-brown_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/p/mp11-brown_alt1.jpg", "stock": 238, "colors": [], "sizes": [], "type": "variable" }, { "id": 1823, "sku": "WP08", "name": "Bardot Capri", "short": "This is a variable product called a Bardot Capri", "desc": "Black is back &mdash; was it ever gone? &mdash; which means all your favorite tops will get along with the comfortable and versatile Bardot Capri. &bull; Black capris with pink waistband.&bull; Cropped leggings.&bull; Waistband drawcord. &bull; Flat, thin and flattering.&bull; Made with organic fabr", "category": "Pants", "top": "Clothing", "gender": "women", "price": 48.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp08-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp08-black_back.jpg", "stock": 241, "colors": [], "sizes": [], "type": "variable" }, { "id": 692, "sku": "MP01", "name": "Caesar Warm-Up Pant", "short": "This is a variable product called a Caesar Warm-Up Pant", "desc": "Command your workout and keep your muscles limber in the Caesar Warm-Up Pant. Engineered CoolTech&trade; fabric wicks away moisture so you don't have to worry about sweat and discomfort. The drawstring-adjustable waist helps make sure your pants fit properly. &bull; Light gray heather knit straight ", "category": "Pants", "top": "Clothing", "gender": "men", "price": 35.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/p/mp01-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/p/mp01-gray_back.jpg", "stock": 230, "colors": [], "sizes": [], "type": "variable" }, { "id": 1830, "sku": "WP09", "name": "Carina Basic Capri", "short": "This is a variable product called a Carina Basic Capri", "desc": "Perfect as workout pants or \"accessory,\" the Carina Basic Capri is comfy as it is practical -- try it under skirts or shorts. &bull; Black capris with rouching detail.&bull; 93% cotton, 7% spandex.&bull; Elasticized waistband.&bull; Reinforced seams with exposed topstitching.&bull; Soft, medium-weig", "category": "Pants", "top": "Clothing", "gender": "women", "price": 51.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp09-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp09-black_alt1.jpg", "stock": 14, "colors": [], "sizes": [], "type": "variable" }, { "id": 1795, "sku": "WP04", "name": "Cora Parachute Pant", "short": "This is a variable product called a Cora Parachute Pant", "desc": "Good for running, hiking, lounging or stretching, the Cora Parachute Pant presents comfortable styling designed to help you look and feel great. &bull; Light blue parachute pants.&bull; Power mesh internal waistband for support.&bull; Internal waistband pocket.&bull; Antimicrobial finish.", "category": "Pants", "top": "Clothing", "gender": "women", "price": 75.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp04-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp04-blue_alt1.jpg", "stock": 242, "colors": [], "sizes": [], "type": "variable" }, { "id": 835, "sku": "MP12", "name": "Cronus Yoga Pant", "short": "This is a variable product called a Cronus Yoga Pant", "desc": "Guys who love yoga love this all-purpose yoga pant. Designed for yoga, gym and lounge, the Cronus Yoga Pant is a loose-fitting, classic-style pant made from a lightweight blend of cotton, recycled polyester and a touch of Spandex for the stretch factor. &bull; Drawstring waist.&bull; Loose, straight", "category": "Pants", "top": "Clothing", "gender": "men", "price": 48.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/p/mp12-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/p/mp12-black_back.jpg", "stock": 172, "colors": [], "sizes": [], "type": "variable" }, { "id": 1837, "sku": "WP10", "name": "Daria Bikram Pant", "short": "This is a variable product called a Daria Bikram Pant", "desc": "The Daria Bikram Pant is designed to stretch and move with your body to make extreme yoga extremely comfortable. &bull; Heather gray capris with pink striped waist.&bull; Flatlock seams.&bull; Interior pocket.", "category": "Pants", "top": "Clothing", "gender": "women", "price": 51.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp10-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp10-gray_alt1.jpg", "stock": 170, "colors": [], "sizes": [], "type": "variable" }, { "id": 1851, "sku": "WP12", "name": "Deirdre Relaxed-Fit Capri", "short": "This is a variable product called a Deirdre Relaxed-Fit Capri", "desc": "Invigorate your yoga practice with this Deirdre Relaxed-Fit Capri, designed to let you move without strain or restriction. &bull; Heather gray capris with mint green waist &amp; accents.&bull; Comfortable, relaxed fit with high rise.&bull; Moisture-wicking, 4-way stretch construction.&bull; Lined wi", "category": "Pants", "top": "Clothing", "gender": "women", "price": 63.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp12-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp12-gray_back.jpg", "stock": 236, "colors": [], "sizes": [], "type": "variable" }, { "id": 1809, "sku": "WP06", "name": "Diana Tights", "short": "This is a variable product called a Diana Tights", "desc": "Perfect for hot bikram session or cool-down stretching. 8-percent stretch means you'll feel like anything is possible in the capri-style Diana Tights. &bull; Black legging with slate details.&bull; Flat-lock, chafe-free side seams.&bull; Vented gusset.&bull; Secret interior pocket.&bull; Sustainable", "category": "Pants", "top": "Clothing", "gender": "women", "price": 59.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp06-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp06-black_alt1.jpg", "stock": 24, "colors": [], "sizes": [], "type": "variable" }, { "id": 1781, "sku": "WP02", "name": "Emma Leggings", "short": "This is a variable product called a Emma Leggings", "desc": "These comfortable and practical leggings pair perfectly with any workout top or casual tee. The Emma's subtle contrast fabric and fit ensure you're stylish without overdoing it. &bull; Light blue heather yoga pants. &bull; Body hugging fit.&bull; Low rise fit.", "category": "Pants", "top": "Clothing", "gender": "women", "price": 42.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp02-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp02-blue_back.jpg", "stock": 59, "colors": [], "sizes": [], "type": "variable" }, { "id": 718, "sku": "MP03", "name": "Geo Insulated Jogging Pant", "short": "This is a variable product called a Geo Insulated Jogging Pant", "desc": "In the cold, even the toughest guys shiver, unless they're in the Geo Insulated Jogging Pant. Lightweight and wind resistant, they block brutal wind gusts and warm you to the bone. Breathable mesh keeps them dry on the inside. &bull; Black polyester spandex pants with zipper pockets.&bull; Reflectiv", "category": "Pants", "top": "Clothing", "gender": "men", "price": 51.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/p/mp03-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/p/mp03-black_alt1.jpg", "stock": 232, "colors": [], "sizes": [], "type": "variable" }, { "id": 1788, "sku": "WP03", "name": "Ida Workout Parachute Pant", "short": "This is a variable product called a Ida Workout Parachute Pant", "desc": "The Ida Workout Parachute Pant is made of lightweight, super-soft fabric that keeps you comfortable during any level of activity, whether Downward Dog or a few trail miles. &bull; Royal blue parachute pants.&bull; Contrast stripe.&bull; Relaxed fit. &bull; Drawstring closure.&bull; Machine wash/dry.", "category": "Pants", "top": "Clothing", "gender": "women", "price": 48.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp03-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp03-blue_alt1.jpg", "stock": 8, "colors": [], "sizes": [], "type": "variable" }, { "id": 1774, "sku": "WP01", "name": "Karmen Yoga Pant", "short": "This is a variable product called a Karmen Yoga Pant", "desc": "Good for running, hiking, lounging or stretching, the Cora Parachute Pant presents comfortable styling designed to help you look and feel great. &bull; Light blue parachute pants.&bull; Power mesh internal waistband for support.&bull; Internal waistband pocket.&bull; Antimicrobial finish.", "category": "Pants", "top": "Clothing", "gender": "women", "price": 39.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp01-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp01-gray_back.jpg", "stock": 11, "colors": [], "sizes": [], "type": "variable" }, { "id": 744, "sku": "MP05", "name": "Kratos Gym Pant", "short": "This is a variable product called a Kratos Gym Pant", "desc": "Like it's namesake god of strength, the Kratos Gym Pant help you bring your best to bear on weight-based, plyometric and endurance exercise. They stretch and support in all the right places while ultra-light construction and moisture-wicking technology provide comfort. &bull; Navy cotton straight le", "category": "Pants", "top": "Clothing", "gender": "men", "price": 57.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/p/mp05-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/p/mp05-blue_back.jpg", "stock": 166, "colors": [], "sizes": [], "type": "variable" }, { "id": 796, "sku": "MP09", "name": "Livingston All-Purpose Tight", "short": "This is a variable product called a Livingston All-Purpose Tight", "desc": "It's not often that your pants work as a yoga staple and a climbing buddy. The Livingston All-Purpose Tight is made with soft, 11% cotton and nylon polymer stretch construction that lets you find the right angle in the studio or on the mountain side. &bull; Breathable stretch organic cotton/spandex.", "category": "Pants", "top": "Clothing", "gender": "men", "price": 75.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/p/mp09-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/p/mp09-blue_alt1.jpg", "stock": 97, "colors": [], "sizes": [], "type": "variable" }, { "id": 757, "sku": "MP06", "name": "Mithra Warmup Pant", "short": "This is a variable product called a Mithra Warmup Pant", "desc": "When you're not sure you're up to the weather, don the Mithra Warmup Pant for a confidence boost. Its supersoft, stretchy fabric and fleece-like finish help prep your muscles and ease your mind. Designed for relaxed, easy-wear fit with handy ankle zips. &bull; Ankle zips.&bull; Elasticized waistband", "category": "Pants", "top": "Clothing", "gender": "men", "price": 28.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/p/mp06-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/p/mp06-gray_back.jpg", "stock": 150, "colors": [], "sizes": [], "type": "variable" }, { "id": 809, "sku": "MP10", "name": "Orestes Yoga Pant", "short": "This is a variable product called a Orestes Yoga Pant", "desc": "The Orestes Yoga Pant is a yoga basic that sustains the novice yogi well into the expert stage. Organic sustainable cotton and 9% stretch spandex let your body bend to your will while your conscience stays clear. &bull; A yoga essential.&bull; Breathable stretch organic cotton/spandex.&bull; Standar", "category": "Pants", "top": "Clothing", "gender": "men", "price": 66.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/p/mp10-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/p/mp10-black_alt1.jpg", "stock": 172, "colors": [], "sizes": [], "type": "variable" }, { "id": 1858, "sku": "WP13", "name": "Portia Capri", "short": "This is a variable product called a Portia Capri", "desc": "From yoga at dawn to evenings curled up with a book, our Portia Capri gives you the comfort and freedom to cruise through the day. The high-waisted design keeps you covered while you're bending and stretching, while contrast stripes promise a dash of sporty style. &bull; Salmon heather capri sweats.", "category": "Pants", "top": "Clothing", "gender": "women", "price": 49.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp13-orange_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp13-orange_back.jpg", "stock": 144, "colors": [], "sizes": [], "type": "variable" }, { "id": 1802, "sku": "WP05", "name": "Sahara Leggings", "short": "This is a variable product called a Sahara Leggings", "desc": "Contrast colors, full-length leggings, and opaque, high-stretch fabric define the Sahara. Flat-locked seams and quick-drying wicking properties prevent chafing. A deep elasticated waistband and carefully positioned seamlines create a flattering silhouette. &bull; Pinstripe legging with rouched ankle", "category": "Pants", "top": "Clothing", "gender": "women", "price": 75.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp05-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp05-gray_alt1.jpg", "stock": 137, "colors": [], "sizes": [], "type": "variable" }, { "id": 731, "sku": "MP04", "name": "Supernova Sport Pant", "short": "This is a variable product called a Supernova Sport Pant", "desc": "Turn the corner and open it up -- your driveway is two blocks away. The Supernova Sport Pant gets you there with key features like moisture-wicking LumaTech&trade; fabric and mesh ventilation. Side seam pockets ensure total convenience during rest periods. &bull; Dark heather gray straight leg cotto", "category": "Pants", "top": "Clothing", "gender": "men", "price": 45.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/p/mp04-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/p/mp04-gray_back.jpg", "stock": 131, "colors": [], "sizes": [], "type": "variable" }, { "id": 1844, "sku": "WP11", "name": "Sylvia Capri", "short": "This is a variable product called a Sylvia Capri", "desc": "Enjoy a sleek, slendering look in new Silvia Capri. Its engineer-designed support fabric feels great and gives the illusion of a slimmer fit. &bull; Green striped capri.&bull; Strategic side seam&bull; Comfort gusset with lining.&bull; Flat seaming.&bull; Wide waistband.&bull; Moisture wicking.", "category": "Pants", "top": "Clothing", "gender": "women", "price": 42.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/p/wp11-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/p/wp11-green_back.jpg", "stock": 130, "colors": [], "sizes": [], "type": "variable" }, { "id": 770, "sku": "MP07", "name": "Thorpe Track Pant", "short": "This is a variable product called a Thorpe Track Pant", "desc": "Thirty degree temps are chilly for most, except when you're in Thorpe Track Pants. These top-of-the-line track bottoms are made from fast-drying, weather-resistant fabric with an internal breathable layer of mesh nylon to wick away moisture. &bull; Moisture transfer properties. &bull; 7% stretch.&bu", "category": "Pants", "top": "Clothing", "gender": "men", "price": 68.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/p/mp07-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/p/mp07-blue_alt1.jpg", "stock": 131, "colors": [], "sizes": [], "type": "variable" }, { "id": 705, "sku": "MP02", "name": "Viktor LumaTech&trade; Pant", "short": "This is a variable product called a Viktor LumaTech&trade; Pant", "desc": "You'll love the new Viktor LumaTech&trade; Pant, with featherweight fleece fabric lining and stretchy, sweat-wicking material. It delivers toasty warmth on the sidelines or in cold-weather training, with reflective trim for a safe finish. &bull; Dark gray polyester/spandex straight leg pants.&bull; ", "category": "Pants", "top": "Clothing", "gender": "men", "price": 46.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/p/mp02-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/p/mp02-gray_back.jpg", "stock": 248, "colors": [], "sizes": [], "type": "variable" }, { "id": 783, "sku": "MP08", "name": "Zeppelin Yoga Pant", "short": "This is a variable product called a Zeppelin Yoga Pant", "desc": "Climb every mountain, or hold every pose, in the all-purpose Zepellin Yoga Pant. With its thin fleece interior and smooth layer-friendly surface, you'll get all the comfort and versatility you need. &bull; Smooth exterior for easy over-layering.&bull; Brushed fleece interior insulates and wicks. &bu", "category": "Pants", "top": "Clothing", "gender": "men", "price": 82.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/p/mp08-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/p/mp08-green_alt1.jpg", "stock": 55, "colors": [], "sizes": [], "type": "variable" }, { "id": 1972, "sku": "WSH10", "name": "Ana Running Short", "short": "This is a variable product called a Ana Running Short", "desc": "Time to lace up your kicks and beat that personal best in the Ana Running Short. It's designed with breathable mesh side panels to help keep you cool while you master the miles. &bull; Black/pink two-layer shorts.&bull; Low-rise elastic waistband.&bull; Relaxed fit. &bull; Ultra-lightweight fabric. ", "category": "Shorts", "top": "Clothing", "gender": "women", "price": 40.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/wsh10-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/wsh10-black_alt1.jpg", "stock": 194, "colors": [], "sizes": [], "type": "variable" }, { "id": 1945, "sku": "WSH06", "name": "Angel Light Running Short", "short": "This is a variable product called a Angel Light Running Short", "desc": "The Angel Light Running Short offers comfort in an ultra-lightweight, breathable package. With fabric infused with all-natural Cocona&reg; performance technology, it can whisk away sweat and block UV rays. &bull; Dark heather gray running shorts.&bull; Snug fit. &bull; Elastic waistband. &bull; Coco", "category": "Shorts", "top": "Clothing", "gender": "women", "price": 42.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/wsh06-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/wsh06-gray_alt1.jpg", "stock": 196, "colors": [], "sizes": [], "type": "variable" }, { "id": 853, "sku": "MSH02", "name": "Apollo Running Short", "short": "This is a variable product called a Apollo Running Short", "desc": "Fleet of foot or slow and steady, you'll be in complete comfort with the Apollo Running Short. Lightweight polyester material lets you move with ease, and mesh side panels promise plenty of ventilation as you work up a sweat. The stretchy elastic waistband delivers a flexible fit. &bull; Black short", "category": "Shorts", "top": "Clothing", "gender": "men", "price": 32.5, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/msh02-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/msh02-black_alt1.jpg", "stock": 118, "colors": [], "sizes": [], "type": "variable" }, { "id": 970, "sku": "MSH11", "name": "Arcadio Gym Short", "short": "This is a variable product called a Arcadio Gym Short", "desc": "The Arcadio Gym short is a basic-looking piece with a lot more than meets the eye: casual-yet-functional relaxed fit, comfortable waistband and adjustable drawstring. Everything you need, plus more. &bull; Royal blue cotton shorts.&bull; Built-in mesh brief. &bull; 87% Spandex 13% Lycra.&bull; Adjus", "category": "Shorts", "top": "Clothing", "gender": "men", "price": 20.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/msh11-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/msh11-blue_back.jpg", "stock": 8, "colors": [], "sizes": [], "type": "variable" }, { "id": 1922, "sku": "WSH04", "name": "Artemis Running Short", "short": "This is a variable product called a Artemis Running Short", "desc": "Discover smooth jogging and chic comfort each time you slip into the Artemis Running Short. A unique maritime-inspired design and oolor theme features a stretchy drawstring waist. &bull; Black rouched shorts with mint waist. &bull; Soft, lightweight construction.&bull; LumaTech&trade; wicking techno", "category": "Shorts", "top": "Clothing", "gender": "women", "price": 45.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/wsh04-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/wsh04-black_alt1.jpg", "stock": 150, "colors": [], "sizes": [], "type": "variable" }, { "id": 1938, "sku": "WSH05", "name": "Bess Yoga Short", "short": "This is a variable product called a Bess Yoga Short", "desc": "Designed for intense physical activity &ndash; think bikram &ndash; our Bess Yoga Short features moisture-wicking, four-way stretch fabric that lets you move in every which way. A vented gusset adds breathability and range of motion. &bull; Navy cotton shorts with light bue waist detail.&bull; Front", "category": "Shorts", "top": "Clothing", "gender": "women", "price": 28.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/wsh05-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/wsh05-blue_back.jpg", "stock": 233, "colors": [], "sizes": [], "type": "variable" }, { "id": 848, "sku": "MSH01", "name": "Cobalt CoolTech&trade; Fitness Short", "short": "This is a variable product called a Cobalt CoolTech&trade; Fitness Short", "desc": "It's not rocket surgery: the hotter your conditions, the more you need to stay cool. That's why the Cobalt CoolTech&trade; Fitness Short features built-in technology and side perforations that keep you ventilated and comfortable for the distance. The elastic waist promises a personalized fit. &bull;", "category": "Shorts", "top": "Clothing", "gender": "men", "price": 44.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/msh01-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/msh01-blue_back.jpg", "stock": 91, "colors": [], "sizes": [], "type": "variable" }, { "id": 1952, "sku": "WSH07", "name": "Echo Fit Compression Short", "short": "This is a variable product called a Echo Fit Compression Short", "desc": "Your muscles know it's go time the second you pull on the Echo Fit Compression Short. A balance of firm, stimulating squeeze with breathability, it offers the support and comfort you need to give it your all. &bull; Black compression shorts.&bull; High-waisted cut.&bull; Compression fit.&bull; Insea", "category": "Shorts", "top": "Clothing", "gender": "women", "price": 24.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/wsh07-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/wsh07-black_back.jpg", "stock": 75, "colors": [], "sizes": [], "type": "variable" }, { "id": 2000, "sku": "WSH12", "name": "Erika Running Short", "short": "This is a variable product called Erika Running Shorts", "desc": "A great short with a body-hugging design, the Erika Running Short is perfect for runners who prefer a fitted short rather than the traditional baggy variety. &bull; Seafoam pattern running shorts.&bull; Elastic waistband.&bull; Snug fit.&bull; 4'' inseam.&bull; 76% premium brushed Nylon / 24% Spande", "category": "Shorts", "top": "Clothing", "gender": "women", "price": 49.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/wsh12-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/wsh12-green_alt1.jpg", "stock": 217, "colors": [], "sizes": [], "type": "variable" }, { "id": 1874, "sku": "WSH01", "name": "Fiona Fitness Short", "short": "This is a variable product called a Fiona Fitness Short", "desc": "Don't let the plain style fool you: the Fiona Fitness Short demands notice on several levels. Comfort, of course. But also a performance-grade wicking fabric that takes everything you can give. &bull; Black run shorts - cotton/spandex.&bull; 5&rdquo; inseam.&bull; Machine wash/Line dry.", "category": "Shorts", "top": "Clothing", "gender": "women", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/wsh01-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/wsh01-black_back.jpg", "stock": 8, "colors": [], "sizes": [], "type": "variable" }, { "id": 1906, "sku": "WSH03", "name": "Gwen Drawstring Bike Short", "short": "This is a variable product called a Gwen Drawstring Bike Short", "desc": "For a completely modern style with moisture-wicking CoolTech&trade; technology, try the Gwen Drawstring Bike Short. Subtle grays and eye catching stitching combine with an adjustable waist for the perfect look and fit. &bull; Dark heather gray rouched bike shorts.&bull; Fitted. Inseam: 2\". &bull; Ma", "category": "Shorts", "top": "Clothing", "gender": "women", "price": 50.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/wsh03-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/wsh03-gray_alt1.jpg", "stock": 213, "colors": [], "sizes": [], "type": "variable" }, { "id": 892, "sku": "MSH05", "name": "Hawkeye Yoga Short", "short": "This is a variable product called a Hawkeye Yoga Short", "desc": "What more do you need than a sporty yoga short made with organic cotton and a little spandex for mobility? The Hawkeye Yoga Short brings a stylish, standard fit you can sport with confidence outside the studio. &bull; Dark gray shorts with red accents.&bull; 92% Organic Cotton 8% Spandex.&bull; Brea", "category": "Shorts", "top": "Clothing", "gender": "men", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/msh05-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/msh05-gray_alt1.jpg", "stock": 165, "colors": [], "sizes": [], "type": "variable" }, { "id": 1979, "sku": "WSH11", "name": "Ina Compression Short", "short": "This is a variable product called a Ina Compression Short", "desc": "One of Luma's most popular items, the Ina Compression Short has you covered with exceptional support and comfort, whether you're running the trail, riding a bike or ripping out reps. The ventilating fabric offers cool relief and prevents irritating chafing. &bull; Royal blue bike shorts.&bull; Compr", "category": "Shorts", "top": "Clothing", "gender": "women", "price": 49.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/wsh11-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/wsh11-blue_back.jpg", "stock": 173, "colors": [], "sizes": [], "type": "variable" }, { "id": 905, "sku": "MSH06", "name": "Lono Yoga Short", "short": "This is a variable product called a Lono Yoga Short", "desc": "Some shorts are &ldquo;all-purpose,&rdquo; while the Lono Yoga Short has a single purpose: yoga. Soft, flexible outer material moves with your body, and a secure, integrated boxer brief liner gives you extra confidence as you push your physique to the outer limits. &bull; Dark gray shorts with mesh ", "category": "Shorts", "top": "Clothing", "gender": "men", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/msh06-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/msh06-gray_alt1.jpg", "stock": 94, "colors": [], "sizes": [], "type": "variable" }, { "id": 1890, "sku": "WSH02", "name": "Maxima Drawstring Short", "short": "This is a variable product called a Maxima Drawstring Short", "desc": "Even laid-back women like to stay stylish, and that's just what the Maxima Drawstring Short delivers. An elastic waist keeps the fit flexible, on the deck at home or on a trail walk. Sporty tennis flair adds an athletic accent. &bull; Light gray run shorts - cotton polyester. &bull; Contrast binding", "category": "Shorts", "top": "Clothing", "gender": "women", "price": 28.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/wsh02-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/wsh02-gray_back.jpg", "stock": 60, "colors": [], "sizes": [], "type": "variable" }, { "id": 866, "sku": "MSH03", "name": "Meteor Workout Short", "short": "This is a variable product called a Meteor Workout Short", "desc": "Step into the Meteor Workout Short for an incredibly lightweight fitness experience. Its breathable construction and an inner brief provide additional comfort and support, while the adjustable waistband offers the perfect fit to take you to the finish. &bull; Royal blue shorts with light blue accent", "category": "Shorts", "top": "Clothing", "gender": "men", "price": 32.5, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/msh03-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/msh03-blue_alt1.jpg", "stock": 188, "colors": [], "sizes": [], "type": "variable" }, { "id": 1965, "sku": "WSH09", "name": "Mimi All-Purpose Short", "short": "This is a variable product called a Mimi All-Purpose Short", "desc": "You can run, bike or swim in the do-anything, water-resistance Mimi Short. No need to worry about rubbing-induced soreness either, with flatlock seams and soft, chafe-resistant material. &bull; Gray/seafoam two-layer shorts.&bull; Water-resistant construction.&bull; Inner mesh brief for breathable s", "category": "Shorts", "top": "Clothing", "gender": "women", "price": 44.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/wsh09-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/wsh09-gray_alt1.jpg", "stock": 209, "colors": [], "sizes": [], "type": "variable" }, { "id": 931, "sku": "MSH08", "name": "Orestes Fitness Short", "short": "This is a variable product called a Orestes Fitness Short", "desc": "You're out of excuses not to run, lift, or play when you own a pair of the Orestes Fitness Short. You won't complain about the high-performance polyester wicking fabric, reflective safety trim or seamless built-in comfort brief, either. &bull; Black shorts with dark gray accents.&bull; Elasticized w", "category": "Shorts", "top": "Clothing", "gender": "men", "price": 35.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/msh08-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/msh08-black_back.jpg", "stock": 6, "colors": [], "sizes": [], "type": "variable" }, { "id": 983, "sku": "MSH12", "name": "Pierce Gym Short", "short": "This is a variable product called a Pierce Gym Short", "desc": "The Pierce Gym Short is similar to our Arcadio Gym Short, but designed with a fitted, tapered look. A flat waistband with an adjustable drawstring adds comfort, with zippered pockets for safe, easy storage. &bull; Dark red cotton shorts. &bull; 87% Supplex, 13% Lycra.&bull; Adjustable drawstring wai", "category": "Shorts", "top": "Clothing", "gender": "men", "price": 27.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/msh12-red_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/msh12-red_back.jpg", "stock": 83, "colors": [], "sizes": [], "type": "variable" }, { "id": 918, "sku": "MSH07", "name": "Rapha  Sports Short", "short": "This is a variable product called a Rapha Sports Short", "desc": "For those about to sweat, we support you with the Rapha Sports Short. The compression-fit liner surrounds your muscles with the stimulation they need to find your high gear, while moisture-wicking performance fabric helps prevents sweat build-up. &bull; Black shorts with royal accents.&bull; Compres", "category": "Shorts", "top": "Clothing", "gender": "men", "price": 35.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/msh07-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/msh07-black_alt1.jpg", "stock": 149, "colors": [], "sizes": [], "type": "variable" }, { "id": 957, "sku": "MSH10", "name": "Sol Active Short", "short": "This is a variable product called a Sol Active Short", "desc": "You'll let your fear go and push your limits in your new Sol Active Short. Featuring ultra-breathable performance fabric and a flat comfort-fit waistband, the Sol Active Short is perfect for high-intensity circuits or high-heat bikram. &bull; Light blue jersey shorts with mesh detail.&bull; 87% Span", "category": "Shorts", "top": "Clothing", "gender": "men", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/msh10-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/msh10-blue_alt1.jpg", "stock": 162, "colors": [], "sizes": [], "type": "variable" }, { "id": 1958, "sku": "WSH08", "name": "Sybil Running Short", "short": "This is a variable product called a Sybil Running Short", "desc": "Fortunately, it's okay to look cute while you're working out. The Sybil Running Short combines a fun, color-blocked design with breathable mesh fabric for sporty-fun style. &bull; Blue running shorts with green waist.&bull; Drawstring-adjustable waist.&bull; 4\" inseam. Machine wash/line dry.", "category": "Shorts", "top": "Clothing", "gender": "women", "price": 44.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/wsh08-purple_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/wsh08-purple_back.jpg", "stock": 230, "colors": [], "sizes": [], "type": "variable" }, { "id": 879, "sku": "MSH04", "name": "Torque Power Short", "short": "This is a variable product called a Torque Power Short", "desc": "Take a turn in the Torque Power Short and enjoy the smooth, motion-friendly stretch and sweat-wicking, UV-protective material. You'll stay cool, dry and energized miles later. &bull; Light gray shorts.&bull; Fitted design. &bull; Elastic waistband. &bull; Flat-seam construction. &bull; 7\" inseam. &b", "category": "Shorts", "top": "Clothing", "gender": "men", "price": 32.5, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/msh04-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/msh04-gray_back.jpg", "stock": 148, "colors": [], "sizes": [], "type": "variable" }, { "id": 944, "sku": "MSH09", "name": "Troy Yoga Short", "short": "This is a variable product called a Troy Yoga Short", "desc": "The versatile, all-purpose Troy Yoga Short is knee-lengthed with an elastic waistband with drawstring, making it comfortably flexible. The removable LumaTech&trade; wicking liner is also bacteria resistant. &bull; Navy polyester pinstripe shorts.&bull; Woven fabric with moderate stretch.&bull; 62% c", "category": "Shorts", "top": "Clothing", "gender": "men", "price": 24.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/msh09-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/msh09-blue_alt1.jpg", "stock": 16, "colors": [], "sizes": [], "type": "variable" }, { "id": 649, "sku": "MT07", "name": "Argus All-Weather Tank", "short": "This is a variable product called a Argus All-Weather Tank", "desc": "The Argus All-Weather Tank is sure to become your favorite base layer or go-to cover for hot outdoor workouts. With its subtle reflective safely trim, you can even wear it jogging on urban evenings. &bull; Dark gray polyester spandex tank.&bull; Reflective details for nighttime visibility. &bull; St", "category": "Tanks", "top": "Clothing", "gender": "men", "price": 22.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/t/mt07-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/t/mt07-gray_back.jpg", "stock": 126, "colors": [], "sizes": [], "type": "variable" }, { "id": 673, "sku": "MT11", "name": "Atlas Fitness Tank", "short": "This is a variable product called a Atlas Fitness Tank", "desc": "From weekend warrior to Warrior Pose II, no role can beat the Atlas Fitness Tank, a versatile top for gym and yoga studio. Wicking-weave soft fabric helps prevent uncomfortable chafing. &bull; Teal scoop neck cotton tank.&bull; Triblend, soft fabric.&bull; Relaxed fit.", "category": "Tanks", "top": "Clothing", "gender": "men", "price": 18.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/t/mt11-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/t/mt11-blue_back.jpg", "stock": 40, "colors": [], "sizes": [], "type": "variable" }, { "id": 679, "sku": "MT12", "name": "Cassius Sparring Tank", "short": "This is a variable product called a Cassius Sparring Tank", "desc": "Whether you're up against a partner or the clock, the Cassius Sparring Tank is in your corner, moving effortless with your body. The light and loose feel gives you no reason to give up before the bell or the end of the block. &bull; Royal crewneck cotton tank.&bull; Contrast stitching.&bull; Self fa", "category": "Tanks", "top": "Clothing", "gender": "men", "price": 18.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/t/mt12-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/t/mt12-blue_back.jpg", "stock": 238, "colors": [], "sizes": [], "type": "variable" }, { "id": 593, "sku": "MT01", "name": "Erikssen CoolTech&trade; Fitness Tank", "short": "This is a variable product called a Erikssen CoolTech&trade; Fitness Tank", "desc": "A good running tank helps make the miles pass by keep you cool. The Erikssen CoolTech&trade; Fitness Tank completes that mission, with performance fabric engineered to wick perspiration and promote airflow. &bull; Red performance tank.&bull; Slight scoop neckline. &bull; Reflectivity. &bull; Machine", "category": "Tanks", "top": "Clothing", "gender": "men", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/t/mt01-red_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/t/mt01-red_alt1.jpg", "stock": 91, "colors": [], "sizes": [], "type": "variable" }, { "id": 631, "sku": "MT04", "name": "Helios Endurance Tank", "short": "This is a variable product called a Helios Endurance Tank", "desc": "When training pushes your limits, you need gear that works harder than you. Our mesh Helio Training Tank is crafted from super-soft, ultra-lightweight fabric that stretches in all directions to follow your every move, on mat, court or street. &bull; Blue heather tank with gray pocket.&bull; Contrast", "category": "Tanks", "top": "Clothing", "gender": "men", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/t/mt04-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/t/mt04-blue_alt1.jpg", "stock": 166, "colors": [], "sizes": [], "type": "variable" }, { "id": 625, "sku": "MT03", "name": "Primo Endurance Tank", "short": "This is a variable product called a Primo Endurance Tank", "desc": "Chances are your workout goes beyond free weights, which is why the Primo Endurance Tank employs maximum versatility. Run, lift or play ball &ndash; this breathable mesh top will keep you cool during all your activities. &bull; Red heather tank with gray pocket.&bull; Chafe-resistant flatlock seams.", "category": "Tanks", "top": "Clothing", "gender": "men", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/t/mt03-red_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/t/mt03-red_main.jpg", "stock": 179, "colors": [], "sizes": [], "type": "variable" }, { "id": 637, "sku": "MT05", "name": "Rocco Gym Tank", "short": "This is a variable product called a Rocco Gym Tank", "desc": "Weights? Heavy Bag? Bikram? However you get sweaty, you'll be prepared with the moisture-wicking all-purpose Rocco Gym Tank. The free-moving cut gives you maximum range of motion, with flatlock seams to eliminate chafing. &bull; Light blue heather gray tank.&bull; Quick-drying, moisture-wicking.&bul", "category": "Tanks", "top": "Clothing", "gender": "men", "price": 24.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/t/mt05-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/t/mt05-blue_back.jpg", "stock": 34, "colors": [], "sizes": [], "type": "variable" }, { "id": 661, "sku": "MT09", "name": "Sinbad Fitness Tank", "short": "This is a variable product called a Sinbad Fitness Tank", "desc": "Solid in color and construction, the 100% cotton-weave Sinbad Fitness Tank moves with you as you press, hold, crunch and stride your way to the ultimate you. &bull; Teal polyester tank.&bull; Premium fit tank top.&bull; Ultra lightweight.&bull; Naturally odor-resistant.", "category": "Tanks", "top": "Clothing", "gender": "men", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/t/mt09-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/t/mt09-blue_alt1.jpg", "stock": 67, "colors": [], "sizes": [], "type": "variable" }, { "id": 655, "sku": "MT08", "name": "Sparta Gym Tank", "short": "This is a variable product called a Sparta Gym Tank", "desc": "The high performance Sparta Gym Tank is made with thin, light, merino wool and aims to be the perfect base layer or balmy weather running and fitness top. &bull; Green polyester tank.&bull; Ultra lightweight.&bull; Naturally odor-resistant.&bull; Close-to-body athletic fit.&bull; Chafe-resistant fla", "category": "Tanks", "top": "Clothing", "gender": "men", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/t/mt08-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/t/mt08-green_alt1.jpg", "stock": 218, "colors": [], "sizes": [], "type": "variable" }, { "id": 667, "sku": "MT10", "name": "Tiberius Gym Tank", "short": "This is a variable product called a Tiberius Gym Tank", "desc": "Whether you're classy or just sweaty, the Tiberius Gym Tank helps you look good while you're at it. What's more, its moisture-wicking, quick-drying, anti-microbial and anti-odor construction help ensure you're welcome back to the gym. &bull; Yellow scoop neck cotton tank.&bull; Comfortable, relaxed ", "category": "Tanks", "top": "Clothing", "gender": "men", "price": 18.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/t/mt10-yellow_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/t/mt10-yellow_back.jpg", "stock": 245, "colors": [], "sizes": [], "type": "variable" }, { "id": 609, "sku": "MT02", "name": "Tristan Endurance Tank", "short": "This is a variable product called a Tristan Endurance Tank", "desc": "Push yourself through punishing runs, plyometric workouts, intense competition and more in our athletic Tristan Endurance Tank. Constructed with built-in moisture-wicking technology, it's designed to keep you completely cool and dry on the long haul. &bull; White performance tank.&bull; Stylish cont", "category": "Tanks", "top": "Clothing", "gender": "men", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/t/mt02-white_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/t/mt02-white_alt1.jpg", "stock": 227, "colors": [], "sizes": [], "type": "variable" }, { "id": 643, "sku": "MT06", "name": "Vulcan Weightlifting Tank", "short": "This is a variable product called a Vulcan Weightlifting Tank", "desc": "The Olympic styled Vulcan Weightlifting Tank features polyester stretch and flex. Hit the rack in sleeveless style and unleash your personal best. This tank is designed to max performance, comfort and range of motion. &bull; Black polyester spandex tank.&bull; 100% polyester.&bull; Freedom of moveme", "category": "Tanks", "top": "Clothing", "gender": "men", "price": 28.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/t/mt06-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/t/mt06-black_back.jpg", "stock": 16, "colors": [], "sizes": [], "type": "variable" }, { "id": 513, "sku": "MS01", "name": "Aero Daily Fitness Tee", "short": "This is a variable product called a Aero Daily Fitness Tee", "desc": "Need an everyday action tee that helps keep you dry? The Aero Daily Fitness Tee is made of 100% polyester wicking knit that funnels moisture away from your skin. Don't be fooled by its classic style; this tee hides premium performance technology beneath its unassuming look. Relaxed fit. Short-Sleeve", "category": "Tees", "top": "Clothing", "gender": "men", "price": 24.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/ms01-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/ms01-blue_back.jpg", "stock": 235, "colors": [], "sizes": [], "type": "variable" }, { "id": 465, "sku": "MS12", "name": "Atomic Endurance Running Tee (Crew-Neck)", "short": "This is a variable product called a Atomic Endurance Running Tee (Crew-Neck)", "desc": "Like it's v-neck counterpart, the crew-neck Atomic Tee will get you to your goal and beyond with its many load-bearing features: ultra-lightweight, moisture-wicking Cocona&reg; fabric, chafe-free flatlock seams and an ergonomic cut that moves with your body. &bull; Red polyester tee.&bull; Crew neck", "category": "Tees", "top": "Clothing", "gender": "men", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/ms12-red_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/ms12-red_alt1.jpg", "stock": 8, "colors": [], "sizes": [], "type": "variable" }, { "id": 449, "sku": "MS11", "name": "Atomic Endurance Running Tee (V-neck)", "short": "This is a variable product called a Atomic Endurance Running Tee (V-neck)", "desc": "Reach your limit and keep on going in the Atomic Endurance Running Tee. Built to help any athlete push past the wall with loads of performance features. &bull; Lime heathered v-neck tee. &bull; Ultra-lightweight.&bull; Moisture-wicking Cocona&reg; fabric.&bull; Machine wash/dry.", "category": "Tees", "top": "Clothing", "gender": "men", "price": 28.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/ms11-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/ms11-green_back.jpg", "stock": 197, "colors": [], "sizes": [], "type": "variable" }, { "id": 481, "sku": "MS03", "name": "Balboa Persistence Tee", "short": "This is a variable product called a Balboa Persistence Tee", "desc": "The Balboa Persistence Tee is a must-have for any athlete, Philadelphia or elsewhere. We took the best of performance apparel, cut the fluff and boiled it down to the basics for a lightweight, quick-drying t-shirt. &bull; Crew neckline. &bull; Semi-fitted. &bull; Cocona&reg; performance fabric. &bul", "category": "Tees", "top": "Clothing", "gender": "men", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/ms03-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/ms03-black_back.jpg", "stock": 5, "colors": [], "sizes": [], "type": "variable" }, { "id": 561, "sku": "MS07", "name": "Deion Long-Sleeve EverCool&trade; Tee", "short": "This is a variable product called a Deion Long-Sleeve EverCool&trade; Tee", "desc": "When you're training, ordinary tees don't cut it. That's why the Dion Long-Sleeve EverCool&trade; Tee features revolutionary Cocona&reg; fabric derived from a renewable resource: coconut shells. This unique material protects you from harmful UV rays, wicks away sweat, controls stink and dries quickl", "category": "Tees", "top": "Clothing", "gender": "men", "price": 39.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/ms07-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/ms07-green_alt1.jpg", "stock": 238, "colors": [], "sizes": [], "type": "variable" }, { "id": 1543, "sku": "WS05", "name": "Desiree Fitness Tee", "short": "This is a variable product called a Desiree Fitness Tee", "desc": "When you're too far to turn back, thank yourself for choosing the Desiree Fitness Tee. Its ultra-lightweight, ultra-breathable fabric wicks sweat away from your body and helps keeps you cool for the distance. &bull; Short-Sleeves.&bull; Performance fabric.&bull; Machine wash/line dry.", "category": "Tees", "top": "Clothing", "gender": "women", "price": 24.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/ws05-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/ws05-black_alt1.jpg", "stock": 170, "colors": [], "sizes": [], "type": "variable" }, { "id": 1495, "sku": "WS11", "name": "Diva Gym Tee", "short": "This is a variable product called a Diva Gym Tee", "desc": "The Diva Gym Tee feels like your favorite right out of the mailbox. Micro-sleeved, lightweight and extra comfortable, it's a casual staple that hides incredible comfort behind a laid-back, carefree look. &bull; Bright yellow v-neck tee.&bull; Moisture-wicking fabric. &bull; Long-Sleeves. &bull; Mach", "category": "Tees", "top": "Clothing", "gender": "women", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/ws11-yellow_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/ws11-yellow_back.jpg", "stock": 147, "colors": [], "sizes": [], "type": "variable" }, { "id": 1415, "sku": "WS06", "name": "Elisa EverCool&trade; Tee", "short": "This is a variable product called a Elisa EverCool&trade; Tee", "desc": "When rising temps threaten to melt you down, Elisa EverCool&trade; Tee brings serious relief. Moisture-wicking fabric pulls sweat away from your skin, while the innovative seams hug your muscles to enhance your range of motion. &bull; Purple heather v-neck tee.&bull; Short-Sleeves.&bull; Luma EverCo", "category": "Tees", "top": "Clothing", "gender": "women", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/ws06-purple_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/ws06-purple_back.jpg", "stock": 128, "colors": [], "sizes": [], "type": "variable" }, { "id": 1367, "sku": "WS02", "name": "Gabrielle Micro Sleeve Top", "short": "This is a variable product called a Gabrielle Micro Sleeve Top", "desc": "Luma's most popular top, the Gabrielle Micro Sleeve is back with even more comfort and style. &bull; Lime green v-neck tee.&bull; Slimming, flattering fit.&bull; Moisture-wicking, quick-drying, anti-microbial, and anti-odor construction.&bull; Longer curved hem provides additional coverage.&bull; 55", "category": "Tees", "top": "Clothing", "gender": "women", "price": 28.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/ws02-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/ws02-green_back.jpg", "stock": 247, "colors": [], "sizes": [], "type": "variable" }, { "id": 401, "sku": "MS04", "name": "Gobi HeatTec&reg; Tee", "short": "This is a variable product called a Gobi HeatTec&reg; Tee", "desc": "When the training gets intense, the Gobi HeatTec&reg; Tee works as hard as you do to maintain your cool. The moisture-wicking material promises drier comfort, while breathable side panels deliver extra stretch that's sure to keep you moving. &bull; Orange micropolyester shirt.&bull; HeatTec&reg; wic", "category": "Tees", "top": "Clothing", "gender": "men", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/ms04-orange_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/ms04-orange_back.jpg", "stock": 44, "colors": [], "sizes": [], "type": "variable" }, { "id": 1527, "sku": "WS01", "name": "Gwyn Endurance Tee", "short": "This is a variable product called a Gwyn Endurance Tee", "desc": "When the miles add up, comfort is crucial. The short-sleeve Gwyn Endurance Tee is designed with an ultra-lightweight blend of breathable fabrics to help you tackle your training. Female-specific seams and a sporty v-neckline offer subtle style. &bull; Short-Sleeves.&bull; Machine wash/line dry.", "category": "Tees", "top": "Clothing", "gender": "women", "price": 24.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/ws01-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/ws01-black_back.jpg", "stock": 179, "colors": [], "sizes": [], "type": "variable" }, { "id": 417, "sku": "MS05", "name": "Helios EverCool&trade; Tee", "short": "This is a variable product called a Helios EverCool&trade; Tee", "desc": "Pumping iron or dialing the track, you've got cool comfort on your side in our short-sleeve Helios EverCool&trade; Tee. The fabric is infused with moisture-wicking technology that pulls sweat off your skin for speedy evaporation. Stretchy fabric gussets encourage ventilation while increasing your ra", "category": "Tees", "top": "Clothing", "gender": "men", "price": 24.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/ms05-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/ms05-blue_back.jpg", "stock": 90, "colors": [], "sizes": [], "type": "variable" }, { "id": 1383, "sku": "WS03", "name": "Iris Workout Top", "short": "This is a variable product called a Iris Workout Top", "desc": "The Iris Workout Top is sleeveless, body fitting and moisture wicking for a fashionable, functional garment. Rouched textures add style. &bull; Pink heather rouched v-neck.&bull; Scoop neckline.&bull; Angled flat seams.&bull; Moisture wicking.&bull; Body skimming.&bull; 83% Polyester / 11% TENCEL&re", "category": "Tees", "top": "Clothing", "gender": "women", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/ws03-red_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/ws03-red_alt1.jpg", "stock": 143, "colors": [], "sizes": [], "type": "variable" }, { "id": 1431, "sku": "WS07", "name": "Juliana Short-Sleeve Tee", "short": "This is a variable product called a Juliana Short-Sleeve Tee", "desc": "The Juliana Short-Sleeve Tee gives you more than sporty style. Consider that its soft Cocona&reg; polyester fabric accelerates evaporation to help keep you dry and comfortable, too. And a selection of colors lets you get more than one. &bull; Black scoop neck tee.&bull; Side rouching.&bull; Relaxed ", "category": "Tees", "top": "Clothing", "gender": "women", "price": 42.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/ws07-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/ws07-black_alt1.jpg", "stock": 7, "colors": [], "sizes": [], "type": "variable" }, { "id": 1479, "sku": "WS10", "name": "Karissa V-Neck Tee", "short": "This is a variable product called a Karissa V-Neck Tee", "desc": "The Karissa V-Neck Tee features a semi-fitted shape that's flattering for every figure. You can hit the gym with confidence while it hugs curves and hides common \"problem\" areas. &bull; Pink heather soft v-neck tee. &bull; Luma signature micro sleeves.&bull; Semi-fitted. &bull; Machine wash/dry.", "category": "Tees", "top": "Clothing", "gender": "women", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/ws10-red_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/ws10-red_back.jpg", "stock": 111, "colors": [], "sizes": [], "type": "variable" }, { "id": 1399, "sku": "WS04", "name": "Layla Tee", "short": "This is a variable product called a Layla Tee", "desc": "Work out or hang out in chic style in the Layla Tee. With a lightweight sheer design and a roomy neckline, this tee fits you comfortably while looking stylish. &bull; Teal tee.&bull; Long back hem.&bull; Dropped shoulders.", "category": "Tees", "top": "Clothing", "gender": "women", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/ws04-green_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/ws04-green_back.jpg", "stock": 149, "colors": [], "sizes": [], "type": "variable" }, { "id": 545, "sku": "MS10", "name": "Logan  HeatTec&reg; Tee", "short": "This is a variable product called a Logan HeatTec&reg; Tee", "desc": "Soft and lightweight, the Logan HeatTec&reg; Tee gets you through the long haul in total comfort. It boasts superior sweat-wicking performance to keep skin dry and cool, and strategic flat-lock seams to resist chafing. &bull; Semi-fitted. &bull; Crew neckline. &bull; Machine wash/tumble dry.", "category": "Tees", "top": "Clothing", "gender": "men", "price": 24.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/ms10-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/ms10-blue_alt1.jpg", "stock": 219, "colors": [], "sizes": [], "type": "variable" }, { "id": 1447, "sku": "WS08", "name": "Minerva LumaTech&trade; V-Tee", "short": "This is a variable product called a Minerva LumaTech&trade; V-Tee", "desc": "Don't be fooled by the simple design of our Minerva LumaTech&trade; V-Neck Tee. This classic training top features the same serious sweat-wicking technology as some of the more expensive tees in our lineup. &bull; Navy blue heather soft v-neck tee.&bull; Flatlock seams for chafe-free comfort.&bull; ", "category": "Tees", "top": "Clothing", "gender": "women", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/ws08-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/ws08-blue_back.jpg", "stock": 45, "colors": [], "sizes": [], "type": "variable" }, { "id": 1511, "sku": "WS12", "name": "Radiant Tee", "short": "This is a variable product called a Radiant Tee", "desc": "So light and comfy, you'll love the Radiant Tee's organic fabric, feel, performance and style. You may never want to stop moving in this shirt. &bull; Salmon soft scoop neck tee.&bull; Athletic, semi-form fit.&bull; Flat seams prevent chafing.&bull; 67% Organic Cotton / 28% Hemp / 5% Spandex.", "category": "Tees", "top": "Clothing", "gender": "women", "price": 22.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/ws12-orange_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/ws12-orange_back.jpg", "stock": 84, "colors": [], "sizes": [], "type": "variable" }, { "id": 433, "sku": "MS09", "name": "Ryker LumaTech&trade; Tee (Crew-neck)", "short": "This is a variable product called a Ryker LumaTech&trade; Tee (Crew-neck)", "desc": "The crew-neck Ryker LumaTech&trade; Tee hides premium performance technology beneath unassuming looks. The featherweight blend of fabrics wicks away moisture to keep you cool and dry in every phase of your active life. &bull; Royal polyester tee with black accents.&bull; Relaxed fit. &bull; Short-Sl", "category": "Tees", "top": "Clothing", "gender": "men", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/ms09-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/ms09-blue_alt1.jpg", "stock": 39, "colors": [], "sizes": [], "type": "variable" }, { "id": 529, "sku": "MS02", "name": "Ryker LumaTech&trade; Tee (V-neck)", "short": "This is a variable product called a Ryker LumaTech&trade; Tee (V-neck)", "desc": "Don't be fooled by its classic style; the Ryker LumaTech&trade; Tee embodies the future of performance apparel. Its featherweight blend of fabrics wicks away moisture to keep you cool and dry, whether racking up miles, hitting three pointers or strutting the boardwalk. &bull; Relaxed fit. &bull; Sho", "category": "Tees", "top": "Clothing", "gender": "men", "price": 28.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/ms02-gray_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/ms02-gray_alt1.jpg", "stock": 217, "colors": [], "sizes": [], "type": "variable" }, { "id": 577, "sku": "MS08", "name": "Strike Endurance Tee", "short": "This is a variable product called a Strike Endurance Tee", "desc": "While grit and purpose keep you going, it helps to have a little extra comfort, too. Our Strike Long-Sleeve Endurance Tee helps ensures a photo-ready finish with advanced sweat-wicking technology for a cool, dry feel. &bull; Loose fit. &bull; Ribbed cuffs/collar. &bull; Machine wash/dry.", "category": "Tees", "top": "Clothing", "gender": "men", "price": 39.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/ms08-black_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/ms08-black_back.jpg", "stock": 124, "colors": [], "sizes": [], "type": "variable" }, { "id": 1463, "sku": "WS09", "name": "Tiffany Fitness Tee", "short": "This is a variable product called a Tiffany Fitness Tee", "desc": "You'll work out and look cute doing it in the short-sleeve Tiffany Fitness Tee. The longer length and fitted cut offer a smoother silhouette, while scoop neck pattern and drape effects lend a trendy touch. &bull; Teal soft scoop neck tee.&bull; Contrast stitching pattern. &bull; Machine wash/dry.", "category": "Tees", "top": "Clothing", "gender": "women", "price": 28.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/s/ws09-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/s/ws09-blue_back.jpg", "stock": 205, "colors": [], "sizes": [], "type": "variable" }, { "id": 497, "sku": "MS06", "name": "Zoltan Gym Tee", "short": "This is a variable product called a Zoltan Gym Tee", "desc": "This short-sleeve wonder works twice as hard to give you good gym days and good looks. The Zoltan Gym Tee helps you stay comfortable, while the looser sleeves and flatlock seams keep you moving in chafe-free comfort. &bull; Relaxed fit. &bull; Crew neckline. &bull; Machine wash/dry.", "category": "Tees", "top": "Clothing", "gender": "men", "price": 29.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/s/ms06-blue_main.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/s/ms06-blue_alt1.jpg", "stock": 42, "colors": [], "sizes": [], "type": "variable" }, { "id": 2039, "sku": "24-WB02", "name": "Compete Track Tote", "short": "This is a simple product called Compete Track Tote", "desc": "The Compete Track Tote holds a host of exercise supplies with ease. Stash your towel, jacket and street shoes inside. Tuck water bottles in easy-access external spaces. Perfect for trips to gym or yoga studio, with dual top handles for convenience to and from. Two-way zippers. Contrast detailing. W ", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/b/wb02-green-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/b/wb02-green-0.jpg", "stock": 220, "colors": [], "sizes": [], "type": "simple" }, { "id": 2033, "sku": "24-MB03", "name": "Crown Summit Backpack", "short": "This is a simple product called Crown Summit Backpack", "desc": "The Crown Summit Backpack is equally at home in a gym locker, study cube or a pup tent, so be sure yours is packed with books, a bag lunch, water bottles, yoga block, laptop, or whatever else you want in hand. Rugged enough for day hikes and camping trips, it has two large zippered compartments and ", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 38.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/b/mb03-black-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/b/mb03-black-0_alt1.jpg", "stock": 84, "colors": [], "sizes": [], "type": "simple" }, { "id": 2042, "sku": "24-WB03", "name": "Driven Backpack", "short": "This is a simple product called Driven Backpack", "desc": "School books, camp gear and yoga clothes get along just fine in the all-purpose Driven Backpack. Crafted with tough polyester ripstop fabric, it's outfitted with rubberized end panels and padded, adjustable shoulder straps. The roomy main compartment features molded foam pockets that host everything", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 36.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/b/wb03-purple-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/b/wb03-purple-0.jpg", "stock": 184, "colors": [], "sizes": [], "type": "simple" }, { "id": 2041, "sku": "24-WB06", "name": "Endeavor Daytrip Backpack", "short": "This is a simple product called Endeavor Daytrip Backpack", "desc": "With more room than it appears, the Endeavor Daytrip Backpack will hold a whole day's worth of books, binders and gym clothes. The spacious main compartment includes a dedicated laptop sleeve. Two other compartments offer extra storage space. Foam-padded adjustable shoulder straps. 900D polyester. O", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 33.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/b/wb06-red-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/b/wb06-red-0_alt1.jpg", "stock": 203, "colors": [], "sizes": [], "type": "simple" }, { "id": 2036, "sku": "24-MB02", "name": "Fusion Backpack", "short": "This is a simple product called Fusion Backpack", "desc": "With the Fusion Backpack strapped on, every trek is an adventure - even a bus ride to work. That's partly because two large zippered compartments store everything you need, while a front zippered pocket and side mesh pouches are perfect for stashing those little extras, in case you change your mind ", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 59.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/b/mb02-gray-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/b/mb02-blue-0.jpg", "stock": 212, "colors": [], "sizes": [], "type": "simple" }, { "id": 2037, "sku": "24-UB02", "name": "Impulse Duffle", "short": "This is a simple product called Impulse Duffle", "desc": "Good for beach trips, track meets, yoga retreats and more, the Impulse Duffle is the companion you'll want at your side. A large U-shaped opening makes packing a hassle-free affair, while a zippered interior pocket keeps jewelry and other small valuables safely tucked out of sight. Wheeled. Dual car", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 74.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/u/b/ub02-black-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/u/b/ub02-black-0.jpg", "stock": 201, "colors": [], "sizes": [], "type": "simple" }, { "id": 2031, "sku": "24-MB01", "name": "Joust Duffle Bag", "short": "This is a simple product called Joust Duffle Bag", "desc": "The sporty Joust Duffle Bag can't be beat - not in the gym, not on the luggage carousel, not anywhere. Big enough to haul a basketball or soccer ball and some sneakers with plenty of room to spare, it's ideal for athletes with places to go. Dual top handles. Adjustable shoulder strap. Full-length zi", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 34.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/b/mb01-blue-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/b/mb01-blue-0.jpg", "stock": 134, "colors": [], "sizes": [], "type": "simple" }, { "id": 2043, "sku": "24-WB07", "name": "Overnight Duffle", "short": "This is a simple product called Overnight Duffle", "desc": "For long weekends away, camping outings and business trips, the Overnight Duffle can't be beat. The dual handles make it a cinch to carry, while the durable waterproof exterior helps you worry less about the weather. With multiple organizational pockets inside and out, it's the perfect travel compan", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 45.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/b/wb07-brown-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/b/wb07-brown-0.jpg", "stock": 120, "colors": [], "sizes": [], "type": "simple" }, { "id": 2044, "sku": "24-WB04", "name": "Push It Messenger Bag", "short": "This is a simple product called Push It Messenger Bag", "desc": "The name says so, but the Push It Messenger Bag is much more than a busy commuter's tote. It's a closet away from home when you're pedaling from class or work to gym and back or home again. It's the perfect size and shape for laptop, folded clothes, even extra shoes. Adjustable crossbody strap. Top ", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 45.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/b/wb04-blue-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/b/wb04-blue-0.jpg", "stock": 59, "colors": [], "sizes": [], "type": "simple" }, { "id": 2035, "sku": "24-MB06", "name": "Rival Field Messenger", "short": "This is a simple product called Rival Field Messenger", "desc": "The Rival Field Messenger packs all your campus, studio or trail essentials inside a unique design of soft, textured leather - with loads of character to spare. Two exterior pockets keep all your smaller items handy, and the roomy interior offers even more space. Leather construction. Adjustable fab", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 45.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/b/mb06-gray-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/b/mb06-gray-0.jpg", "stock": 113, "colors": [], "sizes": [], "type": "simple" }, { "id": 2040, "sku": "24-WB05", "name": "Savvy Shoulder Tote", "short": "This is a simple product called Savvy Shoulder Tote", "desc": "Powerwalking to the gym or strolling to the local coffeehouse, the Savvy Shoulder Tote lets you stash your essentials in sporty style! A top-loading compartment provides quick and easy access to larger items, while zippered pockets on the front and side hold cash, credit cards and phone. Water-resis", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 32.0, "sale": 24.0, "img": "http://eimages.valtim.com/acme-images/product/w/b/wb05-red-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/b/wb05-red-0.jpg", "stock": 98, "colors": [], "sizes": [], "type": "simple" }, { "id": 2032, "sku": "24-MB04", "name": "Strive Shoulder Pack", "short": "This is a simple product called Strive Shoulder Pack", "desc": "Convenience is next to nothing when your day is crammed with action. So whether you're heading to class, gym, or the unbeaten path, make sure you've got your Strive Shoulder Pack stuffed with all your essentials, and extras as well. Zippered main compartment. Front zippered pocket. Side mesh pocket.", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/b/mb04-black-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/b/mb04-black-0_alt1.jpg", "stock": 216, "colors": [], "sizes": [], "type": "simple" }, { "id": 2038, "sku": "24-WB01", "name": "Voyage Yoga Bag", "short": "This is a simple product called Voyage Yoga Bag", "desc": "Everything you need for a trip to the gym will fit inside this surprisingly spacious Voyage Yoga Bag. Stock it with a water bottle, change of clothes, pair of shoes, and even a few beauty products. Fits inside a locker and zips shut for security. Slip pocket on front. Contrast piping. Durable nylon ", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/b/wb01-black-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/b/wb01-black-0.jpg", "stock": 33, "colors": [], "sizes": [], "type": "simple" }, { "id": 2034, "sku": "24-MB05", "name": "Wayfarer Messenger Bag", "short": "This is a simple product called Wayfarer Messenger Bag", "desc": "Perfect for class, work or the gym, the Wayfarer Messenger Bag is packed with pockets. The dual-buckle flap closure reveals an organizational panel, and the roomy main compartment has spaces for your laptop and a change of clothes. An adjustable shoulder strap and easy-grip handle promise easy carry", "category": "Bags", "top": "Gear", "gender": "unisex", "price": 45.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/b/mb05-black-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/b/mb05-black-0.jpg", "stock": 217, "colors": [], "sizes": [], "type": "simple" }, { "id": 2022, "sku": "24-UG06", "name": "Affirm Water Bottle", "short": "This is a simple product called Affirm Water Bottle", "desc": "You'll stay hydrated with ease with the Affirm Water Bottle by your side or in hand. Measurements on the outside help you keep track of how much you're drinking, while the screw-top lid prevents spills. A metal carabiner clip allows you to attach it to the outside of a backpack or bag for easy acces", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 7.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/u/g/ug06-lb-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/u/g/ug06-lb-0.jpg", "stock": 99, "colors": [], "sizes": [], "type": "simple" }, { "id": 2023, "sku": "24-UG07", "name": "Dual Handle Cardio Ball", "short": "This is a simple product called Dual Handle Cardio Ball", "desc": "Make the most of your limited workout window with our Dual-Handle Cardio Ball. The 15-lb ball maximizes the effort-impact to your abdominal, upper arm and lower-body muscles. It features a handle on each side for a firm, secure grip. Durable plastic shell with sand fill. Two handles. 15 lbs.", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 12.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/u/g/ug07-bk-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/u/g/ug07-bk-0_alt1.jpg", "stock": 78, "colors": [], "sizes": [], "type": "simple" }, { "id": 2026, "sku": "24-UG05", "name": "Go-Get'r Pushup Grips", "short": "This is a simple product called Go-Get'r Pushup Grips", "desc": "The Go-Get'r Pushup Grips safely provide the extra range of motion you need for a deep-dip routine targeting core, shoulder, chest and arm strength. Do fewer pushups using more energy, getting better results faster than the standard floor-level technique yield. Durable foam grips. Supportive base.", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 19.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/u/g/ug05-gr-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/u/g/ug05-gr-0.jpg", "stock": 21, "colors": [], "sizes": [], "type": "simple" }, { "id": 2030, "sku": "24-UG03", "name": "Harmony Lumaflex&trade; Strength Band Kit", "short": "This is a simple product called Harmony Lumaflex&trade; Strength Band Kit", "desc": "Forget fancy machines and costly memberships - the Harmony Lumaflex&trade; Stength Band Kit is all you need for an amazing workout. The kit has everything you need for a range of strengthening and toning exercises. Three flex bands. Textured, ergonomic grips. Adjustable lengths. Mesh carry bag inclu", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 22.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/u/g/ug03-bk-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/u/g/ug03-bk-0.jpg", "stock": 210, "colors": [], "sizes": [], "type": "simple" }, { "id": 2025, "sku": "24-UG02", "name": "Pursuit Lumaflex&trade; Tone Band", "short": "This is a simple product called Pursuit Lumaflex&trade; Tone Band", "desc": "Save your knees and joints while strengthening arms, legs and core with the Pursuit Lumaflex&trade; Tone Band. This ultra-durable training tool lets you complete a full-body workout free of bulky machines or free weights. Home and/or gym use. Ergonomic handles.", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 16.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/u/g/ug02-bk-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/u/g/ug02-bk-0.jpg", "stock": 165, "colors": [], "sizes": [], "type": "simple" }, { "id": 2027, "sku": "24-UG01", "name": "Quest Lumaflex&trade; Band", "short": "This is a simple product called Quest Lumaflex&trade; Band", "desc": "From leg extensions and standing squats to a full range of floor routines, the Quest Strength Band brings your own personal gym with you everywhere you go. Designed to add extra resistance to any training session, this strength band helps you tone and define your muscles. You'll love the extra comfo", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 19.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/u/g/ug01-bk-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/u/g/ug01-bk-0.jpg", "stock": 127, "colors": [], "sizes": [], "type": "simple" }, { "id": 2029, "sku": "24-WG088", "name": "Sprite Foam Roller", "short": "This is a simple product called Sprite Foam Roller", "desc": "It hurts so good to use the Sprite Foam Roller on achy, tired muscles for myofascial massage therapy. Or you can add this fundamental piece to your Pilates and yoga accouterment, or apply towards core stability, strengthening and balance exercise. 6'' wide by 12'' long. Safe for myofascial release. ", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 19.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-foam-roller.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-foam-roller.jpg", "stock": 40, "colors": [], "sizes": [], "type": "simple" }, { "id": 2028, "sku": "24-WG084", "name": "Sprite Foam Yoga Brick", "short": "This is a simple product called Sprite Foam Yoga Brick", "desc": "Our top-selling yoga prop, the 4-inch, high-quality Sprite Foam Yoga Brick is popular among yoga novices and studio professionals alike. An essential yoga accessory, the yoga brick is a critical tool for finding balance and alignment in many common yoga poses. Choose from 5 color options. Standard L", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 5.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-brick.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-brick.jpg", "stock": 128, "colors": [], "sizes": [], "type": "simple" }, { "id": 2013, "sku": "24-WG081-gray", "name": "Sprite Stasis Ball 55 cm", "short": "This is a simple product called Sprite Stasis Ball 55 cm", "desc": "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. Durable, b", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 23.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-gray.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-gray.jpg", "stock": 158, "colors": [], "sizes": [], "type": "simple" }, { "id": 2014, "sku": "24-WG081-pink", "name": "Sprite Stasis Ball 55 cm", "short": "This is a simple product called Sprite Stasis Ball 55 cm", "desc": "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. Durable, b", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 23.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-pink.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-pink.jpg", "stock": 68, "colors": [], "sizes": [], "type": "simple" }, { "id": 2015, "sku": "24-WG081-blue", "name": "Sprite Stasis Ball 55 cm", "short": "This is a simple product called Sprite Stasis Ball 55 cm", "desc": "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. Durable, b", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 23.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball.jpg", "stock": 182, "colors": [], "sizes": [], "type": "simple" }, { "id": 2016, "sku": "24-WG082-gray", "name": "Sprite Stasis Ball 65 cm", "short": "This is a simple product called Sprite Stasis Ball 65 cm", "desc": "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. 65 cm plas", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 27.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-gray.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-gray.jpg", "stock": 72, "colors": [], "sizes": [], "type": "simple" }, { "id": 2017, "sku": "24-WG082-pink", "name": "Sprite Stasis Ball 65 cm", "short": "This is a simple product called Sprite Stasis Ball 65 cm", "desc": "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. 65 cm plas", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 27.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-pink.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-pink.jpg", "stock": 208, "colors": [], "sizes": [], "type": "simple" }, { "id": 2018, "sku": "24-WG082-blue", "name": "Sprite Stasis Ball 65 cm", "short": "This is a simple product called Sprite Stasis Ball 65 cm", "desc": "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. 65 cm plas", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 27.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball.jpg", "stock": 87, "colors": [], "sizes": [], "type": "simple" }, { "id": 2019, "sku": "24-WG083-gray", "name": "Sprite Stasis Ball 75 cm", "short": "This is a simple product called Sprite Stasis Ball 75 cm", "desc": "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. 75 cm plas", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-gray.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-gray.jpg", "stock": 239, "colors": [], "sizes": [], "type": "simple" }, { "id": 2020, "sku": "24-WG083-pink", "name": "Sprite Stasis Ball 75 cm", "short": "This is a simple product called Sprite Stasis Ball 75 cm", "desc": "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. 75 cm plas", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-pink.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-pink.jpg", "stock": 136, "colors": [], "sizes": [], "type": "simple" }, { "id": 2021, "sku": "24-WG083-blue", "name": "Sprite Stasis Ball 75 cm", "short": "This is a simple product called Sprite Stasis Ball 75 cm", "desc": "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. 75 cm plas", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 32.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball.jpg", "stock": 103, "colors": [], "sizes": [], "type": "simple" }, { "id": 2012, "sku": "24-WG087", "name": "Sprite Yoga Strap 10 foot", "short": "This is a simple product called Sprite Yoga Strap 10 foot", "desc": "The Sprite Yoga Strap is your untiring partner in demanding stretches, holds and alignment routines. The strap's 100% organic cotton fabric is woven tightly to form a soft, textured yet non-slip surface. The plastic clasp buckle is easily adjustable, lightweight and durable under strain. 10' long x ", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 21.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-strap.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-strap.jpg", "stock": 149, "colors": [], "sizes": [], "type": "simple" }, { "id": 2010, "sku": "24-WG085", "name": "Sprite Yoga Strap 6 foot", "short": "This is a simple product called Sprite Yoga Strap 6 foot", "desc": "The Sprite Yoga Strap is your untiring partner in demanding stretches, holds and alignment routines. The strap's 100% organic cotton fabric is woven tightly to form a soft, textured yet non-slip surface. The plastic clasp buckle is easily adjustable, lightweight and durable under strain. 100% soft a", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 14.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-strap.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-strap.jpg", "stock": 48, "colors": [], "sizes": [], "type": "simple" }, { "id": 2011, "sku": "24-WG086", "name": "Sprite Yoga Strap 8 foot", "short": "This is a simple product called Sprite Yoga Strap 8 foot", "desc": "The Sprite Yoga Strap is your untiring partner in demanding stretches, holds and alignment routines. The strap's 100% organic cotton fabric is woven tightly to form a soft, textured yet non-slip surface. The plastic clasp buckle is easily adjustable, lightweight and durable under strain. 8' long x 1", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 17.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-strap.jpg", "img2": "http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-strap.jpg", "stock": 66, "colors": [], "sizes": [], "type": "simple" }, { "id": 2024, "sku": "24-UG04", "name": "Zing Jump Rope", "short": "This is a simple product called Zing Jump Rope", "desc": "One of the world's simplest and most portable exercise devices, a jump rope enables endless variations and fitness output. The Zing Jump Rope goes anywhere and can be used any time. It is adjustable in length and has contoured foam handles for a great grip. Contoured foam handles. Adjustable length.", "category": "Fitness Equipment", "top": "Gear", "gender": "unisex", "price": 12.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/u/g/ug04-bk-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/u/g/ug04-bk-0.jpg", "stock": 198, "colors": [], "sizes": [], "type": "simple" }, { "id": 2001, "sku": "24-MG04", "name": "Aim Analog Watch", "short": "This is a simple product called Aim Analog Watch", "desc": "Stay light-years ahead of the competition with our Aim Analog Watch. The flexible, rubberized strap is contoured to conform to the shape of your wrist for a comfortable all-day fit. The face features three illuminated hands, a digital read-out of the current time, and stopwatch functions. Japanese q", "category": "Watches", "top": "Gear", "gender": "unisex", "price": 45.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/g/mg04-bk-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/g/mg04-bk-0.jpg", "stock": 163, "colors": [], "sizes": [], "type": "simple" }, { "id": 2007, "sku": "24-WG01", "name": "Bolo Sport Watch", "short": "This is a simple product called Bolo Sport Watch", "desc": "The Bolo Sport Watch is sleek, sporty and sized just right to fit your smaller wrist. Easy to read and set up, it features a large digital face and button-activated alarm and stopwatch. The soft-touch resin band promises no-pinch comfort, while the water-resistant design lets you take your workout t", "category": "Watches", "top": "Gear", "gender": "unisex", "price": 49.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/g/wg01-bk-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/g/wg01-bk-0.jpg", "stock": 126, "colors": [], "sizes": [], "type": "simple" }, { "id": 2008, "sku": "24-WG03", "name": "Clamber Watch", "short": "This is a simple product called Clamber Watch", "desc": "Keep track of time on the treadmill or trail with our Clamber Watch. The flexible, rubberized strap is contoured to conform to your wrist for a comfortable fit all day. The face features an illuminated digital read-out of the current time and stopwatch functions. Japanese quartz movement. Strap fits", "category": "Watches", "top": "Gear", "gender": "unisex", "price": 54.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/g/wg03-gr-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/g/wg03-sa-0.jpg", "stock": 111, "colors": [], "sizes": [], "type": "simple" }, { "id": 2004, "sku": "24-MG05", "name": "Cruise Dual Analog Watch", "short": "This is a simple product called Cruise Dual Analog Watch", "desc": "Whether you're traveling or wish you were, you'll never let time zones perplex you again with the Cruise Dual Analog Watch. The thick, adjustable band promises a comfortable, personalized fit to this classy, modern time piece. Two dials. Stainless steel case. Adjustable leather band.", "category": "Watches", "top": "Gear", "gender": "unisex", "price": 55.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/g/mg05-br-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/g/mg05-br-0.jpg", "stock": 148, "colors": [], "sizes": [], "type": "simple" }, { "id": 2005, "sku": "24-MG02", "name": "Dash Digital Watch", "short": "This is a simple product called Dash Digital Watch", "desc": "The Dash Digital Watch will challenge you to push harder and longer. Log workouts by date, average, and segment times, and recharge by setting hydration and nutrition alarms. This watch is styled with a sleek, square face and durable rubber strap for a long life. Digital display. LED backlight. Rubb", "category": "Watches", "top": "Gear", "gender": "unisex", "price": 92.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/g/mg02-bk-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/g/mg02-bk-0.jpg", "stock": 16, "colors": [], "sizes": [], "type": "simple" }, { "id": 2009, "sku": "24-WG02", "name": "Didi Sport Watch", "short": "This is a simple product called Didi Sport Watch", "desc": "The Didi Sport Watch helps you keep your workout plan down to the second. The vertical, digital face looks sleek and futuristic. This watch is programmed with tons of helpful features such as a timer, an alarm clock, a pedometer, and more to help make your excercise more productive. Digital display.", "category": "Watches", "top": "Gear", "gender": "unisex", "price": 92.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/g/wg02-bk-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/g/wg02-bk-0.jpg", "stock": 44, "colors": [], "sizes": [], "type": "simple" }, { "id": 2002, "sku": "24-MG01", "name": "Endurance Watch", "short": "This is a simple product called Endurance Watch", "desc": "It's easy to track and monitor your training progress with the Endurance Watch. You'll see standard info like time, date and day of the week, but it also functions for the serious high-mileage athete: lap counter, stopwatch, distance, heart rate, speed/pace, cadence and altitude. Digital display. LE", "category": "Watches", "top": "Gear", "gender": "unisex", "price": 49.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/g/mg01-bk-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/g/mg01-bk-0.jpg", "stock": 218, "colors": [], "sizes": [], "type": "simple" }, { "id": 2006, "sku": "24-WG09", "name": "Luma Analog Watch", "short": "This is a simple product called Luma Analog Watch", "desc": "Designed to stand up to your active lifestyle, this women's Luma Analog Watch features a tasteful brushed chrome finish and a stainless steel, water-resistant construction for lasting durability. Precision Miyota&reg; three-hand movement.", "category": "Watches", "top": "Gear", "gender": "unisex", "price": 43.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/w/g/wg09-gr-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/w/g/wg09-gr-0.jpg", "stock": 11, "colors": [], "sizes": [], "type": "simple" }, { "id": 2003, "sku": "24-MG03", "name": "Summit Watch", "short": "This is a simple product called Summit Watch", "desc": "Trek high and low in the attractive Summit Watch, which features a digital LED display with time and date, stopwatch, lap counter, and 3-second backlight. It can also calculate the number of steps taken and calories burned. Brushed metal case. Water resistant (100 meters). Buckle clasp. Strap fits 7", "category": "Watches", "top": "Gear", "gender": "unisex", "price": 54.0, "sale": 0, "img": "http://eimages.valtim.com/acme-images/product/m/g/mg03-br-0.jpg", "img2": "http://eimages.valtim.com/acme-images/product/m/g/mg03-br-0.jpg", "stock": 93, "colors": [], "sizes": [], "type": "simple" }];
import pandas as pd 
df = pd.read_csv(r"P:\AI_Empire\Projects\voice-commerce-agent\tests\wc-product-export-7-4-2026-1775552442494.csv")
csv_products = df.to_dict("records")
from rapidfuzz import fuzz

print(csv_products)
def find_match(product, csv_products, threshold=85):

    best_match = None
    best_score = 0

    for csv_p in csv_products:
        import re

        def normalize(text):
            text = text.lower()
            text = re.sub(r'[^a-z0-9 ]', '', text)
            return text
        score = fuzz.token_sort_ratio(
            normalize(product["name"]),
            normalize(csv_p["Name"])
)

        if score > best_score:
            best_score = score
            best_match = csv_p

    if best_score >= threshold:
        return best_match

    return None
count = 0
print ("Before products", PRODUCTS)
for product in PRODUCTS:

    match = find_match(product, csv_products)

    if match:
        count +=1
        print(product["id"] ,'-->', match["ID"])
        product["id"] = match["ID"]
print("Products After" , PRODUCTS)
print(count)

In [74]:
Products = [{'id': 1646, 'sku': 'WT08', 'name': 'Antonia Racer Tank', 'short': 'This is a variable product called a Antonia Racer Tank', 'desc': "You won't know what you like best about the Antonia Racer Tank: soft, stretchy, lightweight fabric? Super-cute colorblocked details? Whatever it is, this piece is sure to quickly move to the top of your workout rotation. &bull; Machine wash.&bull; Line dry.", 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 34.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/t/wt08-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/t/wt08-black_alt1.jpg', 'stock': 138, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1620, 'sku': 'WT01', 'name': 'Bella Tank', 'short': 'This is a variable product called a Bella Tank', 'desc': "Style, performance and comfort mix it up in the Bella Tank. With striking color block contrast, fitted form and a built-in bra, you'll be supported and stylish at the same time. &bull; Navy blue tank top - cotton.&bull; Feminine scoop neckline.&bull; Power mesh lining in shelf bra for superior suppo", 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/t/wt01-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/t/wt01-blue_back.jpg', 'stock': 39, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1647, 'sku': 'WT09', 'name': 'Breathe-Easy Tank', 'short': 'This is a variable product called a Breathe-Easy Tank', 'desc': "The Breathe Easy Tank is so soft, lightweight, and comfortable, you won't even know it's there -- until its high-tech Cocona&reg; fabric starts wicking sweat away from your body to help you stay dry and focused. Layer it over your favorite sports bra and get moving. &bull; Machine wash/dry.&bull; Co", 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 34.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/t/wt09-white_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/t/wt09-white_back.jpg', 'stock': 161, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1371, 'sku': 'WB03', 'name': 'Celeste Sports Bra', 'short': 'This is a variable product called a Celeste Sports Bra', 'desc': "Whatever your goals for the day's workout, the Celeste Sports Bra lets you do it in comfort and coolness, plus enhanced support and shaping. A power mesh back zone and moisture-wicking fabric ensure you stay dry. &bull; Mint bra top. &bull; Seam-free interior molded cups&bull; Odor control. &bull; U", 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 39.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/b/wb03-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/b/wb03-green_alt1.jpg', 'stock': 140, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1625, 'sku': 'WT06', 'name': 'Chloe Compete Tank', 'short': 'This is a variable product called a Chloe Compete Tank', 'desc': "You've earned your figure, so stay cool and let it do all the talking in the form-fitted, sleek racerback Chloe Compete Tank. Designed for total range of motion and performance, the Nona is made with highly breathable mesh fabric. &bull; Royal blue tank top - nylon/spandex.&bull; Flatlock stitching.", 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 39.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/t/wt06-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/t/wt06-blue_back.jpg', 'stock': 157, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1369, 'sku': 'WB01', 'name': 'Electra Bra Top', 'short': 'This is a variable product called a Electra Bra Top', 'desc': 'A heavenly soft and stylish eco garment, the Electra Bra Top is perfect for wearing on its own as a hot yoga top or layered under a tank. &bull; Gray rouched bra top.&bull; Attractive back straps feature contrasting motif fabric.&bull; Interior bra top is lined with breathable mesh.&bull; Elastic un', 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 39.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/b/wb01-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/b/wb01-gray_alt1.jpg', 'stock': 56, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1370, 'sku': 'WB02', 'name': 'Erica Evercool Sports Bra', 'short': 'This is a variable product called a Erica Evercool Sports Bra', 'desc': 'Perfect for medium-impact workouts, our Erica EverCool&trade; Sports Bra brings a brilliant combo of comfort and style. Moisture-wicking technology keeps you dry, and the flattering hybrid racerback promises an unbeatable range of motion. &bull; Honeycomb light blue bra top.&bull; Elastic hem.&bull;', 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 39.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/b/wb02-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/b/wb02-blue_alt1.jpg', 'stock': 203, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1624, 'sku': 'WT05', 'name': 'Leah Yoga Top', 'short': 'This is a variable product called a Leah Yoga Top', 'desc': 'The Leah Yoga Top offers a practical, comfortable upper that will not compromise your style. Body hugging fit and interior shelf bra make it suitable for active or leisure pursuits. &bull; Blue heather rouched tank top.&bull; Camisole tank top.&bull; Banding and shirring details.&bull; Body hugging ', 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 39.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/t/wt05-purple_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/t/wt05-purple_alt1.jpg', 'stock': 137, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1373, 'sku': 'WB05', 'name': 'Lucia Cross-Fit Bra', 'short': 'This is a variable product called a Lucia Cross-Fit Bra', 'desc': 'Being cool is a big part of being comfy, which is why the Lucia Cross-Fit Bra features moisture-wicking technology as well as soft, lightweight fabric. &bull; Black/white bra top. &bull; Criss-cross back design. &bull; Machine wash/dry.', 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 39.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/b/wb05-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/b/wb05-black_back.jpg', 'stock': 43, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1626, 'sku': 'WT07', 'name': 'Maya Tunic', 'short': 'This is a variable product called a Maya Tunic', 'desc': 'With abutted seams and moisture wicking capacity, the Maya Tunic lets you work out in complete comfort. &bull; Mint green heather tunic-style tank. &bull; Wrapped back with cut out detail.&bull; Drawcord detail at end.&bull; Abutted seams.', 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/t/wt07-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/t/wt07-green_alt1.jpg', 'stock': 64, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1623, 'sku': 'WT04', 'name': 'Nona Fitness Tank', 'short': 'This is a variable product called a Nona Fitness Tank', 'desc': "It doesn't matter if your goal is 5 miles or an hour of Vinyasa, because our Nona Fitness Tank does it all. You don't have to sacrifice either -- this v-neck top features smooth, chafe-free seams and a breathable mesh back insert. Cute, too. &bull; Blue/white striped mesh tank.&bull; Relaxed fit. &b", 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 39.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/t/wt04-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/t/wt04-blue_back.jpg', 'stock': 58, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1622, 'sku': 'WT03', 'name': 'Nora Practice Tank', 'short': 'This is a variable product called a Nora Practice Tank', 'desc': 'A closet go-to, the Nora Practice Tank can be worn at or after workouts or beneath a baggy tee for a laid-back style. Perfect fit and very comfortable! &bull; Pink stripped tank with side rouching.&bull; Pre-shrunk.&bull; Garment dyed.&bull; 92% Organic Cotton/8% Lycra.', 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 39.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/t/wt03-red_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/t/wt03-red_alt1.jpg', 'stock': 77, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1372, 'sku': 'WB04', 'name': 'Prima Compete Bra Top', 'short': 'This is a variable product called a Prima Compete Bra Top', 'desc': "Pull on the Prima Compete Bra Top and you'll love the fabric: soft, stretchy, and ultra lightweight. But you'll also love the racerback cut, for freer movement through all your athletic feats. `&bull; Colorblocked details.&bull; Machine wash/line dry.", 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 24.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/b/wb04-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/b/wb04-blue-0.jpg', 'stock': 9, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1621, 'sku': 'WT02', 'name': 'Zoe Tank', 'short': 'This is a variable product called a Zoe Tank', 'desc': 'The Zoe Tank leads with eye-catching fitness-pop looks, keeping your studio style uptrend. The inner shelf bra and soft, breathable fabric create a tank top that looks, feels and fits great. &bull; Salmon heather tank top.&bull; 1" elastic band on inner bra.&bull; Mesh lining on shelf bra for suppor', 'category': 'Bras & Tanks', 'top': 'Clothing', 'gender': 'women', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/t/wt02-orange_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/t/wt02-orange_back.jpg', 'stock': 152, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1169, 'sku': 'MH09', 'name': 'Abominable Hoodie', 'short': 'This is a variable product called a Abominable Hoodie', 'desc': "It took CoolTech&trade; weather apparel know-how and lots of wind-resistant fabric to get the Abominable Hoodie just right. It's aggressively warm when it needs to be, while maintaining your comfort in milder climes. &bull; Blue heather hoodie.&bull; Relaxed fit. &bull; Moisture-wicking. &bull; Mach", 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 69.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh09-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh09-blue_alt1.jpg', 'stock': 189, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1172, 'sku': 'MH12', 'name': 'Ajax Full-Zip Sweatshirt', 'short': 'This is a variable product called a Ajax Full-Zip Sweatshirt', 'desc': 'The Ajax Full-Zip Sweatshirt makes the optimal layering or outer piece for archers, golfers, hikers and virtually any other sportsmen. Not only does it have top-notch moisture-wicking abilities, but the tight-weave fabric also prevents pilling from repeated wash-and-wear cycles. &bull; Mint striped ', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 69.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh12-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh12-green_alt1.jpg', 'stock': 139, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1409, 'sku': 'WH09', 'name': 'Ariel Roll Sleeve Sweatshirt', 'short': 'This is a variable product called a Ariel Roll Sleeve Sweatshirt', 'desc': 'Soft, sleek and subtle, the Ariel Roll Sleeve Sweatshirt is a nuanced fitness garment for all occasions. It works equally well as a workout piece or in a casual social setting. &bull; Purple two-tone lightweight hoodie. &bull; 100% cotton.&bull; Adjustable roll sleeves for Long-Sleeve or 3/4 sleeve.', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'women', 'price': 39.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/h/wh09-purple_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/h/wh09-purple_alt1.jpg', 'stock': 209, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1403, 'sku': 'WH03', 'name': 'Autumn Pullie', 'short': 'This is a variable product called a Autumn Pullie', 'desc': 'With ultra-soft fleece fabric and an athletic design, our short-sleeve Autumn Pullie delivers a comfortable fit that makes it an everyday essential. A luxurious roll neck protects you from elements. &bull; Cayenne Short-Sleeve roll neck sweatshirt. &bull; Relaxed fit. &bull; Short-Sleeves. &bull; Ma', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'women', 'price': 57.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/h/wh03-red_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/h/wh03-red_alt1.jpg', 'stock': 139, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1163, 'sku': 'MH03', 'name': 'Bruno Compete Hoodie', 'short': 'This is a variable product called a Bruno Compete Hoodie', 'desc': "Stay comfortable and stay in the race no matter what the weather's up to. The Bruno Compete Hoodie's water-repellent exterior shields you from the elements, while advanced fabric technology inside wicks moisture to keep you dry. &bull; Full zip black hoodie pullover. &bull; Adjustable drawstring hoo", 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 63.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh03-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh03-black_alt1.jpg', 'stock': 167, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1408, 'sku': 'WH08', 'name': 'Cassia Funnel Sweatshirt', 'short': 'This is a variable product called a Cassia Funnel Sweatshirt', 'desc': 'The casual Cassia Funnel Sweatshirt sports front angled pockets and a drawstring stretch funnel hoodie. A cowl front neck pairs with a tagless label at back neck. &bull; White full zip hoodie with gray detail.&bull; 65% Cotton/28% Nylon/7% Spandex.&bull; Front slash pockets.&bull; Tagless label at b', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'women', 'price': 48.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/h/wh08-white_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/h/wh08-white_alt1.jpg', 'stock': 5, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1161, 'sku': 'MH01', 'name': 'Chaz Kangeroo Hoodie', 'short': 'This is a variable product called a Chaz Kangeroo Hoodie', 'desc': 'Ideal for cold-weather training or work outdoors, the Chaz Hoodie promises superior warmth with every wear. Thick material blocks out the wind as ribbed cuffs and bottom band seal in body heat. &bull; Two-tone gray heather hoodie.&bull; Drawstring-adjustable hood. &bull; Machine wash/dry.', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 52.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh01-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh01-gray_alt1.jpg', 'stock': 149, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1433, 'sku': 'WH12', 'name': 'Circe Hooded Ice Fleece', 'short': 'This is a variable product called a Circe Hooded Ice Fleece', 'desc': 'Keep shivers at bay with the Circe Hooded Ice Fleece. Ultra-thick, high-pile fleece traps your body heat and keeps cold outside. The drawstring hood and ribbed details add extra coziness and touches of classic style. Its relaxed fit is loose enough to layer comfortably over your shirt. Full-zip fron', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'women', 'price': 68.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/h/wh12-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/h/wh12-gray_back.jpg', 'stock': 110, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1406, 'sku': 'WH06', 'name': 'Daphne Full-Zip Hoodie', 'short': 'This is a variable product called a Daphne Full-Zip Hoodie', 'desc': 'The Daphne Hoodie is an attractive yet rugged layering or outer piece, especially for chilly days en route the studio, doing yardwork or enjoying outdoor recreation. &bull; Purple full zip hoodie with pink accents. &bull; Heather texture.&bull; 4-way stretch.&bull; Pre-shrunk.&bull; Hood lined in ve', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'women', 'price': 59.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/h/wh06-purple_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/h/wh06-purple_alt1.jpg', 'stock': 9, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1411, 'sku': 'WH11', 'name': 'Eos V-Neck Hoodie', 'short': 'This is a variable product called a Eos V-Neck Hoodie', 'desc': 'Getting chilly before class starts? Wear the Eos on your way to and from yoga for a cute and cozy warmup piece. Reach for its reliable comfort and enjoy a super-soft blend of fabrics finished in sporty style that includes a hidden kangaroo pocket. Semi-fitted. Long-Sleeve. Machine wash/line dry.', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'women', 'price': 54.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/h/wh11-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/h/wh11-blue_back.jpg', 'stock': 169, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1164, 'sku': 'MH04', 'name': 'Frankie  Sweatshirt', 'short': 'This is a variable product called a Frankie Sweatshirt', 'desc': 'The Frankie Sweatshirt is your best friend at long afternoon stadium stints or winter trailside campsites. The soft fleece fabric keeps you toasty as moisture-wicking technology kicks in when the sun comes out. &bull; Light green crewneck sweatshirt.&bull; Hand pockets.&bull; Relaxed fit.&bull; Mach', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 60.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh04-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh04-green_alt1.jpg', 'stock': 42, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1171, 'sku': 'MH11', 'name': 'Grayson Crewneck Sweatshirt', 'short': 'This is a variable product called a Grayson Crewneck Sweatshirt', 'desc': "The Grayson Crewneck Sweatshirt gives you that ageless, classic look &ndash; a style that comes back around nearly every season. What's more, its performance and temp-control values are always in vogue. &bull; Cream crewneck sweatshirt with black accents.&bull; 80% cotton/20% polyester fleece. &bull", 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 64.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh11-white_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh11-white_alt1.jpg', 'stock': 214, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1410, 'sku': 'WH10', 'name': 'Helena Hooded Fleece', 'short': 'This is a variable product called a Helena Hooded Fleece', 'desc': 'Wear this hoodie after the gym or before a chilly weather bike ride. Either way, this versatile sweatshirt offers an effortlessly appealing silhouette and a super-comfy fit. Smooth, stretchy fabric creates flattering shape, while the full-zip placket and hood help block gusty winds. Full zip. Banded', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'women', 'price': 55.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/h/wh10-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/h/wh10-gray_alt1.jpg', 'stock': 190, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1402, 'sku': 'WH02', 'name': 'Hera Pullover Hoodie', 'short': 'This is a variable product called a Hera Pullover Hoodie', 'desc': 'Get ready to rule the studio and dominate the yoga mat in the Hera Pullover Hoodie, a cozy yet classy look for any level of yogi. &bull; Teal with purple stiching.&bull; Hoodie pullover.&bull; Snug fit.', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'women', 'price': 48.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/h/wh02-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/h/wh02-blue_alt1.jpg', 'stock': 81, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1167, 'sku': 'MH07', 'name': 'Hero Hoodie', 'short': 'This is a variable product called a Hero Hoodie', 'desc': 'Gray and black color blocking sets you apart as the Hero Hoodie keeps you warm on the bus, campus or cold mean streets. Slanted outsize front pockets keep your style real . . . convenient. &bull; Full-zip gray and black hoodie.&bull; Ribbed hem.&bull; Standard fit.&bull; Drawcord hood cinch.&bull; W', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 54.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh07-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh07-gray_alt1.jpg', 'stock': 162, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1165, 'sku': 'MH05', 'name': 'Hollister Backyard Sweatshirt', 'short': 'This is a variable product called a Hollister Backyard Sweatshirt', 'desc': "Kick off your weekend in the Hollister Backyard Sweatshirt. Whether you're raking leaves or flipping burgers, this comfy layer blocks the bite of the crisp autumn air. Puffy thick from hood to hem, it traps heat against your core. &bull; Cream crewneck sweatshirt with navy sleeves/trim.&bull; Relaxe", 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 52.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh05-white_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh05-white_alt1.jpg', 'stock': 75, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1170, 'sku': 'MH10', 'name': 'Mach Street Sweatshirt', 'short': 'This is a variable product called a Mach Street Sweatshirt', 'desc': "From hard streets to asphalt track, the Mach Street Sweatshirt holds up to wear and wind and rain. An infusion of performance and stylish comfort, with moisture-wicking LumaTech&trade; fabric, it's bound to become an everyday part of your active lifestyle &bull; Navy heather crewneck sweatshirt.&bul", 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 62.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh10-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh10-blue_alt1.jpg', 'stock': 145, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1173, 'sku': 'MH13', 'name': 'Marco Lightweight Active Hoodie', 'short': 'This is a variable product called a Marco Lightweight Active Hoodie', 'desc': 'For cold-weather training or post-game layering, you need something more than a basic fleece. Our Marco Lightweight Active Hoodie brings both style and performance to the plate, court, or touchline. The smooth-faced, brushed-back fabric blocks wind and traps body heat, while integrated Cocona&reg; f', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 74.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh13-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh13-blue_alt1.jpg', 'stock': 108, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1404, 'sku': 'WH04', 'name': 'Miko Pullover Hoodie', 'short': 'This is a variable product called a Miko Pullover Hoodie', 'desc': "After quality gym time, put on the Miko Pullover Hoody and keep your body warm. You'll find it's fashionable enough for the streets, but comfy enough to relax in at home. &bull; Teal two-tone hoodie.&bull; Low scoop neckline.&bull; Adjustable hood drawstrings.&bull; Longer rounded hemline for extra ", 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'women', 'price': 69.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/h/wh04-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/h/wh04-blue_alt1.jpg', 'stock': 24, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1374, 'sku': 'WH01', 'name': 'Mona Pullover Hoodlie', 'short': 'This is a variable product called a Mona Pullover Hoodlie', 'desc': "Whether you're after energizing activity or eye-catching apparel, the Mona Pullover is what you want. You'll stay warm and look fashionable, wherever you are. &bull; Light green heathered hoodie. &bull; Long-Sleeve, pullover. &bull; Long elliptical hem for extra coverage. &bull; Deep button placket ", 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'women', 'price': 57.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/h/wh01-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/h/wh01-green_alt1.jpg', 'stock': 29, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1168, 'sku': 'MH08', 'name': 'Oslo Trek Hoodie', 'short': 'This is a variable product called a Oslo Trek Hoodie', 'desc': 'Chilly weather is just an excuse to throw on your toasty, handsome new Oslo Trek Hoodie. It features an adjustable drawstring hood and a kangaroo pocket for extra hand warmth. The ultra-soft, cozy lining will have you wishing for more brisk days. &bull; Brown hoodie with black detail.&bull; Pullover', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 42.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh08-brown_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh08-brown_alt1.jpg', 'stock': 144, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1407, 'sku': 'WH07', 'name': 'Phoebe Zipper Sweatshirt', 'short': 'This is a variable product called a Phoebe Zipper Sweatshirt', 'desc': "A sophisticated layer of warmth awaits you in our full-zip sweatshirt jacket. You'll reach for this one in any season to enjoy its sturdy exterior and plush interior. &bull; Gray full zip hoodie with yellow detail.&bull; Hand-warmer pockets. &bull; Zip MP3 pocket with outlet for earphones wire. &bul", 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'women', 'price': 59.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/h/wh07-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/h/wh07-gray_alt1.jpg', 'stock': 181, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1405, 'sku': 'WH05', 'name': 'Selene Yoga Hoodie', 'short': 'This is a variable product called a Selene Yoga Hoodie', 'desc': 'The Selene Yoga Hoodie gets you to and from the studio in semi form-fitted comfort. Snug, sleek contours add fashion to function in this free-moving yoga warm up sweatshirt. &bull; Ivory heather full zip 3/4 sleeve hoodie.&bull; Zip pocket at arm for convenient storage.&bull; 24.0" body length.&bull', 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'women', 'price': 42.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/h/wh05-white_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/h/wh05-white_back.jpg', 'stock': 74, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1166, 'sku': 'MH06', 'name': 'Stark Fundamental Hoodie', 'short': 'This is a variable product called a Stark Fundamental Hoodie', 'desc': "You don't need bells and whistles when performance speaks for itself. The full-zip Stark Fundamental Hoodie give just what you need. Hood and fleece lining keep you warm, while breathable fabric and wicking technology won't let you overheat. &bull; Navy specked full zip hoodie.&bull; Ribbed cuffs, b", 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 42.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh06-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh06-blue_alt1.jpg', 'stock': 146, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1162, 'sku': 'MH02', 'name': 'Teton Pullover Hoodie', 'short': 'This is a variable product called a Teton Pullover Hoodie', 'desc': "This Teton Pullover Hoodie gives you more than laid-back style. It's equipped with moisture-wicking fabric to keep light and dry inside, especially in chilly-weather workouts. An elasticized hem lets you move about freely. &bull; Black pullover hoodie.&bull; Soft, brushed interior. &bull; Front hand", 'category': 'Hoodies & Sweatshirts', 'top': 'Clothing', 'gender': 'men', 'price': 70.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/h/mh02-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/h/mh02-black_alt1.jpg', 'stock': 208, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1441, 'sku': 'WJ08', 'name': 'Adrienne Trek Jacket', 'short': 'This is a variable product called a Adrienne Trek Jacket', 'desc': "You're ready for a cross-country jog or a coffee on the patio in the Adrienne Trek Jacket. Its style is unique with stand collar and drawstrings, and it fits like a jacket should. &bull; gray 1/4 zip pullover.&bull; Comfortable, relaxed fit.&bull; Front zip for venting.&bull; Spacious, kangaroo pock", 'category': 'Jackets', 'top': 'Clothing', 'gender': 'women', 'price': 57.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/j/wj08-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/j/wj08-gray_alt1.jpg', 'stock': 174, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1436, 'sku': 'WJ03', 'name': 'Augusta Pullover Jacket', 'short': 'This is a variable product called a Augusta Pullover Jacket', 'desc': "It's hard to be uncomfortable in the Augusta Pullover Jacket with &frac14; zip. With an incredibly soft fleece lining and textured outer fabric, it offers reliable protection from the elements and a cozy fit as well. &bull; Pink half-zip pullover. &bull; Front pouch pockets. &bull; Fold-down collar.", 'category': 'Jackets', 'top': 'Clothing', 'gender': 'women', 'price': 57.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/j/wj03-red_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/j/wj03-red_alt1.jpg', 'stock': 182, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1174, 'sku': 'MJ01', 'name': 'Beaumont Summit Kit', 'short': 'This is a variable product called a Beaumont Summit Kit', 'desc': "The smooth nylon shell around the Beaumont Summit Kit combats wind, reinforced with a cold-fighting brushed fleece layer. The jacket is reversible, giving you a new look for the return trek. Ample pocket space rounds out this hiker's paradise package. &bull; Yellow full zip rain jacket.&bull; Full-z", 'category': 'Jackets', 'top': 'Clothing', 'gender': 'men', 'price': 42.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/j/mj01-yellow_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/j/mj01-yellow_alt1.jpg', 'stock': 61, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1175, 'sku': 'MJ02', 'name': 'Hyperion Elements Jacket', 'short': 'This is a variable product called a Hyperion Elements Jacket', 'desc': 'Boldly face high winds, frigid temps and stormy weather the whole winter through in the Hyperion Elements Jacket. LumaTech&trade; insulating technology helps maintain your core temperature and wick sweat. The smooth shell is water repellent and quilted to retain body heat. &bull; Lime 1/4 zip pullov', 'category': 'Jackets', 'top': 'Clothing', 'gender': 'men', 'price': 51.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/j/mj02-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/j/mj02-green_alt1.jpg', 'stock': 202, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1440, 'sku': 'WJ07', 'name': 'Inez Full Zip Jacket', 'short': 'This is a variable product called a Inez Full Zip Jacket', 'desc': "The Inez Full Zip jacket is more than a cute layer. It's a full-on tech wonder you'll use for outdoor, pre and post workout. You'll love how the highlighted zipper and sleeve safety trim protect and catch looks. &bull; Purple heather inset full zip jacket.&bull; Full zip hoodie.&bull; Contrast bindi", 'category': 'Jackets', 'top': 'Clothing', 'gender': 'women', 'price': 59.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/j/wj07-purple_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/j/wj07-purple_alt1.jpg', 'stock': 244, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1437, 'sku': 'WJ04', 'name': 'Ingrid Running Jacket', 'short': 'This is a variable product called a Ingrid Running Jacket', 'desc': 'The Ingrid Running Jacket combines sleek design and high performance with slim, contoured fit and moisture-wicking fabric. It features a full-zip construction and a collared neck to keep the elements out and body heat in. &bull; Slim fit. &bull; Moisture-wicking fabric. &bull; Two side pockets. &bul', 'category': 'Jackets', 'top': 'Clothing', 'gender': 'women', 'price': 84.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/j/wj04-white_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/j/wj04-white_alt1.jpg', 'stock': 28, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1463, 'sku': 'WJ09', 'name': 'Jade Yoga Jacket', 'short': 'This is a variable product called a Jade Yoga Jacket', 'desc': 'If only all your other jackets were as comfortable as the relaxed-fit Jade Yoga Jacket, perfect for use during stretching, biking to and from studio or strolling on breezy fall days. &bull; Seafoam 1/4 zip pullover with purple stitching.&bull; Lightweight, quick-drying, water-resistant construction.', 'category': 'Jackets', 'top': 'Clothing', 'gender': 'women', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/j/wj09-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/j/wj09-green_alt1.jpg', 'stock': 143, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1435, 'sku': 'WJ02', 'name': 'Josie Yoga Jacket', 'short': 'This is a variable product called a Josie Yoga Jacket', 'desc': 'When your near future includes yoga, the cozy comfort of the Josie Yoga Jacket gets your mind and body ready. Stretchy CoolTech&trade; fabric with zipper pockets makes this jacket the right gear for studio time or teatime after. &bull; Slate rouched neck pullover.&bull; Moisture-wicking fabric.&bull', 'category': 'Jackets', 'top': 'Clothing', 'gender': 'women', 'price': 56.25, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/j/wj02-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/j/wj02-gray_alt1.jpg', 'stock': 177, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1439, 'sku': 'WJ06', 'name': 'Juno Jacket', 'short': 'This is a variable product called a Juno Jacket', 'desc': "On colder-than-comfortable mornings, you'll love warming up in the Juno All-Ways Performanc Jacket, designed to compete with wind and chill. Built-in Cocona&reg; technology aids evaporation, while a special zip placket and stand-up collar keep your neck protected. &bull; Adjustable hood.&bull; Fleec", 'category': 'Jackets', 'top': 'Clothing', 'gender': 'women', 'price': 77.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/j/wj06-purple_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/j/wj06-purple_alt1.jpg', 'stock': 39, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1178, 'sku': 'MJ06', 'name': 'Jupiter All-Weather Trainer', 'short': 'This is a variable product called a Jupiter All-Weather Trainer', 'desc': 'Inclement climate be damned. With your breathable, water-resistant Jupiter All-Weather Trainer, you can focus on fuel and form and leave protection and comfort to us. The fabric is a special light fleece that wicks moisture and insulates at once. &bull; Relaxed fit. &bull; Hand pockets. &bull; Machi', 'category': 'Jackets', 'top': 'Clothing', 'gender': 'men', 'price': 56.99, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/j/mj06-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/j/mj06-blue_alt1.jpg', 'stock': 91, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1177, 'sku': 'MJ04', 'name': 'Kenobi Trail Jacket', 'short': 'This is a variable product called a Kenobi Trail Jacket', 'desc': 'Aside from sealed seams to keep moisture out and body heat in, the Kenobi Trail Vest is all about media convenience. Use your phone and music player without ever taking it out of its cozy compartment: an attached remote on the left chest lets you change songs, pause and raise/lower volume. &bull; Bl', 'category': 'Jackets', 'top': 'Clothing', 'gender': 'men', 'price': 47.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/j/mj04-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/j/mj04-black_alt1.jpg', 'stock': 201, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1180, 'sku': 'MJ08', 'name': 'Lando Gym Jacket', 'short': 'This is a variable product called a Lando Gym Jacket', 'desc': "The Lando Gym Jacket offers strategic ventilation at perspiration-prone areas, while moisture-wicking technology helps you stay dry. Side-seam pockets house your favorite media device, so you're plugged in when working out. &bull; Gray polyester/spandex full zip jacket with orange lining. &bull; Rig", 'category': 'Jackets', 'top': 'Clothing', 'gender': 'men', 'price': 99.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/j/mj08-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/j/mj08-gray_alt1.jpg', 'stock': 220, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1182, 'sku': 'MJ10', 'name': 'Mars HeatTech&trade; Pullover', 'short': 'This is a variable product called a Mars HeatTech&trade; Pullover', 'desc': 'The Mars HeatTech&trade; Jacket defense against winter climes so you can play offense in the park, on the trail or in the deck chair. Great for the ski lodge or stadium, it features a wind- and water-resistant outer shell with a draw-cord hood and deep pockets for storage. &bull; Red 1/4 zip pullove', 'category': 'Jackets', 'top': 'Clothing', 'gender': 'men', 'price': 66.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/j/mj10-red_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/j/mj10-red_alt1.jpg', 'stock': 113, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1176, 'sku': 'MJ03', 'name': 'Montana Wind Jacket', 'short': 'This is a variable product called a Montana Wind Jacket', 'desc': 'Light-as-a-feather wind protection for runners, walkers and outdoor fitness buffs, the Montana Wind Jacket can be stuffed into your pocket for portable protection. Its stylish, move-with-you design makes it especially versatile. Adjustable hood. Split pocket. Thumb holes. Machine wash/hang to dry.', 'category': 'Jackets', 'top': 'Clothing', 'gender': 'men', 'price': 49.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/j/mj03-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/j/mj03-black_alt1.jpg', 'stock': 44, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1464, 'sku': 'WJ10', 'name': 'Nadia Elements Shell', 'short': 'This is a variable product called a Nadia Elements Shell', 'desc': "Protect yourself from wind and rain in the stylish Nadia Elements Shell. It repels water using hydro-resistant fabric, with fleece lining that adds a touch of warmth. It's finished with bold contrast zippers, adjustable cuffs and a hood. &bull; Zippered front. &bull; Zippered side pockets. &bull; Dr", 'category': 'Jackets', 'top': 'Clothing', 'gender': 'women', 'price': 69.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/j/wj10-yellow_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/j/wj10-yellow_alt1.jpg', 'stock': 201, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1465, 'sku': 'WJ11', 'name': 'Neve Studio Dance Jacket', 'short': 'This is a variable product called a Neve Studio Dance Jacket', 'desc': "If you're constantly on the move, the Neve Studio Dance Jacket is for you. It's not just for dance, either, with a tight fit that works as a mid-layer. The reversible design makes it even more versatile. &bull; Bright blue 1/4 zip pullover.&bull; CoolTech&trade; liner is sweat-wicking.&bull; Sleeve ", 'category': 'Jackets', 'top': 'Clothing', 'gender': 'women', 'price': 69.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/j/wj11-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/j/wj11-blue_alt1.jpg', 'stock': 94, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1466, 'sku': 'WJ12', 'name': 'Olivia 1/4 Zip Light Jacket', 'short': 'This is a variable product called a Olivia 1/4 Zip Light Jacket', 'desc': 'Running errands or headed to the gym, you just want to be comfortable. The Olivia Light Jacket promises that, plus a laid-back look. This zip-up is designed with shoulder stripes for an athletic touch, and banded waist and contoured seams for a flattering silhouette. &bull; Loose fit.&bull; Reflecti', 'category': 'Jackets', 'top': 'Clothing', 'gender': 'women', 'price': 77.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/j/wj12-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/j/wj12-blue_alt1.jpg', 'stock': 191, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1179, 'sku': 'MJ07', 'name': 'Orion Two-Tone Fitted Jacket', 'short': 'This is a variable product called a Orion Two-Tone Fitted Jacket', 'desc': "While you're getting fit, you need a fitted jacket to match. Striking color blocking patterns on hood, shoulders and arms are hallmarks of the Orion Two-Tone Fitted Jacket. They provide eye-catching contrast against the rich, torso tones of this 100% polyester, moisture-wicking essential. &bull; Red", 'category': 'Jackets', 'top': 'Clothing', 'gender': 'men', 'price': 72.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/j/mj07-red_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/j/mj07-red_alt1.jpg', 'stock': 123, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1184, 'sku': 'MJ12', 'name': 'Proteus Fitness Jackshirt', 'short': 'This is a variable product called a Proteus Fitness Jackshirt', 'desc': 'Part jacket, part shirt, the Proteus Fitness Jackshirt makes an ideal companion for outdoor training, camping or loafing on crisp days. Natural Cocona&reg; technology brings breathable comfort and increased dryness along with UV protection and odor management. The drop-tail hem provides extra covera', 'category': 'Jackets', 'top': 'Clothing', 'gender': 'men', 'price': 45.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/j/mj12-orange_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/j/mj12-orange_alt1.jpg', 'stock': 248, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1438, 'sku': 'WJ05', 'name': 'Riona Full Zip Jacket', 'short': 'This is a variable product called a Riona Full Zip Jacket', 'desc': 'The Riona Basic Zip Jacket makes the perfect extra layer for cold-weather workouts. It features amazing breathability and moisture management, but full-length zipper lets you moderate your core temperature even more. &bull; Brown heather full zip rouched jacket.&bull; Side hand pockets for extra sto', 'category': 'Jackets', 'top': 'Clothing', 'gender': 'women', 'price': 60.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/j/wj05-brown_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/j/wj05-brown_alt1.jpg', 'stock': 203, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1434, 'sku': 'WJ01', 'name': 'Stellar Solar Jacket', 'short': 'This is a variable product called a Stellar Solar Jacket', 'desc': "Beat the heat and protect yourself from sunrays with the Stellar Solar Jacket. It's loaded with all the engineered features you need for an intense, safe outdoor workout: 100% UV protection, a breathable perforated construction, and advanced moisture-wicking technology. &bull; Loose fit. &bull; Refl", 'category': 'Jackets', 'top': 'Clothing', 'gender': 'women', 'price': 75.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/j/wj01-red_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/j/wj01-red_alt1.jpg', 'stock': 189, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1181, 'sku': 'MJ09', 'name': 'Taurus Elements Shell', 'short': 'This is a variable product called a Taurus Elements Shell', 'desc': "What's a little rain or snow when you're inside the Taurus Elements Shell? This specially engineered Cocona&reg; jacket lets you enjoy the great outdoors and brave the elements, thanks to the all-waterproof, breathable exterior. &bull; Yellow 1/4 zip pullover. &bull; Two chest pockets.&bull; Standar", 'category': 'Jackets', 'top': 'Clothing', 'gender': 'men', 'price': 65.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/j/mj09-yellow_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/j/mj09-yellow_alt1.jpg', 'stock': 181, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1183, 'sku': 'MJ11', 'name': 'Typhon Performance Fleece-lined Jacket', 'short': 'This is a variable product called a Typhon Performance Fleece-lined Jacket', 'desc': 'Ironmen and couch warriors both reach for the Typhon Performance Fleece-lined Jacket. After all, no man can resist ultra-soft microfleece lining. Flatlock seams make it ideal for wearing over everything from tanks and tees to high-tech base layers. &bull; Black full-zip flight jacket.&bull; Cocona&r', 'category': 'Jackets', 'top': 'Clothing', 'gender': 'men', 'price': 60.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/j/mj11-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/j/mj11-black_alt1.jpg', 'stock': 28, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1493, 'sku': 'WP07', 'name': 'Aeon Capri', 'short': 'This is a variable product called a Aeon Capri', 'desc': 'Reach for the stars and beyond in these Aeon Capri pant. With a soft, comfortable feel and moisture wicking fabric, these duo-tone leggings are easy to wear -- and wear attractively. &bull; Black capris with teal accents.&bull; Thick, 3" flattering waistband.&bull; Media pocket on inner waistband.&b', 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 48.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp07-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp07-black_alt1.jpg', 'stock': 157, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1216, 'sku': 'MP11', 'name': 'Aether Gym Pant', 'short': 'This is a variable product called a Aether Gym Pant', 'desc': 'The Aether Gym Pant is built for the studio, but adapts perfectly well to outdoor and gym environs too. With lightweight stretch fabric and water-repellent exterior, the Aether is ready for all comers. &bull; Pants/shorts convertible.&bull; Lightweight moisture wicking. &bull; Water repellent.&bull;', 'category': 'Pants', 'top': 'Clothing', 'gender': 'men', 'price': 74.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/p/mp11-brown_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/p/mp11-brown_alt1.jpg', 'stock': 238, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1494, 'sku': 'WP08', 'name': 'Bardot Capri', 'short': 'This is a variable product called a Bardot Capri', 'desc': 'Black is back &mdash; was it ever gone? &mdash; which means all your favorite tops will get along with the comfortable and versatile Bardot Capri. &bull; Black capris with pink waistband.&bull; Cropped leggings.&bull; Waistband drawcord. &bull; Flat, thin and flattering.&bull; Made with organic fabr', 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 48.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp08-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp08-black_back.jpg', 'stock': 241, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1185, 'sku': 'MP01', 'name': 'Caesar Warm-Up Pant', 'short': 'This is a variable product called a Caesar Warm-Up Pant', 'desc': "Command your workout and keep your muscles limber in the Caesar Warm-Up Pant. Engineered CoolTech&trade; fabric wicks away moisture so you don't have to worry about sweat and discomfort. The drawstring-adjustable waist helps make sure your pants fit properly. &bull; Light gray heather knit straight ", 'category': 'Pants', 'top': 'Clothing', 'gender': 'men', 'price': 35.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/p/mp01-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/p/mp01-gray_back.jpg', 'stock': 230, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1495, 'sku': 'WP09', 'name': 'Carina Basic Capri', 'short': 'This is a variable product called a Carina Basic Capri', 'desc': 'Perfect as workout pants or "accessory," the Carina Basic Capri is comfy as it is practical -- try it under skirts or shorts. &bull; Black capris with rouching detail.&bull; 93% cotton, 7% spandex.&bull; Elasticized waistband.&bull; Reinforced seams with exposed topstitching.&bull; Soft, medium-weig', 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 51.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp09-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp09-black_alt1.jpg', 'stock': 14, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1470, 'sku': 'WP04', 'name': 'Cora Parachute Pant', 'short': 'This is a variable product called a Cora Parachute Pant', 'desc': 'Good for running, hiking, lounging or stretching, the Cora Parachute Pant presents comfortable styling designed to help you look and feel great. &bull; Light blue parachute pants.&bull; Power mesh internal waistband for support.&bull; Internal waistband pocket.&bull; Antimicrobial finish.', 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 75.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp04-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp04-blue_alt1.jpg', 'stock': 242, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1217, 'sku': 'MP12', 'name': 'Cronus Yoga Pant', 'short': 'This is a variable product called a Cronus Yoga Pant', 'desc': 'Guys who love yoga love this all-purpose yoga pant. Designed for yoga, gym and lounge, the Cronus Yoga Pant is a loose-fitting, classic-style pant made from a lightweight blend of cotton, recycled polyester and a touch of Spandex for the stretch factor. &bull; Drawstring waist.&bull; Loose, straight', 'category': 'Pants', 'top': 'Clothing', 'gender': 'men', 'price': 48.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/p/mp12-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/p/mp12-black_back.jpg', 'stock': 172, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1496, 'sku': 'WP10', 'name': 'Daria Bikram Pant', 'short': 'This is a variable product called a Daria Bikram Pant', 'desc': 'The Daria Bikram Pant is designed to stretch and move with your body to make extreme yoga extremely comfortable. &bull; Heather gray capris with pink striped waist.&bull; Flatlock seams.&bull; Interior pocket.', 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 51.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp10-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp10-gray_alt1.jpg', 'stock': 170, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1521, 'sku': 'WP12', 'name': 'Deirdre Relaxed-Fit Capri', 'short': 'This is a variable product called a Deirdre Relaxed-Fit Capri', 'desc': 'Invigorate your yoga practice with this Deirdre Relaxed-Fit Capri, designed to let you move without strain or restriction. &bull; Heather gray capris with mint green waist &amp; accents.&bull; Comfortable, relaxed fit with high rise.&bull; Moisture-wicking, 4-way stretch construction.&bull; Lined wi', 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 63.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp12-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp12-gray_back.jpg', 'stock': 236, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1492, 'sku': 'WP06', 'name': 'Diana Tights', 'short': 'This is a variable product called a Diana Tights', 'desc': "Perfect for hot bikram session or cool-down stretching. 8-percent stretch means you'll feel like anything is possible in the capri-style Diana Tights. &bull; Black legging with slate details.&bull; Flat-lock, chafe-free side seams.&bull; Vented gusset.&bull; Secret interior pocket.&bull; Sustainable", 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 59.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp06-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp06-black_alt1.jpg', 'stock': 24, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1468, 'sku': 'WP02', 'name': 'Emma Leggings', 'short': 'This is a variable product called a Emma Leggings', 'desc': "These comfortable and practical leggings pair perfectly with any workout top or casual tee. The Emma's subtle contrast fabric and fit ensure you're stylish without overdoing it. &bull; Light blue heather yoga pants. &bull; Body hugging fit.&bull; Low rise fit.", 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 42.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp02-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp02-blue_back.jpg', 'stock': 59, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1187, 'sku': 'MP03', 'name': 'Geo Insulated Jogging Pant', 'short': 'This is a variable product called a Geo Insulated Jogging Pant', 'desc': "In the cold, even the toughest guys shiver, unless they're in the Geo Insulated Jogging Pant. Lightweight and wind resistant, they block brutal wind gusts and warm you to the bone. Breathable mesh keeps them dry on the inside. &bull; Black polyester spandex pants with zipper pockets.&bull; Reflectiv", 'category': 'Pants', 'top': 'Clothing', 'gender': 'men', 'price': 51.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/p/mp03-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/p/mp03-black_alt1.jpg', 'stock': 232, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1469, 'sku': 'WP03', 'name': 'Ida Workout Parachute Pant', 'short': 'This is a variable product called a Ida Workout Parachute Pant', 'desc': 'The Ida Workout Parachute Pant is made of lightweight, super-soft fabric that keeps you comfortable during any level of activity, whether Downward Dog or a few trail miles. &bull; Royal blue parachute pants.&bull; Contrast stripe.&bull; Relaxed fit. &bull; Drawstring closure.&bull; Machine wash/dry.', 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 48.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp03-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp03-blue_alt1.jpg', 'stock': 8, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1467, 'sku': 'WP01', 'name': 'Karmen Yoga Pant', 'short': 'This is a variable product called a Karmen Yoga Pant', 'desc': 'Good for running, hiking, lounging or stretching, the Cora Parachute Pant presents comfortable styling designed to help you look and feel great. &bull; Light blue parachute pants.&bull; Power mesh internal waistband for support.&bull; Internal waistband pocket.&bull; Antimicrobial finish.', 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 39.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp01-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp01-gray_back.jpg', 'stock': 11, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1189, 'sku': 'MP05', 'name': 'Kratos Gym Pant', 'short': 'This is a variable product called a Kratos Gym Pant', 'desc': "Like it's namesake god of strength, the Kratos Gym Pant help you bring your best to bear on weight-based, plyometric and endurance exercise. They stretch and support in all the right places while ultra-light construction and moisture-wicking technology provide comfort. &bull; Navy cotton straight le", 'category': 'Pants', 'top': 'Clothing', 'gender': 'men', 'price': 57.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/p/mp05-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/p/mp05-blue_back.jpg', 'stock': 166, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1214, 'sku': 'MP09', 'name': 'Livingston All-Purpose Tight', 'short': 'This is a variable product called a Livingston All-Purpose Tight', 'desc': "It's not often that your pants work as a yoga staple and a climbing buddy. The Livingston All-Purpose Tight is made with soft, 11% cotton and nylon polymer stretch construction that lets you find the right angle in the studio or on the mountain side. &bull; Breathable stretch organic cotton/spandex.", 'category': 'Pants', 'top': 'Clothing', 'gender': 'men', 'price': 75.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/p/mp09-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/p/mp09-blue_alt1.jpg', 'stock': 97, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1190, 'sku': 'MP06', 'name': 'Mithra Warmup Pant', 'short': 'This is a variable product called a Mithra Warmup Pant', 'desc': "When you're not sure you're up to the weather, don the Mithra Warmup Pant for a confidence boost. Its supersoft, stretchy fabric and fleece-like finish help prep your muscles and ease your mind. Designed for relaxed, easy-wear fit with handy ankle zips. &bull; Ankle zips.&bull; Elasticized waistband", 'category': 'Pants', 'top': 'Clothing', 'gender': 'men', 'price': 28.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/p/mp06-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/p/mp06-gray_back.jpg', 'stock': 150, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1215, 'sku': 'MP10', 'name': 'Orestes Yoga Pant', 'short': 'This is a variable product called a Orestes Yoga Pant', 'desc': 'The Orestes Yoga Pant is a yoga basic that sustains the novice yogi well into the expert stage. Organic sustainable cotton and 9% stretch spandex let your body bend to your will while your conscience stays clear. &bull; A yoga essential.&bull; Breathable stretch organic cotton/spandex.&bull; Standar', 'category': 'Pants', 'top': 'Clothing', 'gender': 'men', 'price': 66.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/p/mp10-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/p/mp10-black_alt1.jpg', 'stock': 172, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1522, 'sku': 'WP13', 'name': 'Portia Capri', 'short': 'This is a variable product called a Portia Capri', 'desc': "From yoga at dawn to evenings curled up with a book, our Portia Capri gives you the comfort and freedom to cruise through the day. The high-waisted design keeps you covered while you're bending and stretching, while contrast stripes promise a dash of sporty style. &bull; Salmon heather capri sweats.", 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 49.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp13-orange_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp13-orange_back.jpg', 'stock': 144, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1491, 'sku': 'WP05', 'name': 'Sahara Leggings', 'short': 'This is a variable product called a Sahara Leggings', 'desc': 'Contrast colors, full-length leggings, and opaque, high-stretch fabric define the Sahara. Flat-locked seams and quick-drying wicking properties prevent chafing. A deep elasticated waistband and carefully positioned seamlines create a flattering silhouette. &bull; Pinstripe legging with rouched ankle', 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 75.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp05-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp05-gray_alt1.jpg', 'stock': 137, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1188, 'sku': 'MP04', 'name': 'Supernova Sport Pant', 'short': 'This is a variable product called a Supernova Sport Pant', 'desc': 'Turn the corner and open it up -- your driveway is two blocks away. The Supernova Sport Pant gets you there with key features like moisture-wicking LumaTech&trade; fabric and mesh ventilation. Side seam pockets ensure total convenience during rest periods. &bull; Dark heather gray straight leg cotto', 'category': 'Pants', 'top': 'Clothing', 'gender': 'men', 'price': 45.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/p/mp04-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/p/mp04-gray_back.jpg', 'stock': 131, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1497, 'sku': 'WP11', 'name': 'Sylvia Capri', 'short': 'This is a variable product called a Sylvia Capri', 'desc': 'Enjoy a sleek, slendering look in new Silvia Capri. Its engineer-designed support fabric feels great and gives the illusion of a slimmer fit. &bull; Green striped capri.&bull; Strategic side seam&bull; Comfort gusset with lining.&bull; Flat seaming.&bull; Wide waistband.&bull; Moisture wicking.', 'category': 'Pants', 'top': 'Clothing', 'gender': 'women', 'price': 42.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/p/wp11-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/p/wp11-green_back.jpg', 'stock': 130, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1212, 'sku': 'MP07', 'name': 'Thorpe Track Pant', 'short': 'This is a variable product called a Thorpe Track Pant', 'desc': "Thirty degree temps are chilly for most, except when you're in Thorpe Track Pants. These top-of-the-line track bottoms are made from fast-drying, weather-resistant fabric with an internal breathable layer of mesh nylon to wick away moisture. &bull; Moisture transfer properties. &bull; 7% stretch.&bu", 'category': 'Pants', 'top': 'Clothing', 'gender': 'men', 'price': 68.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/p/mp07-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/p/mp07-blue_alt1.jpg', 'stock': 131, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1186, 'sku': 'MP02', 'name': 'Viktor LumaTech&trade; Pant', 'short': 'This is a variable product called a Viktor LumaTech&trade; Pant', 'desc': "You'll love the new Viktor LumaTech&trade; Pant, with featherweight fleece fabric lining and stretchy, sweat-wicking material. It delivers toasty warmth on the sidelines or in cold-weather training, with reflective trim for a safe finish. &bull; Dark gray polyester/spandex straight leg pants.&bull; ", 'category': 'Pants', 'top': 'Clothing', 'gender': 'men', 'price': 46.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/p/mp02-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/p/mp02-gray_back.jpg', 'stock': 248, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1213, 'sku': 'MP08', 'name': 'Zeppelin Yoga Pant', 'short': 'This is a variable product called a Zeppelin Yoga Pant', 'desc': "Climb every mountain, or hold every pose, in the all-purpose Zepellin Yoga Pant. With its thin fleece interior and smooth layer-friendly surface, you'll get all the comfort and versatility you need. &bull; Smooth exterior for easy over-layering.&bull; Brushed fleece interior insulates and wicks. &bu", 'category': 'Pants', 'top': 'Clothing', 'gender': 'men', 'price': 82.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/p/mp08-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/p/mp08-green_alt1.jpg', 'stock': 55, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1617, 'sku': 'WSH10', 'name': 'Ana Running Short', 'short': 'This is a variable product called a Ana Running Short', 'desc': "Time to lace up your kicks and beat that personal best in the Ana Running Short. It's designed with breathable mesh side panels to help keep you cool while you master the miles. &bull; Black/pink two-layer shorts.&bull; Low-rise elastic waistband.&bull; Relaxed fit. &bull; Ultra-lightweight fabric. ", 'category': 'Shorts', 'top': 'Clothing', 'gender': 'women', 'price': 40.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/wsh10-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/wsh10-black_alt1.jpg', 'stock': 194, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1586, 'sku': 'WSH06', 'name': 'Angel Light Running Short', 'short': 'This is a variable product called a Angel Light Running Short', 'desc': 'The Angel Light Running Short offers comfort in an ultra-lightweight, breathable package. With fabric infused with all-natural Cocona&reg; performance technology, it can whisk away sweat and block UV rays. &bull; Dark heather gray running shorts.&bull; Snug fit. &bull; Elastic waistband. &bull; Coco', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'women', 'price': 42.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/wsh06-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/wsh06-gray_alt1.jpg', 'stock': 196, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1273, 'sku': 'MSH02', 'name': 'Apollo Running Short', 'short': 'This is a variable product called a Apollo Running Short', 'desc': "Fleet of foot or slow and steady, you'll be in complete comfort with the Apollo Running Short. Lightweight polyester material lets you move with ease, and mesh side panels promise plenty of ventilation as you work up a sweat. The stretchy elastic waistband delivers a flexible fit. &bull; Black short", 'category': 'Shorts', 'top': 'Clothing', 'gender': 'men', 'price': 32.5, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/msh02-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/msh02-black_alt1.jpg', 'stock': 118, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1311, 'sku': 'MSH11', 'name': 'Arcadio Gym Short', 'short': 'This is a variable product called a Arcadio Gym Short', 'desc': 'The Arcadio Gym short is a basic-looking piece with a lot more than meets the eye: casual-yet-functional relaxed fit, comfortable waistband and adjustable drawstring. Everything you need, plus more. &bull; Royal blue cotton shorts.&bull; Built-in mesh brief. &bull; 87% Spandex 13% Lycra.&bull; Adjus', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'men', 'price': 20.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/msh11-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/msh11-blue_back.jpg', 'stock': 8, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1584, 'sku': 'WSH04', 'name': 'Artemis Running Short', 'short': 'This is a variable product called a Artemis Running Short', 'desc': 'Discover smooth jogging and chic comfort each time you slip into the Artemis Running Short. A unique maritime-inspired design and oolor theme features a stretchy drawstring waist. &bull; Black rouched shorts with mint waist. &bull; Soft, lightweight construction.&bull; LumaTech&trade; wicking techno', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'women', 'price': 45.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/wsh04-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/wsh04-black_alt1.jpg', 'stock': 150, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1585, 'sku': 'WSH05', 'name': 'Bess Yoga Short', 'short': 'This is a variable product called a Bess Yoga Short', 'desc': 'Designed for intense physical activity &ndash; think bikram &ndash; our Bess Yoga Short features moisture-wicking, four-way stretch fabric that lets you move in every which way. A vented gusset adds breathability and range of motion. &bull; Navy cotton shorts with light bue waist detail.&bull; Front', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'women', 'price': 28.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/wsh05-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/wsh05-blue_back.jpg', 'stock': 233, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1272, 'sku': 'MSH01', 'name': 'Cobalt CoolTech&trade; Fitness Short', 'short': 'This is a variable product called a Cobalt CoolTech&trade; Fitness Short', 'desc': "It's not rocket surgery: the hotter your conditions, the more you need to stay cool. That's why the Cobalt CoolTech&trade; Fitness Short features built-in technology and side perforations that keep you ventilated and comfortable for the distance. The elastic waist promises a personalized fit. &bull;", 'category': 'Shorts', 'top': 'Clothing', 'gender': 'men', 'price': 44.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/msh01-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/msh01-blue_back.jpg', 'stock': 91, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1587, 'sku': 'WSH07', 'name': 'Echo Fit Compression Short', 'short': 'This is a variable product called a Echo Fit Compression Short', 'desc': "Your muscles know it's go time the second you pull on the Echo Fit Compression Short. A balance of firm, stimulating squeeze with breathability, it offers the support and comfort you need to give it your all. &bull; Black compression shorts.&bull; High-waisted cut.&bull; Compression fit.&bull; Insea", 'category': 'Shorts', 'top': 'Clothing', 'gender': 'women', 'price': 24.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/wsh07-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/wsh07-black_back.jpg', 'stock': 75, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1619, 'sku': 'WSH12', 'name': 'Erika Running Short', 'short': 'This is a variable product called Erika Running Shorts', 'desc': "A great short with a body-hugging design, the Erika Running Short is perfect for runners who prefer a fitted short rather than the traditional baggy variety. &bull; Seafoam pattern running shorts.&bull; Elastic waistband.&bull; Snug fit.&bull; 4'' inseam.&bull; 76% premium brushed Nylon / 24% Spande", 'category': 'Shorts', 'top': 'Clothing', 'gender': 'women', 'price': 49.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/wsh12-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/wsh12-green_alt1.jpg', 'stock': 217, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1558, 'sku': 'WSH01', 'name': 'Fiona Fitness Short', 'short': 'This is a variable product called a Fiona Fitness Short', 'desc': "Don't let the plain style fool you: the Fiona Fitness Short demands notice on several levels. Comfort, of course. But also a performance-grade wicking fabric that takes everything you can give. &bull; Black run shorts - cotton/spandex.&bull; 5&rdquo; inseam.&bull; Machine wash/Line dry.", 'category': 'Shorts', 'top': 'Clothing', 'gender': 'women', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/wsh01-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/wsh01-black_back.jpg', 'stock': 8, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1583, 'sku': 'WSH03', 'name': 'Gwen Drawstring Bike Short', 'short': 'This is a variable product called a Gwen Drawstring Bike Short', 'desc': 'For a completely modern style with moisture-wicking CoolTech&trade; technology, try the Gwen Drawstring Bike Short. Subtle grays and eye catching stitching combine with an adjustable waist for the perfect look and fit. &bull; Dark heather gray rouched bike shorts.&bull; Fitted. Inseam: 2". &bull; Ma', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'women', 'price': 50.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/wsh03-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/wsh03-gray_alt1.jpg', 'stock': 213, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1305, 'sku': 'MSH05', 'name': 'Hawkeye Yoga Short', 'short': 'This is a variable product called a Hawkeye Yoga Short', 'desc': 'What more do you need than a sporty yoga short made with organic cotton and a little spandex for mobility? The Hawkeye Yoga Short brings a stylish, standard fit you can sport with confidence outside the studio. &bull; Dark gray shorts with red accents.&bull; 92% Organic Cotton 8% Spandex.&bull; Brea', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'men', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/msh05-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/msh05-gray_alt1.jpg', 'stock': 165, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1618, 'sku': 'WSH11', 'name': 'Ina Compression Short', 'short': 'This is a variable product called a Ina Compression Short', 'desc': "One of Luma's most popular items, the Ina Compression Short has you covered with exceptional support and comfort, whether you're running the trail, riding a bike or ripping out reps. The ventilating fabric offers cool relief and prevents irritating chafing. &bull; Royal blue bike shorts.&bull; Compr", 'category': 'Shorts', 'top': 'Clothing', 'gender': 'women', 'price': 49.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/wsh11-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/wsh11-blue_back.jpg', 'stock': 173, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1306, 'sku': 'MSH06', 'name': 'Lono Yoga Short', 'short': 'This is a variable product called a Lono Yoga Short', 'desc': 'Some shorts are &ldquo;all-purpose,&rdquo; while the Lono Yoga Short has a single purpose: yoga. Soft, flexible outer material moves with your body, and a secure, integrated boxer brief liner gives you extra confidence as you push your physique to the outer limits. &bull; Dark gray shorts with mesh ', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'men', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/msh06-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/msh06-gray_alt1.jpg', 'stock': 94, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1582, 'sku': 'WSH02', 'name': 'Maxima Drawstring Short', 'short': 'This is a variable product called a Maxima Drawstring Short', 'desc': "Even laid-back women like to stay stylish, and that's just what the Maxima Drawstring Short delivers. An elastic waist keeps the fit flexible, on the deck at home or on a trail walk. Sporty tennis flair adds an athletic accent. &bull; Light gray run shorts - cotton polyester. &bull; Contrast binding", 'category': 'Shorts', 'top': 'Clothing', 'gender': 'women', 'price': 28.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/wsh02-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/wsh02-gray_back.jpg', 'stock': 60, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1274, 'sku': 'MSH03', 'name': 'Meteor Workout Short', 'short': 'This is a variable product called a Meteor Workout Short', 'desc': 'Step into the Meteor Workout Short for an incredibly lightweight fitness experience. Its breathable construction and an inner brief provide additional comfort and support, while the adjustable waistband offers the perfect fit to take you to the finish. &bull; Royal blue shorts with light blue accent', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'men', 'price': 32.5, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/msh03-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/msh03-blue_alt1.jpg', 'stock': 188, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1589, 'sku': 'WSH09', 'name': 'Mimi All-Purpose Short', 'short': 'This is a variable product called a Mimi All-Purpose Short', 'desc': 'You can run, bike or swim in the do-anything, water-resistance Mimi Short. No need to worry about rubbing-induced soreness either, with flatlock seams and soft, chafe-resistant material. &bull; Gray/seafoam two-layer shorts.&bull; Water-resistant construction.&bull; Inner mesh brief for breathable s', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'women', 'price': 44.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/wsh09-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/wsh09-gray_alt1.jpg', 'stock': 209, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1308, 'sku': 'MSH08', 'name': 'Orestes Fitness Short', 'short': 'This is a variable product called a Orestes Fitness Short', 'desc': "You're out of excuses not to run, lift, or play when you own a pair of the Orestes Fitness Short. You won't complain about the high-performance polyester wicking fabric, reflective safety trim or seamless built-in comfort brief, either. &bull; Black shorts with dark gray accents.&bull; Elasticized w", 'category': 'Shorts', 'top': 'Clothing', 'gender': 'men', 'price': 35.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/msh08-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/msh08-black_back.jpg', 'stock': 6, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1312, 'sku': 'MSH12', 'name': 'Pierce Gym Short', 'short': 'This is a variable product called a Pierce Gym Short', 'desc': 'The Pierce Gym Short is similar to our Arcadio Gym Short, but designed with a fitted, tapered look. A flat waistband with an adjustable drawstring adds comfort, with zippered pockets for safe, easy storage. &bull; Dark red cotton shorts. &bull; 87% Supplex, 13% Lycra.&bull; Adjustable drawstring wai', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'men', 'price': 27.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/msh12-red_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/msh12-red_back.jpg', 'stock': 83, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1307, 'sku': 'MSH07', 'name': 'Rapha  Sports Short', 'short': 'This is a variable product called a Rapha Sports Short', 'desc': 'For those about to sweat, we support you with the Rapha Sports Short. The compression-fit liner surrounds your muscles with the stimulation they need to find your high gear, while moisture-wicking performance fabric helps prevents sweat build-up. &bull; Black shorts with royal accents.&bull; Compres', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'men', 'price': 35.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/msh07-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/msh07-black_alt1.jpg', 'stock': 149, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1310, 'sku': 'MSH10', 'name': 'Sol Active Short', 'short': 'This is a variable product called a Sol Active Short', 'desc': "You'll let your fear go and push your limits in your new Sol Active Short. Featuring ultra-breathable performance fabric and a flat comfort-fit waistband, the Sol Active Short is perfect for high-intensity circuits or high-heat bikram. &bull; Light blue jersey shorts with mesh detail.&bull; 87% Span", 'category': 'Shorts', 'top': 'Clothing', 'gender': 'men', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/msh10-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/msh10-blue_alt1.jpg', 'stock': 162, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1588, 'sku': 'WSH08', 'name': 'Sybil Running Short', 'short': 'This is a variable product called a Sybil Running Short', 'desc': 'Fortunately, it\'s okay to look cute while you\'re working out. The Sybil Running Short combines a fun, color-blocked design with breathable mesh fabric for sporty-fun style. &bull; Blue running shorts with green waist.&bull; Drawstring-adjustable waist.&bull; 4" inseam. Machine wash/line dry.', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'women', 'price': 44.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/wsh08-purple_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/wsh08-purple_back.jpg', 'stock': 230, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1304, 'sku': 'MSH04', 'name': 'Torque Power Short', 'short': 'This is a variable product called a Torque Power Short', 'desc': 'Take a turn in the Torque Power Short and enjoy the smooth, motion-friendly stretch and sweat-wicking, UV-protective material. You\'ll stay cool, dry and energized miles later. &bull; Light gray shorts.&bull; Fitted design. &bull; Elastic waistband. &bull; Flat-seam construction. &bull; 7" inseam. &b', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'men', 'price': 32.5, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/msh04-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/msh04-gray_back.jpg', 'stock': 148, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1309, 'sku': 'MSH09', 'name': 'Troy Yoga Short', 'short': 'This is a variable product called a Troy Yoga Short', 'desc': 'The versatile, all-purpose Troy Yoga Short is knee-lengthed with an elastic waistband with drawstring, making it comfortably flexible. The removable LumaTech&trade; wicking liner is also bacteria resistant. &bull; Navy polyester pinstripe shorts.&bull; Woven fabric with moderate stretch.&bull; 62% c', 'category': 'Shorts', 'top': 'Clothing', 'gender': 'men', 'price': 24.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/msh09-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/msh09-blue_alt1.jpg', 'stock': 16, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1342, 'sku': 'MT07', 'name': 'Argus All-Weather Tank', 'short': 'This is a variable product called a Argus All-Weather Tank', 'desc': 'The Argus All-Weather Tank is sure to become your favorite base layer or go-to cover for hot outdoor workouts. With its subtle reflective safely trim, you can even wear it jogging on urban evenings. &bull; Dark gray polyester spandex tank.&bull; Reflective details for nighttime visibility. &bull; St', 'category': 'Tanks', 'top': 'Clothing', 'gender': 'men', 'price': 22.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/t/mt07-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/t/mt07-gray_back.jpg', 'stock': 126, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1367, 'sku': 'MT11', 'name': 'Atlas Fitness Tank', 'short': 'This is a variable product called a Atlas Fitness Tank', 'desc': 'From weekend warrior to Warrior Pose II, no role can beat the Atlas Fitness Tank, a versatile top for gym and yoga studio. Wicking-weave soft fabric helps prevent uncomfortable chafing. &bull; Teal scoop neck cotton tank.&bull; Triblend, soft fabric.&bull; Relaxed fit.', 'category': 'Tanks', 'top': 'Clothing', 'gender': 'men', 'price': 18.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/t/mt11-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/t/mt11-blue_back.jpg', 'stock': 40, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1368, 'sku': 'MT12', 'name': 'Cassius Sparring Tank', 'short': 'This is a variable product called a Cassius Sparring Tank', 'desc': "Whether you're up against a partner or the clock, the Cassius Sparring Tank is in your corner, moving effortless with your body. The light and loose feel gives you no reason to give up before the bell or the end of the block. &bull; Royal crewneck cotton tank.&bull; Contrast stitching.&bull; Self fa", 'category': 'Tanks', 'top': 'Clothing', 'gender': 'men', 'price': 18.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/t/mt12-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/t/mt12-blue_back.jpg', 'stock': 238, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1313, 'sku': 'MT01', 'name': 'Erikssen CoolTech&trade; Fitness Tank', 'short': 'This is a variable product called a Erikssen CoolTech&trade; Fitness Tank', 'desc': 'A good running tank helps make the miles pass by keep you cool. The Erikssen CoolTech&trade; Fitness Tank completes that mission, with performance fabric engineered to wick perspiration and promote airflow. &bull; Red performance tank.&bull; Slight scoop neckline. &bull; Reflectivity. &bull; Machine', 'category': 'Tanks', 'top': 'Clothing', 'gender': 'men', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/t/mt01-red_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/t/mt01-red_alt1.jpg', 'stock': 91, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1339, 'sku': 'MT04', 'name': 'Helios Endurance Tank', 'short': 'This is a variable product called a Helios Endurance Tank', 'desc': 'When training pushes your limits, you need gear that works harder than you. Our mesh Helio Training Tank is crafted from super-soft, ultra-lightweight fabric that stretches in all directions to follow your every move, on mat, court or street. &bull; Blue heather tank with gray pocket.&bull; Contrast', 'category': 'Tanks', 'top': 'Clothing', 'gender': 'men', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/t/mt04-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/t/mt04-blue_alt1.jpg', 'stock': 166, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1338, 'sku': 'MT03', 'name': 'Primo Endurance Tank', 'short': 'This is a variable product called a Primo Endurance Tank', 'desc': 'Chances are your workout goes beyond free weights, which is why the Primo Endurance Tank employs maximum versatility. Run, lift or play ball &ndash; this breathable mesh top will keep you cool during all your activities. &bull; Red heather tank with gray pocket.&bull; Chafe-resistant flatlock seams.', 'category': 'Tanks', 'top': 'Clothing', 'gender': 'men', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/t/mt03-red_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/t/mt03-red_main.jpg', 'stock': 179, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1340, 'sku': 'MT05', 'name': 'Rocco Gym Tank', 'short': 'This is a variable product called a Rocco Gym Tank', 'desc': "Weights? Heavy Bag? Bikram? However you get sweaty, you'll be prepared with the moisture-wicking all-purpose Rocco Gym Tank. The free-moving cut gives you maximum range of motion, with flatlock seams to eliminate chafing. &bull; Light blue heather gray tank.&bull; Quick-drying, moisture-wicking.&bul", 'category': 'Tanks', 'top': 'Clothing', 'gender': 'men', 'price': 24.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/t/mt05-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/t/mt05-blue_back.jpg', 'stock': 34, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1344, 'sku': 'MT09', 'name': 'Sinbad Fitness Tank', 'short': 'This is a variable product called a Sinbad Fitness Tank', 'desc': 'Solid in color and construction, the 100% cotton-weave Sinbad Fitness Tank moves with you as you press, hold, crunch and stride your way to the ultimate you. &bull; Teal polyester tank.&bull; Premium fit tank top.&bull; Ultra lightweight.&bull; Naturally odor-resistant.', 'category': 'Tanks', 'top': 'Clothing', 'gender': 'men', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/t/mt09-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/t/mt09-blue_alt1.jpg', 'stock': 67, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1343, 'sku': 'MT08', 'name': 'Sparta Gym Tank', 'short': 'This is a variable product called a Sparta Gym Tank', 'desc': 'The high performance Sparta Gym Tank is made with thin, light, merino wool and aims to be the perfect base layer or balmy weather running and fitness top. &bull; Green polyester tank.&bull; Ultra lightweight.&bull; Naturally odor-resistant.&bull; Close-to-body athletic fit.&bull; Chafe-resistant fla', 'category': 'Tanks', 'top': 'Clothing', 'gender': 'men', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/t/mt08-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/t/mt08-green_alt1.jpg', 'stock': 218, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1345, 'sku': 'MT10', 'name': 'Tiberius Gym Tank', 'short': 'This is a variable product called a Tiberius Gym Tank', 'desc': "Whether you're classy or just sweaty, the Tiberius Gym Tank helps you look good while you're at it. What's more, its moisture-wicking, quick-drying, anti-microbial and anti-odor construction help ensure you're welcome back to the gym. &bull; Yellow scoop neck cotton tank.&bull; Comfortable, relaxed ", 'category': 'Tanks', 'top': 'Clothing', 'gender': 'men', 'price': 18.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/t/mt10-yellow_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/t/mt10-yellow_back.jpg', 'stock': 245, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1337, 'sku': 'MT02', 'name': 'Tristan Endurance Tank', 'short': 'This is a variable product called a Tristan Endurance Tank', 'desc': "Push yourself through punishing runs, plyometric workouts, intense competition and more in our athletic Tristan Endurance Tank. Constructed with built-in moisture-wicking technology, it's designed to keep you completely cool and dry on the long haul. &bull; White performance tank.&bull; Stylish cont", 'category': 'Tanks', 'top': 'Clothing', 'gender': 'men', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/t/mt02-white_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/t/mt02-white_alt1.jpg', 'stock': 227, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1341, 'sku': 'MT06', 'name': 'Vulcan Weightlifting Tank', 'short': 'This is a variable product called a Vulcan Weightlifting Tank', 'desc': 'The Olympic styled Vulcan Weightlifting Tank features polyester stretch and flex. Hit the rack in sleeveless style and unleash your personal best. This tank is designed to max performance, comfort and range of motion. &bull; Black polyester spandex tank.&bull; 100% polyester.&bull; Freedom of moveme', 'category': 'Tanks', 'top': 'Clothing', 'gender': 'men', 'price': 28.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/t/mt06-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/t/mt06-black_back.jpg', 'stock': 16, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1218, 'sku': 'MS01', 'name': 'Aero Daily Fitness Tee', 'short': 'This is a variable product called a Aero Daily Fitness Tee', 'desc': "Need an everyday action tee that helps keep you dry? The Aero Daily Fitness Tee is made of 100% polyester wicking knit that funnels moisture away from your skin. Don't be fooled by its classic style; this tee hides premium performance technology beneath its unassuming look. Relaxed fit. Short-Sleeve", 'category': 'Tees', 'top': 'Clothing', 'gender': 'men', 'price': 24.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/ms01-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/ms01-blue_back.jpg', 'stock': 235, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1271, 'sku': 'MS12', 'name': 'Atomic Endurance Running Tee (Crew-Neck)', 'short': 'This is a variable product called a Atomic Endurance Running Tee (Crew-Neck)', 'desc': "Like it's v-neck counterpart, the crew-neck Atomic Tee will get you to your goal and beyond with its many load-bearing features: ultra-lightweight, moisture-wicking Cocona&reg; fabric, chafe-free flatlock seams and an ergonomic cut that moves with your body. &bull; Red polyester tee.&bull; Crew neck", 'category': 'Tees', 'top': 'Clothing', 'gender': 'men', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/ms12-red_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/ms12-red_alt1.jpg', 'stock': 8, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1270, 'sku': 'MS11', 'name': 'Atomic Endurance Running Tee (V-neck)', 'short': 'This is a variable product called a Atomic Endurance Running Tee (V-neck)', 'desc': 'Reach your limit and keep on going in the Atomic Endurance Running Tee. Built to help any athlete push past the wall with loads of performance features. &bull; Lime heathered v-neck tee. &bull; Ultra-lightweight.&bull; Moisture-wicking Cocona&reg; fabric.&bull; Machine wash/dry.', 'category': 'Tees', 'top': 'Clothing', 'gender': 'men', 'price': 28.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/ms11-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/ms11-green_back.jpg', 'stock': 197, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1241, 'sku': 'MS03', 'name': 'Balboa Persistence Tee', 'short': 'This is a variable product called a Balboa Persistence Tee', 'desc': 'The Balboa Persistence Tee is a must-have for any athlete, Philadelphia or elsewhere. We took the best of performance apparel, cut the fluff and boiled it down to the basics for a lightweight, quick-drying t-shirt. &bull; Crew neckline. &bull; Semi-fitted. &bull; Cocona&reg; performance fabric. &bul', 'category': 'Tees', 'top': 'Clothing', 'gender': 'men', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/ms03-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/ms03-black_back.jpg', 'stock': 5, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1245, 'sku': 'MS07', 'name': 'Deion Long-Sleeve EverCool&trade; Tee', 'short': 'This is a variable product called a Deion Long-Sleeve EverCool&trade; Tee', 'desc': "When you're training, ordinary tees don't cut it. That's why the Dion Long-Sleeve EverCool&trade; Tee features revolutionary Cocona&reg; fabric derived from a renewable resource: coconut shells. This unique material protects you from harmful UV rays, wicks away sweat, controls stink and dries quickl", 'category': 'Tees', 'top': 'Clothing', 'gender': 'men', 'price': 39.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/ms07-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/ms07-green_alt1.jpg', 'stock': 238, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1527, 'sku': 'WS05', 'name': 'Desiree Fitness Tee', 'short': 'This is a variable product called a Desiree Fitness Tee', 'desc': "When you're too far to turn back, thank yourself for choosing the Desiree Fitness Tee. Its ultra-lightweight, ultra-breathable fabric wicks sweat away from your body and helps keeps you cool for the distance. &bull; Short-Sleeves.&bull; Performance fabric.&bull; Machine wash/line dry.", 'category': 'Tees', 'top': 'Clothing', 'gender': 'women', 'price': 24.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/ws05-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/ws05-black_alt1.jpg', 'stock': 170, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1556, 'sku': 'WS11', 'name': 'Diva Gym Tee', 'short': 'This is a variable product called a Diva Gym Tee', 'desc': "The Diva Gym Tee feels like your favorite right out of the mailbox. Micro-sleeved, lightweight and extra comfortable, it's a casual staple that hides incredible comfort behind a laid-back, carefree look. &bull; Bright yellow v-neck tee.&bull; Moisture-wicking fabric. &bull; Long-Sleeves. &bull; Mach", 'category': 'Tees', 'top': 'Clothing', 'gender': 'women', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/ws11-yellow_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/ws11-yellow_back.jpg', 'stock': 147, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1528, 'sku': 'WS06', 'name': 'Elisa EverCool&trade; Tee', 'short': 'This is a variable product called a Elisa EverCool&trade; Tee', 'desc': 'When rising temps threaten to melt you down, Elisa EverCool&trade; Tee brings serious relief. Moisture-wicking fabric pulls sweat away from your skin, while the innovative seams hug your muscles to enhance your range of motion. &bull; Purple heather v-neck tee.&bull; Short-Sleeves.&bull; Luma EverCo', 'category': 'Tees', 'top': 'Clothing', 'gender': 'women', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/ws06-purple_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/ws06-purple_back.jpg', 'stock': 128, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1524, 'sku': 'WS02', 'name': 'Gabrielle Micro Sleeve Top', 'short': 'This is a variable product called a Gabrielle Micro Sleeve Top', 'desc': "Luma's most popular top, the Gabrielle Micro Sleeve is back with even more comfort and style. &bull; Lime green v-neck tee.&bull; Slimming, flattering fit.&bull; Moisture-wicking, quick-drying, anti-microbial, and anti-odor construction.&bull; Longer curved hem provides additional coverage.&bull; 55", 'category': 'Tees', 'top': 'Clothing', 'gender': 'women', 'price': 28.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/ws02-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/ws02-green_back.jpg', 'stock': 247, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1242, 'sku': 'MS04', 'name': 'Gobi HeatTec&reg; Tee', 'short': 'This is a variable product called a Gobi HeatTec&reg; Tee', 'desc': "When the training gets intense, the Gobi HeatTec&reg; Tee works as hard as you do to maintain your cool. The moisture-wicking material promises drier comfort, while breathable side panels deliver extra stretch that's sure to keep you moving. &bull; Orange micropolyester shirt.&bull; HeatTec&reg; wic", 'category': 'Tees', 'top': 'Clothing', 'gender': 'men', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/ms04-orange_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/ms04-orange_back.jpg', 'stock': 44, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1523, 'sku': 'WS01', 'name': 'Gwyn Endurance Tee', 'short': 'This is a variable product called a Gwyn Endurance Tee', 'desc': 'When the miles add up, comfort is crucial. The short-sleeve Gwyn Endurance Tee is designed with an ultra-lightweight blend of breathable fabrics to help you tackle your training. Female-specific seams and a sporty v-neckline offer subtle style. &bull; Short-Sleeves.&bull; Machine wash/line dry.', 'category': 'Tees', 'top': 'Clothing', 'gender': 'women', 'price': 24.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/ws01-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/ws01-black_back.jpg', 'stock': 179, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1243, 'sku': 'MS05', 'name': 'Helios EverCool&trade; Tee', 'short': 'This is a variable product called a Helios EverCool&trade; Tee', 'desc': "Pumping iron or dialing the track, you've got cool comfort on your side in our short-sleeve Helios EverCool&trade; Tee. The fabric is infused with moisture-wicking technology that pulls sweat off your skin for speedy evaporation. Stretchy fabric gussets encourage ventilation while increasing your ra", 'category': 'Tees', 'top': 'Clothing', 'gender': 'men', 'price': 24.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/ms05-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/ms05-blue_back.jpg', 'stock': 90, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1525, 'sku': 'WS03', 'name': 'Iris Workout Top', 'short': 'This is a variable product called a Iris Workout Top', 'desc': 'The Iris Workout Top is sleeveless, body fitting and moisture wicking for a fashionable, functional garment. Rouched textures add style. &bull; Pink heather rouched v-neck.&bull; Scoop neckline.&bull; Angled flat seams.&bull; Moisture wicking.&bull; Body skimming.&bull; 83% Polyester / 11% TENCEL&re', 'category': 'Tees', 'top': 'Clothing', 'gender': 'women', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/ws03-red_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/ws03-red_alt1.jpg', 'stock': 143, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1552, 'sku': 'WS07', 'name': 'Juliana Short-Sleeve Tee', 'short': 'This is a variable product called a Juliana Short-Sleeve Tee', 'desc': 'The Juliana Short-Sleeve Tee gives you more than sporty style. Consider that its soft Cocona&reg; polyester fabric accelerates evaporation to help keep you dry and comfortable, too. And a selection of colors lets you get more than one. &bull; Black scoop neck tee.&bull; Side rouching.&bull; Relaxed ', 'category': 'Tees', 'top': 'Clothing', 'gender': 'women', 'price': 42.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/ws07-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/ws07-black_alt1.jpg', 'stock': 7, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1555, 'sku': 'WS10', 'name': 'Karissa V-Neck Tee', 'short': 'This is a variable product called a Karissa V-Neck Tee', 'desc': 'The Karissa V-Neck Tee features a semi-fitted shape that\'s flattering for every figure. You can hit the gym with confidence while it hugs curves and hides common "problem" areas. &bull; Pink heather soft v-neck tee. &bull; Luma signature micro sleeves.&bull; Semi-fitted. &bull; Machine wash/dry.', 'category': 'Tees', 'top': 'Clothing', 'gender': 'women', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/ws10-red_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/ws10-red_back.jpg', 'stock': 111, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1526, 'sku': 'WS04', 'name': 'Layla Tee', 'short': 'This is a variable product called a Layla Tee', 'desc': 'Work out or hang out in chic style in the Layla Tee. With a lightweight sheer design and a roomy neckline, this tee fits you comfortably while looking stylish. &bull; Teal tee.&bull; Long back hem.&bull; Dropped shoulders.', 'category': 'Tees', 'top': 'Clothing', 'gender': 'women', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/ws04-green_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/ws04-green_back.jpg', 'stock': 149, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1269, 'sku': 'MS10', 'name': 'Logan  HeatTec&reg; Tee', 'short': 'This is a variable product called a Logan HeatTec&reg; Tee', 'desc': 'Soft and lightweight, the Logan HeatTec&reg; Tee gets you through the long haul in total comfort. It boasts superior sweat-wicking performance to keep skin dry and cool, and strategic flat-lock seams to resist chafing. &bull; Semi-fitted. &bull; Crew neckline. &bull; Machine wash/tumble dry.', 'category': 'Tees', 'top': 'Clothing', 'gender': 'men', 'price': 24.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/ms10-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/ms10-blue_alt1.jpg', 'stock': 219, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1553, 'sku': 'WS08', 'name': 'Minerva LumaTech&trade; V-Tee', 'short': 'This is a variable product called a Minerva LumaTech&trade; V-Tee', 'desc': "Don't be fooled by the simple design of our Minerva LumaTech&trade; V-Neck Tee. This classic training top features the same serious sweat-wicking technology as some of the more expensive tees in our lineup. &bull; Navy blue heather soft v-neck tee.&bull; Flatlock seams for chafe-free comfort.&bull; ", 'category': 'Tees', 'top': 'Clothing', 'gender': 'women', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/ws08-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/ws08-blue_back.jpg', 'stock': 45, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1557, 'sku': 'WS12', 'name': 'Radiant Tee', 'short': 'This is a variable product called a Radiant Tee', 'desc': "So light and comfy, you'll love the Radiant Tee's organic fabric, feel, performance and style. You may never want to stop moving in this shirt. &bull; Salmon soft scoop neck tee.&bull; Athletic, semi-form fit.&bull; Flat seams prevent chafing.&bull; 67% Organic Cotton / 28% Hemp / 5% Spandex.", 'category': 'Tees', 'top': 'Clothing', 'gender': 'women', 'price': 22.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/ws12-orange_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/ws12-orange_back.jpg', 'stock': 84, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1268, 'sku': 'MS09', 'name': 'Ryker LumaTech&trade; Tee (Crew-neck)', 'short': 'This is a variable product called a Ryker LumaTech&trade; Tee (Crew-neck)', 'desc': 'The crew-neck Ryker LumaTech&trade; Tee hides premium performance technology beneath unassuming looks. The featherweight blend of fabrics wicks away moisture to keep you cool and dry in every phase of your active life. &bull; Royal polyester tee with black accents.&bull; Relaxed fit. &bull; Short-Sl', 'category': 'Tees', 'top': 'Clothing', 'gender': 'men', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/ms09-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/ms09-blue_alt1.jpg', 'stock': 39, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1240, 'sku': 'MS02', 'name': 'Ryker LumaTech&trade; Tee (V-neck)', 'short': 'This is a variable product called a Ryker LumaTech&trade; Tee (V-neck)', 'desc': "Don't be fooled by its classic style; the Ryker LumaTech&trade; Tee embodies the future of performance apparel. Its featherweight blend of fabrics wicks away moisture to keep you cool and dry, whether racking up miles, hitting three pointers or strutting the boardwalk. &bull; Relaxed fit. &bull; Sho", 'category': 'Tees', 'top': 'Clothing', 'gender': 'men', 'price': 28.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/ms02-gray_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/ms02-gray_alt1.jpg', 'stock': 217, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1246, 'sku': 'MS08', 'name': 'Strike Endurance Tee', 'short': 'This is a variable product called a Strike Endurance Tee', 'desc': 'While grit and purpose keep you going, it helps to have a little extra comfort, too. Our Strike Long-Sleeve Endurance Tee helps ensures a photo-ready finish with advanced sweat-wicking technology for a cool, dry feel. &bull; Loose fit. &bull; Ribbed cuffs/collar. &bull; Machine wash/dry.', 'category': 'Tees', 'top': 'Clothing', 'gender': 'men', 'price': 39.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/ms08-black_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/ms08-black_back.jpg', 'stock': 124, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1554, 'sku': 'WS09', 'name': 'Tiffany Fitness Tee', 'short': 'This is a variable product called a Tiffany Fitness Tee', 'desc': "You'll work out and look cute doing it in the short-sleeve Tiffany Fitness Tee. The longer length and fitted cut offer a smoother silhouette, while scoop neck pattern and drape effects lend a trendy touch. &bull; Teal soft scoop neck tee.&bull; Contrast stitching pattern. &bull; Machine wash/dry.", 'category': 'Tees', 'top': 'Clothing', 'gender': 'women', 'price': 28.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/s/ws09-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/s/ws09-blue_back.jpg', 'stock': 205, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 1244, 'sku': 'MS06', 'name': 'Zoltan Gym Tee', 'short': 'This is a variable product called a Zoltan Gym Tee', 'desc': 'This short-sleeve wonder works twice as hard to give you good gym days and good looks. The Zoltan Gym Tee helps you stay comfortable, while the looser sleeves and flatlock seams keep you moving in chafe-free comfort. &bull; Relaxed fit. &bull; Crew neckline. &bull; Machine wash/dry.', 'category': 'Tees', 'top': 'Clothing', 'gender': 'men', 'price': 29.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/s/ms06-blue_main.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/s/ms06-blue_alt1.jpg', 'stock': 42, 'colors': [], 'sizes': [], 'type': 'variable'}, {'id': 2039, 'sku': '24-WB02', 'name': 'Compete Track Tote', 'short': 'This is a simple product called Compete Track Tote', 'desc': 'The Compete Track Tote holds a host of exercise supplies with ease. Stash your towel, jacket and street shoes inside. Tuck water bottles in easy-access external spaces. Perfect for trips to gym or yoga studio, with dual top handles for convenience to and from. Two-way zippers. Contrast detailing. W ', 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/b/wb02-green-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/b/wb02-green-0.jpg', 'stock': 220, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2033, 'sku': '24-MB03', 'name': 'Crown Summit Backpack', 'short': 'This is a simple product called Crown Summit Backpack', 'desc': 'The Crown Summit Backpack is equally at home in a gym locker, study cube or a pup tent, so be sure yours is packed with books, a bag lunch, water bottles, yoga block, laptop, or whatever else you want in hand. Rugged enough for day hikes and camping trips, it has two large zippered compartments and ', 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 38.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/b/mb03-black-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/b/mb03-black-0_alt1.jpg', 'stock': 84, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2042, 'sku': '24-WB03', 'name': 'Driven Backpack', 'short': 'This is a simple product called Driven Backpack', 'desc': "School books, camp gear and yoga clothes get along just fine in the all-purpose Driven Backpack. Crafted with tough polyester ripstop fabric, it's outfitted with rubberized end panels and padded, adjustable shoulder straps. The roomy main compartment features molded foam pockets that host everything", 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 36.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/b/wb03-purple-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/b/wb03-purple-0.jpg', 'stock': 184, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2041, 'sku': '24-WB06', 'name': 'Endeavor Daytrip Backpack', 'short': 'This is a simple product called Endeavor Daytrip Backpack', 'desc': "With more room than it appears, the Endeavor Daytrip Backpack will hold a whole day's worth of books, binders and gym clothes. The spacious main compartment includes a dedicated laptop sleeve. Two other compartments offer extra storage space. Foam-padded adjustable shoulder straps. 900D polyester. O", 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 33.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/b/wb06-red-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/b/wb06-red-0_alt1.jpg', 'stock': 203, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2036, 'sku': '24-MB02', 'name': 'Fusion Backpack', 'short': 'This is a simple product called Fusion Backpack', 'desc': "With the Fusion Backpack strapped on, every trek is an adventure - even a bus ride to work. That's partly because two large zippered compartments store everything you need, while a front zippered pocket and side mesh pouches are perfect for stashing those little extras, in case you change your mind ", 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 59.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/b/mb02-gray-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/b/mb02-blue-0.jpg', 'stock': 212, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2037, 'sku': '24-UB02', 'name': 'Impulse Duffle', 'short': 'This is a simple product called Impulse Duffle', 'desc': "Good for beach trips, track meets, yoga retreats and more, the Impulse Duffle is the companion you'll want at your side. A large U-shaped opening makes packing a hassle-free affair, while a zippered interior pocket keeps jewelry and other small valuables safely tucked out of sight. Wheeled. Dual car", 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 74.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/u/b/ub02-black-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/u/b/ub02-black-0.jpg', 'stock': 201, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2031, 'sku': '24-MB01', 'name': 'Joust Duffle Bag', 'short': 'This is a simple product called Joust Duffle Bag', 'desc': "The sporty Joust Duffle Bag can't be beat - not in the gym, not on the luggage carousel, not anywhere. Big enough to haul a basketball or soccer ball and some sneakers with plenty of room to spare, it's ideal for athletes with places to go. Dual top handles. Adjustable shoulder strap. Full-length zi", 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 34.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/b/mb01-blue-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/b/mb01-blue-0.jpg', 'stock': 134, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2043, 'sku': '24-WB07', 'name': 'Overnight Duffle', 'short': 'This is a simple product called Overnight Duffle', 'desc': "For long weekends away, camping outings and business trips, the Overnight Duffle can't be beat. The dual handles make it a cinch to carry, while the durable waterproof exterior helps you worry less about the weather. With multiple organizational pockets inside and out, it's the perfect travel compan", 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 45.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/b/wb07-brown-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/b/wb07-brown-0.jpg', 'stock': 120, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2044, 'sku': '24-WB04', 'name': 'Push It Messenger Bag', 'short': 'This is a simple product called Push It Messenger Bag', 'desc': "The name says so, but the Push It Messenger Bag is much more than a busy commuter's tote. It's a closet away from home when you're pedaling from class or work to gym and back or home again. It's the perfect size and shape for laptop, folded clothes, even extra shoes. Adjustable crossbody strap. Top ", 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 45.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/b/wb04-blue-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/b/wb04-blue-0.jpg', 'stock': 59, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2035, 'sku': '24-MB06', 'name': 'Rival Field Messenger', 'short': 'This is a simple product called Rival Field Messenger', 'desc': 'The Rival Field Messenger packs all your campus, studio or trail essentials inside a unique design of soft, textured leather - with loads of character to spare. Two exterior pockets keep all your smaller items handy, and the roomy interior offers even more space. Leather construction. Adjustable fab', 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 45.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/b/mb06-gray-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/b/mb06-gray-0.jpg', 'stock': 113, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2040, 'sku': '24-WB05', 'name': 'Savvy Shoulder Tote', 'short': 'This is a simple product called Savvy Shoulder Tote', 'desc': 'Powerwalking to the gym or strolling to the local coffeehouse, the Savvy Shoulder Tote lets you stash your essentials in sporty style! A top-loading compartment provides quick and easy access to larger items, while zippered pockets on the front and side hold cash, credit cards and phone. Water-resis', 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 32.0, 'sale': 24.0, 'img': 'http://eimages.valtim.com/acme-images/product/w/b/wb05-red-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/b/wb05-red-0.jpg', 'stock': 98, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2032, 'sku': '24-MB04', 'name': 'Strive Shoulder Pack', 'short': 'This is a simple product called Strive Shoulder Pack', 'desc': "Convenience is next to nothing when your day is crammed with action. So whether you're heading to class, gym, or the unbeaten path, make sure you've got your Strive Shoulder Pack stuffed with all your essentials, and extras as well. Zippered main compartment. Front zippered pocket. Side mesh pocket.", 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/b/mb04-black-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/b/mb04-black-0_alt1.jpg', 'stock': 216, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2038, 'sku': '24-WB01', 'name': 'Voyage Yoga Bag', 'short': 'This is a simple product called Voyage Yoga Bag', 'desc': 'Everything you need for a trip to the gym will fit inside this surprisingly spacious Voyage Yoga Bag. Stock it with a water bottle, change of clothes, pair of shoes, and even a few beauty products. Fits inside a locker and zips shut for security. Slip pocket on front. Contrast piping. Durable nylon ', 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/b/wb01-black-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/b/wb01-black-0.jpg', 'stock': 33, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2034, 'sku': '24-MB05', 'name': 'Wayfarer Messenger Bag', 'short': 'This is a simple product called Wayfarer Messenger Bag', 'desc': 'Perfect for class, work or the gym, the Wayfarer Messenger Bag is packed with pockets. The dual-buckle flap closure reveals an organizational panel, and the roomy main compartment has spaces for your laptop and a change of clothes. An adjustable shoulder strap and easy-grip handle promise easy carry', 'category': 'Bags', 'top': 'Gear', 'gender': 'unisex', 'price': 45.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/b/mb05-black-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/b/mb05-black-0.jpg', 'stock': 217, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2022, 'sku': '24-UG06', 'name': 'Affirm Water Bottle', 'short': 'This is a simple product called Affirm Water Bottle', 'desc': "You'll stay hydrated with ease with the Affirm Water Bottle by your side or in hand. Measurements on the outside help you keep track of how much you're drinking, while the screw-top lid prevents spills. A metal carabiner clip allows you to attach it to the outside of a backpack or bag for easy acces", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 7.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/u/g/ug06-lb-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/u/g/ug06-lb-0.jpg', 'stock': 99, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2023, 'sku': '24-UG07', 'name': 'Dual Handle Cardio Ball', 'short': 'This is a simple product called Dual Handle Cardio Ball', 'desc': 'Make the most of your limited workout window with our Dual-Handle Cardio Ball. The 15-lb ball maximizes the effort-impact to your abdominal, upper arm and lower-body muscles. It features a handle on each side for a firm, secure grip. Durable plastic shell with sand fill. Two handles. 15 lbs.', 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 12.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/u/g/ug07-bk-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/u/g/ug07-bk-0_alt1.jpg', 'stock': 78, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2026, 'sku': '24-UG05', 'name': "Go-Get'r Pushup Grips", 'short': "This is a simple product called Go-Get'r Pushup Grips", 'desc': "The Go-Get'r Pushup Grips safely provide the extra range of motion you need for a deep-dip routine targeting core, shoulder, chest and arm strength. Do fewer pushups using more energy, getting better results faster than the standard floor-level technique yield. Durable foam grips. Supportive base.", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 19.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/u/g/ug05-gr-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/u/g/ug05-gr-0.jpg', 'stock': 21, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2030, 'sku': '24-UG03', 'name': 'Harmony Lumaflex&trade; Strength Band Kit', 'short': 'This is a simple product called Harmony Lumaflex&trade; Strength Band Kit', 'desc': 'Forget fancy machines and costly memberships - the Harmony Lumaflex&trade; Stength Band Kit is all you need for an amazing workout. The kit has everything you need for a range of strengthening and toning exercises. Three flex bands. Textured, ergonomic grips. Adjustable lengths. Mesh carry bag inclu', 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 22.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/u/g/ug03-bk-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/u/g/ug03-bk-0.jpg', 'stock': 210, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2025, 'sku': '24-UG02', 'name': 'Pursuit Lumaflex&trade; Tone Band', 'short': 'This is a simple product called Pursuit Lumaflex&trade; Tone Band', 'desc': 'Save your knees and joints while strengthening arms, legs and core with the Pursuit Lumaflex&trade; Tone Band. This ultra-durable training tool lets you complete a full-body workout free of bulky machines or free weights. Home and/or gym use. Ergonomic handles.', 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 16.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/u/g/ug02-bk-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/u/g/ug02-bk-0.jpg', 'stock': 165, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2027, 'sku': '24-UG01', 'name': 'Quest Lumaflex&trade; Band', 'short': 'This is a simple product called Quest Lumaflex&trade; Band', 'desc': "From leg extensions and standing squats to a full range of floor routines, the Quest Strength Band brings your own personal gym with you everywhere you go. Designed to add extra resistance to any training session, this strength band helps you tone and define your muscles. You'll love the extra comfo", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 19.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/u/g/ug01-bk-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/u/g/ug01-bk-0.jpg', 'stock': 127, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2029, 'sku': '24-WG088', 'name': 'Sprite Foam Roller', 'short': 'This is a simple product called Sprite Foam Roller', 'desc': "It hurts so good to use the Sprite Foam Roller on achy, tired muscles for myofascial massage therapy. Or you can add this fundamental piece to your Pilates and yoga accouterment, or apply towards core stability, strengthening and balance exercise. 6'' wide by 12'' long. Safe for myofascial release. ", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 19.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-foam-roller.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-foam-roller.jpg', 'stock': 40, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2028, 'sku': '24-WG084', 'name': 'Sprite Foam Yoga Brick', 'short': 'This is a simple product called Sprite Foam Yoga Brick', 'desc': 'Our top-selling yoga prop, the 4-inch, high-quality Sprite Foam Yoga Brick is popular among yoga novices and studio professionals alike. An essential yoga accessory, the yoga brick is a critical tool for finding balance and alignment in many common yoga poses. Choose from 5 color options. Standard L', 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 5.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-brick.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-brick.jpg', 'stock': 128, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2013, 'sku': '24-WG081-gray', 'name': 'Sprite Stasis Ball 55 cm', 'short': 'This is a simple product called Sprite Stasis Ball 55 cm', 'desc': "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. Durable, b", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 23.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-gray.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-gray.jpg', 'stock': 158, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2014, 'sku': '24-WG081-pink', 'name': 'Sprite Stasis Ball 55 cm', 'short': 'This is a simple product called Sprite Stasis Ball 55 cm', 'desc': "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. Durable, b", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 23.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-pink.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-pink.jpg', 'stock': 68, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2015, 'sku': '24-WG081-blue', 'name': 'Sprite Stasis Ball 55 cm', 'short': 'This is a simple product called Sprite Stasis Ball 55 cm', 'desc': "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. Durable, b", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 23.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball.jpg', 'stock': 182, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2016, 'sku': '24-WG082-gray', 'name': 'Sprite Stasis Ball 65 cm', 'short': 'This is a simple product called Sprite Stasis Ball 65 cm', 'desc': "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. 65 cm plas", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 27.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-gray.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-gray.jpg', 'stock': 72, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2017, 'sku': '24-WG082-pink', 'name': 'Sprite Stasis Ball 65 cm', 'short': 'This is a simple product called Sprite Stasis Ball 65 cm', 'desc': "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. 65 cm plas", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 27.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-pink.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-pink.jpg', 'stock': 208, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2018, 'sku': '24-WG082-blue', 'name': 'Sprite Stasis Ball 65 cm', 'short': 'This is a simple product called Sprite Stasis Ball 65 cm', 'desc': "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. 65 cm plas", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 27.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball.jpg', 'stock': 87, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2019, 'sku': '24-WG083-gray', 'name': 'Sprite Stasis Ball 75 cm', 'short': 'This is a simple product called Sprite Stasis Ball 75 cm', 'desc': "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. 75 cm plas", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-gray.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-gray.jpg', 'stock': 239, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2020, 'sku': '24-WG083-pink', 'name': 'Sprite Stasis Ball 75 cm', 'short': 'This is a simple product called Sprite Stasis Ball 75 cm', 'desc': "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. 75 cm plas", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-pink.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball-pink.jpg', 'stock': 136, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2021, 'sku': '24-WG083-blue', 'name': 'Sprite Stasis Ball 75 cm', 'short': 'This is a simple product called Sprite Stasis Ball 75 cm', 'desc': "The Sprite Stasis Ball gives you the toned abs, sides, and back you want by amping up your core workout. With bright colors and a burst-resistant design, it's a must-have for every hard-core exercise addict. Use for abdominal conditioning, balance training, yoga, or even physical therapy. 75 cm plas", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 32.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-stability-ball.jpg', 'stock': 103, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2012, 'sku': '24-WG087', 'name': 'Sprite Yoga Strap 10 foot', 'short': 'This is a simple product called Sprite Yoga Strap 10 foot', 'desc': "The Sprite Yoga Strap is your untiring partner in demanding stretches, holds and alignment routines. The strap's 100% organic cotton fabric is woven tightly to form a soft, textured yet non-slip surface. The plastic clasp buckle is easily adjustable, lightweight and durable under strain. 10' long x ", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 21.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-strap.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-strap.jpg', 'stock': 149, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2010, 'sku': '24-WG085', 'name': 'Sprite Yoga Strap 6 foot', 'short': 'This is a simple product called Sprite Yoga Strap 6 foot', 'desc': "The Sprite Yoga Strap is your untiring partner in demanding stretches, holds and alignment routines. The strap's 100% organic cotton fabric is woven tightly to form a soft, textured yet non-slip surface. The plastic clasp buckle is easily adjustable, lightweight and durable under strain. 100% soft a", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 14.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-strap.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-strap.jpg', 'stock': 48, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2011, 'sku': '24-WG086', 'name': 'Sprite Yoga Strap 8 foot', 'short': 'This is a simple product called Sprite Yoga Strap 8 foot', 'desc': "The Sprite Yoga Strap is your untiring partner in demanding stretches, holds and alignment routines. The strap's 100% organic cotton fabric is woven tightly to form a soft, textured yet non-slip surface. The plastic clasp buckle is easily adjustable, lightweight and durable under strain. 8' long x 1", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 17.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-strap.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/l/u/luma-yoga-strap.jpg', 'stock': 66, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2024, 'sku': '24-UG04', 'name': 'Zing Jump Rope', 'short': 'This is a simple product called Zing Jump Rope', 'desc': "One of the world's simplest and most portable exercise devices, a jump rope enables endless variations and fitness output. The Zing Jump Rope goes anywhere and can be used any time. It is adjustable in length and has contoured foam handles for a great grip. Contoured foam handles. Adjustable length.", 'category': 'Fitness Equipment', 'top': 'Gear', 'gender': 'unisex', 'price': 12.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/u/g/ug04-bk-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/u/g/ug04-bk-0.jpg', 'stock': 198, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2001, 'sku': '24-MG04', 'name': 'Aim Analog Watch', 'short': 'This is a simple product called Aim Analog Watch', 'desc': 'Stay light-years ahead of the competition with our Aim Analog Watch. The flexible, rubberized strap is contoured to conform to the shape of your wrist for a comfortable all-day fit. The face features three illuminated hands, a digital read-out of the current time, and stopwatch functions. Japanese q', 'category': 'Watches', 'top': 'Gear', 'gender': 'unisex', 'price': 45.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/g/mg04-bk-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/g/mg04-bk-0.jpg', 'stock': 163, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2007, 'sku': '24-WG01', 'name': 'Bolo Sport Watch', 'short': 'This is a simple product called Bolo Sport Watch', 'desc': 'The Bolo Sport Watch is sleek, sporty and sized just right to fit your smaller wrist. Easy to read and set up, it features a large digital face and button-activated alarm and stopwatch. The soft-touch resin band promises no-pinch comfort, while the water-resistant design lets you take your workout t', 'category': 'Watches', 'top': 'Gear', 'gender': 'unisex', 'price': 49.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/g/wg01-bk-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/g/wg01-bk-0.jpg', 'stock': 126, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2008, 'sku': '24-WG03', 'name': 'Clamber Watch', 'short': 'This is a simple product called Clamber Watch', 'desc': 'Keep track of time on the treadmill or trail with our Clamber Watch. The flexible, rubberized strap is contoured to conform to your wrist for a comfortable fit all day. The face features an illuminated digital read-out of the current time and stopwatch functions. Japanese quartz movement. Strap fits', 'category': 'Watches', 'top': 'Gear', 'gender': 'unisex', 'price': 54.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/g/wg03-gr-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/g/wg03-sa-0.jpg', 'stock': 111, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2004, 'sku': '24-MG05', 'name': 'Cruise Dual Analog Watch', 'short': 'This is a simple product called Cruise Dual Analog Watch', 'desc': "Whether you're traveling or wish you were, you'll never let time zones perplex you again with the Cruise Dual Analog Watch. The thick, adjustable band promises a comfortable, personalized fit to this classy, modern time piece. Two dials. Stainless steel case. Adjustable leather band.", 'category': 'Watches', 'top': 'Gear', 'gender': 'unisex', 'price': 55.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/g/mg05-br-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/g/mg05-br-0.jpg', 'stock': 148, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2005, 'sku': '24-MG02', 'name': 'Dash Digital Watch', 'short': 'This is a simple product called Dash Digital Watch', 'desc': 'The Dash Digital Watch will challenge you to push harder and longer. Log workouts by date, average, and segment times, and recharge by setting hydration and nutrition alarms. This watch is styled with a sleek, square face and durable rubber strap for a long life. Digital display. LED backlight. Rubb', 'category': 'Watches', 'top': 'Gear', 'gender': 'unisex', 'price': 92.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/g/mg02-bk-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/g/mg02-bk-0.jpg', 'stock': 16, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2009, 'sku': '24-WG02', 'name': 'Didi Sport Watch', 'short': 'This is a simple product called Didi Sport Watch', 'desc': 'The Didi Sport Watch helps you keep your workout plan down to the second. The vertical, digital face looks sleek and futuristic. This watch is programmed with tons of helpful features such as a timer, an alarm clock, a pedometer, and more to help make your excercise more productive. Digital display.', 'category': 'Watches', 'top': 'Gear', 'gender': 'unisex', 'price': 92.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/g/wg02-bk-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/g/wg02-bk-0.jpg', 'stock': 44, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2002, 'sku': '24-MG01', 'name': 'Endurance Watch', 'short': 'This is a simple product called Endurance Watch', 'desc': "It's easy to track and monitor your training progress with the Endurance Watch. You'll see standard info like time, date and day of the week, but it also functions for the serious high-mileage athete: lap counter, stopwatch, distance, heart rate, speed/pace, cadence and altitude. Digital display. LE", 'category': 'Watches', 'top': 'Gear', 'gender': 'unisex', 'price': 49.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/g/mg01-bk-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/g/mg01-bk-0.jpg', 'stock': 218, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2006, 'sku': '24-WG09', 'name': 'Luma Analog Watch', 'short': 'This is a simple product called Luma Analog Watch', 'desc': "Designed to stand up to your active lifestyle, this women's Luma Analog Watch features a tasteful brushed chrome finish and a stainless steel, water-resistant construction for lasting durability. Precision Miyota&reg; three-hand movement.", 'category': 'Watches', 'top': 'Gear', 'gender': 'unisex', 'price': 43.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/w/g/wg09-gr-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/w/g/wg09-gr-0.jpg', 'stock': 11, 'colors': [], 'sizes': [], 'type': 'simple'}, {'id': 2003, 'sku': '24-MG03', 'name': 'Summit Watch', 'short': 'This is a simple product called Summit Watch', 'desc': 'Trek high and low in the attractive Summit Watch, which features a digital LED display with time and date, stopwatch, lap counter, and 3-second backlight. It can also calculate the number of steps taken and calories burned. Brushed metal case. Water resistant (100 meters). Buckle clasp. Strap fits 7', 'category': 'Watches', 'top': 'Gear', 'gender': 'unisex', 'price': 54.0, 'sale': 0, 'img': 'http://eimages.valtim.com/acme-images/product/m/g/mg03-br-0.jpg', 'img2': 'http://eimages.valtim.com/acme-images/product/m/g/mg03-br-0.jpg', 'stock': 93, 'colors': [], 'sizes': [], 'type': 'simple'}]
print(len(Products))
products_sorted = sorted(Products, key=lambda p: p["id"])
# for p in Products :
#     print (p)
# print([p['id'] for p in Products] )
# print([sp['id']for sp in products_sorted] )
# print (products_sorted)
# for p in products_sorted:
#     print(p)
# for p in products_sorted:
#     if p['id'] == 2033 :
#         print (p)

import pandas as pd
demo_products = pd.DataFrame(products_sorted)
counter = 0
file = pd.read_csv(r"D:\phoenix user\Downloads\Woo_Product_Dummy_Data_Set_Simple_and_Variable - Copy (2).csv")
file["Name"] = file["Name"].str.strip()

# print (file.dtypes)
ids = set(file["ID"])
print(len(demo_products['id']))
skus = set(file["SKU"])
print(len(demo_products['sku']))
csv_products = pd.DataFrame(columns=file.columns)
print (csv_products.columns)
for _, p in demo_products.iterrows():
    
    if p["sku"] in skus:
        
        match = file[ (file["SKU"] == p["sku"]) &  (file["Name"] == p["name"])]
        
        if not  match.empty:
            # counter += 1
            # print(p["id"], p["sku"], "MATCH FOUND")
            match['ID'] = p['id']
            # print('match',match)
            csv_products.loc[len(csv_products)] = match.iloc[0]
        else :
            # print(p["id"], p["sku"],p['name'], "MATCH not foudn ono p 111") 
            file_p = file[file['SKU'] == p['sku']]
            # print(file_p["ID"], file_p["SKU"],file_p['Name'], "MATCH not foudn on file 222") 
            # print(file_p.iloc[0]["ID"], file_p.iloc[0]["SKU"], file_p.iloc[0]["Name"] ,"MATCH not foudn on file 222")
print ('csv prducts',len(csv_products))
# for _, p in demo_products.iterrows():

#     if p["id"] in ids:
    
#         match = csv_products[(csv_products["ID"] == p["id"]) & (csv_products["SKU"] == p["sku"])]
    
#         if not match.empty:
#             counter += 1
#             print(p["id"], p["sku"], "MATCH FOUND")
#             fp = csv_products[csv_products['ID'] == p["id"]]
#             print(fp['ID'].values[0] , fp['SKU'].values[0])
            
# print ('counter = ' ,counter)
print(csv_products.info )
csv_products.to_csv('products.csv', index=False)


191
191
191
Index(['ID', 'Type', 'SKU', 'Name', 'Published', 'Is featured?',
       'Visibility in catalog', 'Short description', 'description',
       'Date sale price starts', 'Date sale price ends', 'Tax status',
       'Tax class', 'In stock?', 'Stock', 'Backorders allowed?',
       'Sold individually?', 'Weight (lbs)', 'Length (in)', 'Width (in)',
       'Height (in)', 'Allow customer reviews?', 'Purchase note', 'Sale price',
       'Regular price', 'Categories', 'Tags', 'Shipping class', 'Images',
       'Download limit', 'Download expiry days', 'Parent', 'Grouped products',
       'Upsells', 'Cross-sells', 'External URL', 'Button text', 'Position',
       'Attribute 1 name', 'Attribute 1 value(s)', 'Attribute 2 name',
       'Attribute 2 value(s)', 'Attribute 3 name', 'Attribute 3 value(s)',
       'Attribute 4 name', 'Attribute 4 value(s)', 'Attribute 5 name',
       'Attribute 5 value(s)', 'Meta: _wpcom_is_markdown', 'Download 1 name',
       'Download 1 URL', 'Download 2 name

In [ ]:
response="setup_complete=None server_content=LiveServerContent(\n  input_transcription=Transcription(\n    text=' Hi, can you help me?'\n  )\n) tool_call=None tool_call_cancellation=None usage_metadata=None go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None"
print (response)

In [ ]:
esponse='setup_complete=None server_content=LiveServerContent(\n  model_turn=Content(\n    parts=[\n      Part(\n        text="""**Acknowledge Shopping Request**\n\nI\'ve registered the user\'s initial greeting and request for assistance. My focus is now on responding in a friendly way. I\'ll frame my response as an offer to help with their shopping, keeping it concise and helpful. I need to be sure to adhere to the "friendly Voice Shopping Assistant" instructions.\n\n\n""",\n        thought=True\n      ),\n    ]\n  )\n) tool_call=None tool_call_cancellation=None usage_metadata=None go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None'
print (esponse)

In [ ]:
response="setup_complete=None server_content=LiveServerContent(\n  turn_complete=True\n) tool_call=None tool_call_cancellation=None usage_metadata=UsageMetadata(\n  prompt_token_count=1021,\n  prompt_tokens_details=[\n    ModalityTokenCount(\n      modality=<MediaModality.TEXT: 'TEXT'>,\n      token_count=1020\n    ),\n    ModalityTokenCount(\n      modality=<MediaModality.AUDIO: 'AUDIO'>,\n      token_count=1\n    ),\n  ],\n  response_token_count=63,\n  response_tokens_details=[\n    ModalityTokenCount(\n      modality=<MediaModality.AUDIO: 'AUDIO'>,\n      token_count=63\n    ),\n  ],\n  thoughts_token_count=47,\n  total_token_count=1084\n) go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None"
print (response)

In [ ]:
response="setup_complete=None server_content=LiveServerContent(\n  model_turn=Content(\n    parts=[\n      Part(\n        inline_data=Blob(\n          data=b'\\xf4\\xff\\xf3\\xff\\xf6\\xff\\xf5\\xff\\xf8\\xff\\xf7\\xff\\xfb\\xff\\xfb\\xff\\xfd\\xff\\xfe\\xff\\xff\\xff\\xfd\\xff\\x00\\x00\\x02\\x00\\x03\\x00\\x03\\x00\\x03\\x00\\x04\\x00\\x04\\x00\\x05\\x00\\x04\\x00\\x04\\x00\\x03\\x00\\x05\\x00\\x04\\x00\\x08\\x00\\x07\\x00\\n\\x00\\t\\x00\\x0e\\x00\\x0b\\x00\\x0b\\x00\\x0c\\x00\\x0c\\x00\\x0b\\x00\\n\\x00\\t\\x00\\n\\x00\\x08\\x00\\x08\\x00\\x06\\x00\\x05\\x00\\x03\\x00\\x03\\x00\\x02\\x00\\xff\\xff\\xff\\xff\\xfd\\xff\\xfa...',\n          mime_type='audio/pcm;rate=24000'\n        )\n      ),\n    ]\n  )\n) tool_call=None tool_call_cancellation=None usage_metadata=None go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None"
print (response)

In [ ]:
response="setup_complete=None server_content=LiveServerContent(\n  turn_complete=True\n) tool_call=None tool_call_cancellation=None usage_metadata=UsageMetadata(\n  prompt_token_count=3732,\n  prompt_tokens_details=[\n    ModalityTokenCount(\n      modality=<MediaModality.TEXT: 'TEXT'>,\n      token_count=3624\n    ),\n    ModalityTokenCount(\n      modality=<MediaModality.AUDIO: 'AUDIO'>,\n      token_count=108\n    ),\n  ],\n  response_token_count=103,\n  response_tokens_details=[\n    ModalityTokenCount(\n      modality=<MediaModality.AUDIO: 'AUDIO'>,\n      token_count=103\n    ),\n  ],\n  thoughts_token_count=66,\n  total_token_count=3835\n) go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None"
print (response)


In [ ]:
response='setup_complete=None server_content=LiveServerContent(\n  model_turn=Content(\n    parts=[\n      Part(\n        text="""**Confirming Specific Product**\n\nI\'ve determined the user wants the "Nike Air Zoom Pegasus 40" with product ID 1, but I need to ensure this is the intended product. The user\'s use of "one" confirms a quantity of one, so I will now verify this specific item is correct based on their previous searches. I must verify this selection before proceeding.\n\n\n""",\n        thought=True\n      ),\n    ]\n  )\n) tool_call=None tool_call_cancellation=None usage_metadata=None go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None'
print (response)

In [ ]:
memory=[{'role': 'user', 'text': 'Hi'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' How'}, {'role': 'user', 'text': ' are'}, {'role': 'user', 'text': ' you'}, {'role': 'user', 'text': '?'}, {'role': 'ai', 'text': "I'm doing well, thank you! Are you looking for anything specific in the store?"}, {'role': 'user', 'text': ' can'}, {'role': 'user', 'text': ' you'}, {'role': 'user', 'text': ' help'}, {'role': 'user', 'text': ' me'}, {'role': 'user', 'text': ' with showing?'}, {'role': 'ai', 'text': 'Sure! What kind of products are you interested in?'}, {'role': 'user', 'text': 'Lest show what available'}, {'role': 'ai', 'text': 'What kind of products are you looking for? For example, shoes, electronics, or clothing?'}]
memory=[{'role': 'user', 'text': 'Hi'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': 'What is your name'}, {'role': 'ai', 'text': "I don't have a specific name, I'm your voice shopping assistant."}, {'role': 'ai', 'text': 'I can search for products, get details about them, show your cart, and add or remove items. What are you looking for?'}, {'role': 'user', 'text': ' Okay, what you can help me with?'}, {'role': 'ai', 'text': 'How about some running shoes? Or are you interested in something else?'}, {'role': 'user', 'text': ' Uh anything'}]
memory=[{'role': 'user', 'text': 'Hi how are you'}, {'role': 'ai', 'text': "Hello! I'm good, thank you. How can I help you with your shopping today?"}, {'role': 'ai', 'text': 'I am your voice shopping assistant. How can I help you find anything?'}, {'role': 'user', 'text': " What's your name?"}, {'role': 'ai', 'text': 'I can help you search for products, check your shopping cart, or provide product details. What are you looking for today?'}, {'role': 'user', 'text': ' what you can meet me and brothers'}, {'role': 'ai', 'text': 'We currently have the North Face ThermoBall Jacket available.'}, {'role': 'user', 'text': ' Вот какие же продукты с'}, {'role': 'user', 'text': 'What others'}, {'role': 'ai', 'text': "Is there a specific category or type of product you're interested in?"}]
memory=[{'role': 'user', 'text': 'Hi how aer o'}, {'role': 'ai', 'text': 'I am good, thank you! How can I help you with your shopping today?'}, {'role': 'user', 'text': 'was your IM'}, {'role': 'ai', 'text': "I don't have a name. I'm just a voice shopping assistant. What are you looking for?"}, {'role': 'user', 'text': 'what is product zero'}, {'role': 'ai', 'text': "I'm sorry, but I can't find a product with product ID 0. Is there anything specific you're searching for?"}, {'role': 'user', 'text': 'I mean what is category'}, {'role': 'ai', 'text': 'There\'s no category called "zero". What type of products are you interested in?'}, {'role': 'user', 'text': 'Okay I mean what what is available'}, {'role': 'ai', 'text': 'We have various categories like shoes, clothing, and electronics. What would you like to browse?'}, {'role': 'user', 'text': 'Great lest start with electronics'}, {'role': 'ai', 'text': 'I\'m sorry, but I didn\'t find any products in the "electronics" category. Is there another category you\'re interested in?'}]
memory=[{'role': 'user', 'text': ' Hi, how are you?'}, {'role': 'ai', 'text': "Hello! I'm good, thank you. How can I help you with your shopping today?"}, {'role': 'user', 'text': ' what you can do with'}, {'role': 'ai', 'text': 'I can search for products, show you details, manage your cart, and more. What are you looking for?'}, {'role': 'user', 'text': ' you want to hit it.'}, {'role': 'ai', 'text': "Sorry, I didn't find any products matching 'headset'. Is there something else you're interested in?"}, {'role': 'user', 'text': ' what about?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99 and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': ' what you'}, {'role': 'ai', 'text': 'Both are great choices! The Sony is renowned for noise cancellation and long battery life, while the Apple provides excellent active noise cancellation and transparency modes. It really depends on your priority.'}, {'role': 'user', 'text': ' I think we can go with Sony.'}, {'role': 'ai', 'text': 'Sure! Shall I add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': 'Add 2 of it'}, {'role': 'user', 'text': ' yes added.'}, {'role': 'ai', 'text': 'Done! I have added the Sony WH-1000XM5 to your cart. Your current total is $349.99. Anything else you need?'}, {'role': 'ai', 'text': 'You currently have one in your cart. Shall I change the quantity to 2?'}, {'role': 'user', 'text': 'Yes and add on air pods'}, {'role': 'ai', 'text': "I've updated the Sony quantity to 2 and added the Apple AirPods Pro. Your total is $1299.96. Is there anything else?"}, {'role': 'user', 'text': 'This is good form you,. So on ht cart'}, {'role': 'ai', 'text': 'Your cart has 3x Sony WH-1000XM5 and 1x Apple AirPods Pro, for a total of $1299.96. Are you ready to checkout?'}, {'role': 'user', 'text': 'Add some hoodies'}, {'role': 'ai', 'text': "I'm sorry, I couldn't find any hoodies. Is there something else you were looking for?"}, {'role': 'user', 'text': ' you can search for'}, {'role': 'ai', 'text': 'I found two wireless earbuds: the Sony WH-1000XM5 for $349.99 and the Apple AirPods Pro (2nd Gen) for $249.99. Did you want to add one of those?'}, {'role': 'user', 'text': ' You can search for holdings.'}, {'role': 'ai', 'text': "As I mentioned before, I can't find any hoodies. Perhaps a jacket or sweater?"}, {'role': 'user', 'text': ' try to get all this'}, {'role': 'ai', 'text': 'Could you clarify what "all this" you would like to get? Are you referring to the Sony and Apple earbuds in your cart?'}, {'role': 'user', 'text': ' to all products of clauses.'}, {'role': 'ai', 'text': 'I\'m sorry, I can only search for products by keywords or specific categories, and I don\'t have a general "clothes" category. What specific types of clothing are you interested in?'}, {'role': 'user', 'text': " I'm sure this in the product is as category was closing."}, {'role': 'ai', 'text': "I understand, but my search function didn't find any hoodies using the 'clothing' category. We can try searching for specific types of clothing or other terms if you'd like."}, {'role': 'user', 'text': 'Are you sure you cna find this product Champion Reverse Weave Hoodie'}, {'role': 'ai', 'text': 'Yes, I found it! We have the Champion Reverse Weave Hoodie for $59.99. Would you like to add it to your cart?'}, {'role': 'user', 'text': " Yes, I didn't."}, {'role': 'ai', 'text': "Great! I've added the Champion Hoodie to your cart. Is there anything else?"}, {'role': 'user', 'text': ' no'}, {'role': 'ai', 'text': "Okay. Your current total is $1359.95. Let me know if you're ready to checkout."}, {'role': 'user', 'text': ' Yes. Yes.'}, {'role': 'ai', 'text': 'Great! To complete your purchase, please proceed to the checkout area. Is there anything else I can assist you with today?'}, {'role': 'user', 'text': ' What is final chapter'}, {'role': 'ai', 'text': 'Your final total is $1359.95.'}, {'role': 'user', 'text': ' . Give me fall card.'}, {'role': 'ai', 'text': 'Certainly. Your cart contains: 2x Sony WH-1000XM5, 1x Apple AirPods Pro (2nd Gen), and 1x Champion Reverse Weave Hoodie, for a total of $1359.95. Is that correct?'}, {'role': 'user', 'text': ' yes'}, {'role': 'ai', 'text': 'Perfect! You can proceed to checkout to finalize your order. Thank you for shopping with us!'}]

for msg in memory:
    print(msg)


In [ ]:
memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': "I've added the Sony headphones to your cart. Is there anything else?"}, {'role': 'user', 'text': 'What about jeans'}, {'role': 'ai', 'text': "I found Levi's 501 Original Jeans for $69.99. Are you interested in these?"}, {'role': 'user', 'text': 'Good add it'}, {'role': 'ai', 'text': "Shall I add the Levi's 501 Original Jeans to your cart?"}, {'role': 'user', 'text': 'Ues and add 2 of it'}, {'role': 'ai', 'text': "I've added 2 quantities of the Levi's 501 Jeans to your cart. Is there anything else I can help you with?"}, {'role': 'user', 'text': ' Wat een kracht.'}, {'role': 'ai', 'text': "Your cart contains: 1 Champion Reverse Weave Hoodie, 1 Sony WH-1000XM5 and 2 Levi's 501 Original Jeans. The total is $549.96. Ready to checkout?"}]
for item in memory:
    print(item)
   

2026-03-26T22:47:23.556504Z
2026-03-26T22:49:59.371183Z 


026-03-26T22:53:11.917993Z

I've determined the user wants the "Nike Air Zoom Pegasus 40" with product ID 1, but I need to ensure this is the intended product. The user's use of "one" confirms a quantity of one, so I will now verify this specific item is correct based on their previous searches. I must verify this selection before proceeding

(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent> & p:\AI_Empire\Projects\voice-commerce-agent\.venv\Scripts\Activate.ps1
(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent> & p:\AI_Empire\Projects\voice-commerce-agent\.venv\Scripts\Activate.ps1
(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent>
 *  History restored 

PS P:\AI_Empire\Projects\voice-commerce-agent> & p:\AI_Empire\Projects\voice-commerce-agent\.venv\Scripts\Activate.ps1
(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent> 
 *  History restored 

PS P:\AI_Empire\Projects\voice-commerce-agent> & p:\AI_Empire\Projects\voice-commerce-agent\.venv\Scripts\Activate.ps1
(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent> uv run uvicorn src.voice_commerce.main:app --reload
INFO:     Will watch for changes in these directories: ['P:\\AI_Empire\\Projects\\voice-commerce-agent']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [19424] using WatchFiles
INFO:     Started server process [19312]
INFO:     Waiting for application startup.
2026-03-26T20:12:13.053031Z [info     ] voice_commerce_agent_starting  [src.voice_commerce.main] debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
2026-03-26T20:12:16.679647Z [info     ] voice_handler_created          [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2
INFO:     127.0.0.1:10612 - "WebSocket /ws/voice?session_id=v5_mph3j0z2" [accepted]
2026-03-26T20:12:16.680167Z [info     ] ws_connected                   [voice_commerce.handlers.voice_websocket_handler] client="Address(host='127.0.0.1', port=10612)" session_id=v5_mph3j0z2
INFO:     connection open
2026-03-26T20:12:16.941470Z [info     ] gemini_session_connecting      [voice_commerce.core.voice.gemini_live_handler] model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-26T20:12:17.608773Z [info     ] gemini_session_connected       [voice_commerce.core.voice.gemini_live_handler]
INFO:     127.0.0.1:6023 - "GET /static/test_client.html HTTP/1.1" 304 Not Modified
INFO:     connection closed
2026-03-26T20:12:18.376701Z [info     ] gemini_session_closing         [voice_commerce.core.voice.gemini_live_handler]
2026-03-26T20:12:18.534793Z [info     ] gemini_session_closed_by_server [voice_commerce.core.voice.gemini_live_handler] reason='1000 None. '
2026-03-26T20:12:18.535323Z [info     ] gemini_session_closed_gracefully [voice_commerce.handlers.voice_websocket_handler] reason='1000 None. ' session=v5_mph3j0z2
2026-03-26T20:12:18.535847Z [info     ] gemini_session_closed          [voice_commerce.core.voice.gemini_live_handler]
2026-03-26T20:12:18.536377Z [info     ] ws_disconnected_normally       [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 turns=0
2026-03-26T20:12:18.536377Z [info     ] ws_handler_finished            [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 total_turns=0
2026-03-26T20:12:18.812149Z [info     ] voice_handler_created          [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2
INFO:     127.0.0.1:5957 - "WebSocket /ws/voice?session_id=v5_mph3j0z2" [accepted]
2026-03-26T20:12:18.812149Z [info     ] ws_connected                   [voice_commerce.handlers.voice_websocket_handler] client="Address(host='127.0.0.1', port=5957)" session_id=v5_mph3j0z2
INFO:     connection open
2026-03-26T20:12:18.888479Z [info     ] gemini_session_connecting      [voice_commerce.core.voice.gemini_live_handler] model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-26T20:12:20.015171Z [info     ] gemini_session_connected       [voice_commerce.core.voice.gemini_live_handler]
2026-03-26T20:12:20.711320Z [info     ] input_mode_switched_to_audio   [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2
2026-03-26T20:12:24.232599Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Thôi.'
INFO:     127.0.0.1:12462 - "GET /favicon.ico HTTP/1.1" 404 Not Found
2026-03-26T20:12:26.866114Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}]
2026-03-26T20:12:30.713359Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' What'
2026-03-26T20:12:30.714893Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' is my name?'
2026-03-26T20:12:38.088775Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}]
2026-03-26T20:12:43.111597Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Okay'
2026-03-26T20:12:44.907326Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' my'
2026-03-26T20:12:44.909488Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' name'
2026-03-26T20:12:44.910510Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' is'
2026-03-26T20:12:49.085829Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}]
2026-03-26T20:13:00.345693Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' اي والله دي'
2026-03-26T20:13:00.706061Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'query': 'هدايا'} call_id=function-call-12340314771335619223 session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:00.707093Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'query': 'هدايا'} session=v5_mph3j0z2 tool_name=search_products
2026-03-26T20:13:00.707624Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'query': 'هدايا'} session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:00.708661Z [info     ] search_products                [voice_commerce.core.tools.product_tools] max_price=None query=هدايا
2026-03-26T20:13:00.710162Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=100 tool=search_products
2026-03-26T20:13:00.711746Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="I didn't find anything matching 'هدايا'. Try different keywords, or ask me what " session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:00.713549Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'query': 'هدايا'} call_id=function-call-12340314771335619223 function_call=FunctionCall(      
  args={
    'query': 'هدايا'
  },
  id='function-call-12340314771335619223',
  name='search_products'
)
2026-03-26T20:13:11.100242Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}]
2026-03-26T20:13:14.686226Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' I'
2026-03-26T20:13:14.856672Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' mean'
2026-03-26T20:13:15.000955Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' I'
2026-03-26T20:13:15.368847Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' want'
2026-03-26T20:13:15.752175Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' a holy closing.'
2026-03-26T20:13:16.206677Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'query': 'hoodie clothing'} call_id=function-call-8040951186187757545 session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:16.207227Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'query': 'hoodie clothing'} session=v5_mph3j0z2 tool_name=search_products
2026-03-26T20:13:16.207227Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'query': 'hoodie clothing'} session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:16.208281Z [info     ] search_products                [voice_commerce.core.tools.product_tools] max_price=None query='hoodie clothing'
2026-03-26T20:13:16.208788Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=296 tool=search_products
2026-03-26T20:13:16.209319Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="Found 2 products for 'hoodie clothing':\n• Levi's 501 Original Jeans — $69.99 — i" session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:16.210334Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'query': 'hoodie clothing'} call_id=function-call-8040951186187757545 function_call=FunctionCall(
  args={
    'query': 'hoodie clothing'
  },
  id='function-call-8040951186187757545',
  name='search_products'
)
2026-03-26T20:13:28.504365Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}]
2026-03-26T20:13:35.163588Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Okay'
2026-03-26T20:13:35.325770Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' second one.'
2026-03-26T20:13:39.582613Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}]
2026-03-26T20:13:42.903594Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' E ele'
2026-03-26T20:13:43.139460Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'product_id': 8, 'quantity': 1} call_id=function-call-7697120716237827354 session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:13:43.141072Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'product_id': 8, 'quantity': 1} session=v5_mph3j0z2 tool_name=add_to_cart
2026-03-26T20:13:43.141072Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'product_id': '8', 'quantity': '1'} session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:13:43.141072Z [info     ] add_to_cart                    [voice_commerce.core.tools.cart_tools] product_id=8 quantity=1 session=v5_mph3j0z2
2026-03-26T20:13:43.142598Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=94 tool=add_to_cart
2026-03-26T20:13:43.142598Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview='Added 1× Champion Reverse Weave Hoodie ($59.99 each) to your cart.\nCart total: $' session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:13:43.144180Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'product_id': 8, 'quantity': 1} call_id=function-call-7697120716237827354 function_call=FunctionCall(
  args={
    'product_id': 8,
    'quantity': 1
  },
  id='function-call-7697120716237827354',
  name='add_to_cart'
)
2026-03-26T20:13:49.637101Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}]  
2026-03-26T20:13:56.400325Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' wha'
2026-03-26T20:13:56.586759Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text='t is'
2026-03-26T20:13:56.616779Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' a very'
2026-03-26T20:13:56.864384Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'query': 'belt'} call_id=function-call-15346215065980089208 session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:56.864902Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'query': 'belt'} session=v5_mph3j0z2 tool_name=search_products
2026-03-26T20:13:56.864902Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'query': 'belt'} session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:56.864902Z [info     ] search_products                [voice_commerce.core.tools.product_tools] max_price=None query=belt
2026-03-26T20:13:56.866506Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=99 tool=search_products
2026-03-26T20:13:56.866506Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="I didn't find anything matching 'belt'. Try different keywords, or ask me what c" session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:56.866506Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'query': 'belt'} call_id=function-call-15346215065980089208 function_call=FunctionCall(       
  args={
    'query': 'belt'
  },
  id='function-call-15346215065980089208',
  name='search_products'
)
2026-03-26T20:14:05.856963Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}]
2026-03-26T20:14:22.847509Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' ele'
2026-03-26T20:14:23.039275Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=ctro
2026-03-26T20:14:23.228339Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=nics
2026-03-26T20:14:23.289778Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text='. Can you hear me?'
2026-03-26T20:14:29.000453Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}]
2026-03-26T20:14:31.971384Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Yes.'
2026-03-26T20:14:32.320323Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'query': 'electronics', 'category': 'electronics'} call_id=function-call-2888346715497691110
 session=v5_mph3j0z2 tool=search_products
2026-03-26T20:14:32.320323Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'query': 'electronics', 'category': 'electronics'} session=v5_mph3j0z2 tool_name=search_products
2026-03-26T20:14:32.321933Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'query': 'electronics', 'category': 'electronics'} session=v5_mph3j0z2 tool=search_products       
2026-03-26T20:14:32.322469Z [info     ] search_products                [voice_commerce.core.tools.product_tools] max_price=None query=electronics
2026-03-26T20:14:32.322469Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=106 tool=search_products
2026-03-26T20:14:32.323034Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="I didn't find anything matching 'electronics'. Try different keywords, or ask me" session=v5_mph3j0z2 tool=search_products
2026-03-26T20:14:32.323034Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'query': 'electronics', 'category': 'electronics'} call_id=function-call-2888346715497691110 function_call=FunctionCall(
  args={
    'category': 'electronics',
    'query': 'electronics'
  },
  id='function-call-2888346715497691110',
  name='search_products'
)
2026-03-26T20:14:44.154786Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}]
2026-03-26T20:14:47.151939Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Ye'
2026-03-26T20:14:47.345071Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=s
2026-03-26T20:14:47.537790Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' wha'
2026-03-26T20:14:47.538302Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text='t is headphone?'
2026-03-26T20:14:47.846869Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'query': 'headphones'} call_id=function-call-374181651143909031 session=v5_mph3j0z2 tool=search_products
2026-03-26T20:14:47.846869Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'query': 'headphones'} session=v5_mph3j0z2 tool_name=search_products
2026-03-26T20:14:47.847920Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'query': 'headphones'} session=v5_mph3j0z2 tool=search_products
2026-03-26T20:14:47.848435Z [info     ] search_products                [voice_commerce.core.tools.product_tools] max_price=None query=headphones
2026-03-26T20:14:47.848435Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=306 tool=search_products
2026-03-26T20:14:47.848435Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="Found 2 products for 'headphones':\n• Sony WH-1000XM5 — $349.99 — in stock — ID:4" session=v5_mph3j0z2 tool=search_products
2026-03-26T20:14:47.849956Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'query': 'headphones'} call_id=function-call-374181651143909031 function_call=FunctionCall(   
  args={
    'query': 'headphones'
  },
  id='function-call-374181651143909031',
  name='search_products'
)
2026-03-26T20:15:01.260870Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}]
2026-03-26T20:15:05.382782Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Okay'
2026-03-26T20:15:05.718132Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=', yeah'
2026-03-26T20:15:05.854176Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=,
2026-03-26T20:15:06.053346Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' let'
2026-03-26T20:15:06.185436Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text="'s"
2026-03-26T20:15:06.359878Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' go'
2026-03-26T20:15:06.530489Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' to'
2026-03-26T20:15:09.424562Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Sony.'
2026-03-26T20:15:15.093783Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}]
2026-03-26T20:15:24.004234Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Yes.'
2026-03-26T20:15:24.662894Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'product_id': 4, 'quantity': 1} call_id=function-call-14236623242608552161 session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:15:24.664421Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'product_id': 4, 'quantity': 1} session=v5_mph3j0z2 tool_name=add_to_cart
2026-03-26T20:15:24.664421Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'product_id': '4', 'quantity': '1'} session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:15:24.664421Z [info     ] add_to_cart                    [voice_commerce.core.tools.cart_tools] product_id=4 quantity=1 session=v5_mph3j0z2
2026-03-26T20:15:24.664421Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=83 tool=add_to_cart
2026-03-26T20:15:24.666071Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview='Added 1× Sony WH-1000XM5 ($349.99 each) to your cart.\nCart total: $409.98 (2 ite' session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:15:24.666602Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'product_id': 4, 'quantity': 1} call_id=function-call-14236623242608552161 function_call=FunctionCall(
  args={
    'product_id': 4,
    'quantity': 1
  },
  id='function-call-14236623242608552161',
  name='add_to_cart'
)
2026-03-26T20:15:30.056943Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': "I've added the Sony headphones to your cart. Is there anything else?"}]
2026-03-26T20:15:34.531292Z [info     ] text_message_received          [voice_commerce.handlers.voice_websocket_handler] preview='What about jeans' session_id=v5_mph3j0z2
2026-03-26T20:15:34.935008Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'query': 'jeans'} call_id=function-call-8626671772119431720 session=v5_mph3j0z2 tool=search_products
2026-03-26T20:15:34.935562Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'query': 'jeans'} session=v5_mph3j0z2 tool_name=search_products
2026-03-26T20:15:34.935562Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'query': 'jeans'} session=v5_mph3j0z2 tool=search_products
2026-03-26T20:15:34.936576Z [info     ] search_products                [voice_commerce.core.tools.product_tools] max_price=None query=jeans
2026-03-26T20:15:34.936576Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=174 tool=search_products
2026-03-26T20:15:34.937126Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="Found 1 product for 'jeans':\n• Levi's 501 Original Jeans — $69.99 — in stock — I" session=v5_mph3j0z2 tool=search_products
2026-03-26T20:15:34.938186Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'query': 'jeans'} call_id=function-call-8626671772119431720 function_call=FunctionCall(       
  args={
    'query': 'jeans'
  },
  id='function-call-8626671772119431720',
  name='search_products'
)
2026-03-26T20:15:42.316088Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': "I've added the Sony headphones to your cart. Is there anything else?"}, {'role': 'user', 'text': 'What about jeans'}, {'role': 'ai', 'text': "I found Levi's 501 Original Jeans for $69.99. Are you interested in these?"}]
2026-03-26T20:15:48.066540Z [info     ] text_message_received          [voice_commerce.handlers.voice_websocket_handler] preview='Good add it' session_id=v5_mph3j0z2
2026-03-26T20:15:52.886734Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': "I've added the Sony headphones to your cart. Is there anything else?"}, {'role': 'user', 'text': 'What about jeans'}, {'role': 'ai', 'text': "I found Levi's 501 Original Jeans for $69.99. Are you interested in these?"}, {'role': 'user', 'text': 'Good add it'}, {'role': 'ai', 'text': "Shall I add the Levi's 501 Original Jeans to your cart?"}]
2026-03-26T20:16:00.019591Z [info     ] text_message_received          [voice_commerce.handlers.voice_websocket_handler] preview='Ues and add 2 of it' session_id=v5_mph3j0z2
2026-03-26T20:16:00.497847Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'product_id': 7, 'quantity': 2} call_id=function-call-6455127725345814488 session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:16:00.498377Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'product_id': 7, 'quantity': 2} session=v5_mph3j0z2 tool_name=add_to_cart
2026-03-26T20:16:00.498377Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'product_id': '7', 'quantity': '2'} session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:16:00.499391Z [info     ] add_to_cart                    [voice_commerce.core.tools.cart_tools] product_id=7 quantity=2 session=v5_mph3j0z2
2026-03-26T20:16:00.499898Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=92 tool=add_to_cart
2026-03-26T20:16:00.499898Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="Added 2× Levi's 501 Original Jeans ($69.99 each) to your cart.\nCart total: $549." session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:16:00.500948Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'product_id': 7, 'quantity': 2} call_id=function-call-6455127725345814488 function_call=FunctionCall(
  args={
    'product_id': 7,
    'quantity': 2
  },
  id='function-call-6455127725345814488',
  name='add_to_cart'
)
2026-03-26T20:16:08.143949Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': "I've added the Sony headphones to your cart. Is there anything else?"}, {'role': 'user', 'text': 'What about jeans'}, {'role': 'ai', 'text': "I found Levi's 501 Original Jeans for $69.99. Are you interested in these?"}, {'role': 'user', 'text': 'Good add it'}, {'role': 'ai', 'text': "Shall I add the Levi's 501 Original Jeans to your cart?"}, {'role': 'user', 'text': 'Ues and add 2 of it'}, {'role': 'ai', 'text': "I've added 2 quantities of the Levi's 501 Jeans to your cart. Is there anything else I can help you with?"}]
2026-03-26T20:16:08.519179Z [info     ] input_mode_switched_to_audio   [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2
2026-03-26T20:16:11.301296Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Wat een kracht.'
2026-03-26T20:16:11.598628Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={} call_id=function-call-3075882724188559665 session=v5_mph3j0z2 tool=show_cart
2026-03-26T20:16:11.599643Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={} session=v5_mph3j0z2 tool_name=show_cart
2026-03-26T20:16:11.600158Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={} session=v5_mph3j0z2 tool=show_cart
2026-03-26T20:16:11.600158Z [info     ] show_cart                      [voice_commerce.core.tools.cart_tools] session=v5_mph3j0z2
2026-03-26T20:16:11.600158Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=237 tool=show_cart
2026-03-26T20:16:11.601172Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview='Your cart:\n  • 1× Champion Reverse Weave Hoodie — $59.99 each = $59.99\n  • 1× So' session=v5_mph3j0z2 tool=show_cart
2026-03-26T20:16:11.601172Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={} call_id=function-call-3075882724188559665 function_call=FunctionCall(
  args={},
  id='function-call-3075882724188559665',
  name='show_cart'
)
2026-03-26T20:16:27.545624Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': "I've added the Sony headphones to your cart. Is there anything else?"}, {'role': 'user', 'text': 'What about jeans'}, {'role': 'ai', 'text': "I found Levi's 501 Original Jeans for $69.99. Are you interested in these?"}, {'role': 'user', 'text': 'Good add it'}, {'role': 'ai', 'text': "Shall I add the Levi's 501 Original Jeans to your cart?"}, {'role': 'user', 'text': 'Ues and add 2 of it'}, {'role': 'ai', 'text': "I've added 2 quantities of the Levi's 501 Jeans to your cart. Is there anything else I can help you with?"}, {'role': 'user', 'text': ' Wat een kracht.'}, {'role': 'ai', 'text': "Your cart contains: 1 Champion Reverse Weave Hoodie, 1 Sony WH-1000XM5 and 2 Levi's 501 Original Jeans. The total is $549.96. Ready to checkout?"}]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\core\voice\gemini_live_handler.py'. Reloading...
 INFO:     Shutting down
INFO:     connection closed
2026-03-26T20:16:56.306203Z [info     ] gemini_session_closing         [voice_commerce.core.voice.gemini_live_handler]
INFO:     Waiting for background tasks to complete. (CTRL+C to force quit)
2026-03-26T20:16:56.486442Z [info     ] gemini_session_closed_by_server [voice_commerce.core.voice.gemini_live_handler] reason='1000 None. '
2026-03-26T20:16:56.486442Z [info     ] gemini_session_closed_gracefully [voice_commerce.handlers.voice_websocket_handler] reason='1000 None. ' session=v5_mph3j0z2
2026-03-26T20:16:56.488381Z [info     ] gemini_session_closed          [voice_commerce.core.voice.gemini_live_handler]
2026-03-26T20:16:56.488381Z [info     ] ws_disconnected_normally       [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 turns=34
2026-03-26T20:16:56.489997Z [info     ] ws_handler_finished            [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 total_turns=34
INFO:     Waiting for application shutdown.
2026-03-26T20:16:56.523596Z [info     ] Application shutting down      [src.voice_commerce.main]
INFO:     Application shutdown complete.
INFO:     Finished server process [19312]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\core\voice\gemini_live_handler.py'. Reloading...
 INFO:     Started server process [22228]
INFO:     Waiting for application startup.
2026-03-26T20:16:59.612522Z [info     ] voice_commerce_agent_starting  [src.voice_commerce.main] debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
2026-03-26T20:16:59.785678Z [info     ] Application shutting down      [src.voice_commerce.main]
INFO:     Application shutdown complete.
INFO:     Finished server process [22228]
INFO:     Started server process [21252]
INFO:     Waiting for application startup.
2026-03-26T20:17:02.448424Z [info     ] voice_commerce_agent_starting  [src.voice_commerce.main] debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
2026-03-26T20:17:02.533355Z [info     ] voice_handler_created          [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2
INFO:     127.0.0.1:2030 - "WebSocket /ws/voice?session_id=v5_mph3j0z2" [accepted]
2026-03-26T20:17:02.533355Z [info     ] ws_connected                   [voice_commerce.handlers.voice_websocket_handler] client="Address(host='127.0.0.1', port=2030)" session_id=v5_mph3j0z2
INFO:     connection open
2026-03-26T20:17:02.704288Z [info     ] gemini_session_connecting      [voice_commerce.core.voice.gemini_live_handler] model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-26T20:17:03.857472Z [info     ] gemini_session_connected       [voice_commerce.core.voice.gemini_live_handler]
2026-03-26T20:17:09.019225Z [info     ] input_mode_switched_to_audio   [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2
2026-03-26T20:17:24.094900Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'ai', 'text': 'Sure, I can help with that. What product ID should I remove from your cart?'}]
2026-03-26T20:17:27.494215Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' in my car now.'
2026-03-26T20:17:27.769044Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={} call_id=function-call-11513857942445210840 session=v5_mph3j0z2 tool=show_cart
2026-03-26T20:17:27.769044Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={} session=v5_mph3j0z2 tool_name=show_cart
2026-03-26T20:17:27.769044Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={} session=v5_mph3j0z2 tool=show_cart
2026-03-26T20:17:27.769044Z [info     ] show_cart                      [voice_commerce.core.tools.cart_tools] session=v5_mph3j0z2
2026-03-26T20:17:27.770649Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=65 tool=show_cart
2026-03-26T20:17:27.770649Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview='Your cart is empty. Would you like me to help you find something?' session=v5_mph3j0z2 tool=show_cart
2026-03-26T20:17:27.770649Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={} call_id=function-call-11513857942445210840 function_call=FunctionCall(
  args={},
  id='function-call-11513857942445210840',
  name='show_cart'
)
2026-03-26T20:17:31.922929Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'ai', 'text': 'Sure, I can help with that. What product ID should I remove from your cart?'}, {'role': 'user', 'text': ' in my car now.'}, {'role': 'ai', 'text': 'Your cart is empty. Would you like me to help you find something?'}]


In [ ]:
import asyncio
import os
import logging 
from unittest import result
from google import genai
from google.genai import types
# 1. Define a tool the model can call
log = logging.getLogger(__name__)
def get_current_time():
    """Returns the current system time."""
    from datetime import datetime
    return {"time": datetime.now().strftime("%H:%M:%S")}

# 2. Setup Client
# Ensure you use 'v1alpha' for preview features
client = genai.Client(
    api_key="YOUR_API_KEY", 
    http_options={'api_version': 'v1alpha'}
)

MODEL_ID = "gemini-2.5-flash-native-audio-preview-12-2025"

async def main():
    # 3. Configure the session for audio + text + tools
    config = types.LiveConnectConfig(
        tools=[get_current_time],
        response_modalities=["AUDIO"], # Returns native audio
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        # This returns the text transcript of the model's audio response
        output_audio_transcription=types.AudioTranscriptionConfig()
    )

    async with client.aio.live.connect(model=MODEL_ID, config=config) as session:
        # await session.send_tool_response()
        # Send text input or audio bytes (e.g., from a mic stream)
        # To send audio, replace the string with audio bytes:
        # await session.send(audio_bytes, end_of_turn=True)
        await session.send_client_content(turns= types.Content(role="user", parts=[types.Part(text="What time is it?")]), turn_complete=True)

        try:
            # FIXED: We open the Google stream EXACTLY ONCE and listen forever.
            # No 'while True:' needed, because 'async for' naturally waits for new data.
            async for response in session.receive():
                
                server_content = getattr(response, "server_content", None)
                if not server_content:
                    continue

                # 1. Handle Text, Audio, and Tools
                if server_content.model_turn and server_content.model_turn.parts:
                    for part in server_content.model_turn.parts:
                        
                        if getattr(part, "text", None):
                            log.debug("gemini_text_chunk", length=len(part.text))
                            yield {"type": "text", "text": part.text}

                        if getattr(part, "inline_data", None):
                            log.debug("gemini_audio_chunk", bytes=len(part.inline_data.data))
                            yield {"type": "audio", "data": part.inline_data.data}
                        
                        if getattr(part, "function_call", None):
                            log.info("gemini_tool_call_detected", tool_name=part.function_call.name)
                            yield {
                                "type": "tool_call",
                                "name": part.function_call.name,
                                "args": dict(part.function_call.args or {}),
                                "call_id": getattr(part.function_call, "id", None),
                            }

                # 2. Handle Transcripts (What you said, and what it heard)
                input_trans = getattr(server_content, "input_transcription", None)
                if input_trans and getattr(input_trans, "text", None):
                    log.debug("gemini_input_transcript", text=input_trans.text[:80])
                    yield {"type": "input_transcript", "text": input_trans.text}

                output_trans = getattr(server_content, "output_transcription", None)
                if output_trans and getattr(output_trans, "text", None):
                    log.debug("gemini_output_transcript", text=output_trans.text[:80])
                    yield {"type": "output_transcript", "text": output_trans.text}

                # 3. ── THE MAGIC FIX ──
                # Tell the UI the AI is done speaking so the mic unlocks, 
                # but DO NOT BREAK! The line stays open for the next text or voice input.
                if server_content.turn_complete:
                    yield {"type": "turn_complete"}

        except Exception as exc:
            error_str = str(exc)
            if "1000" in error_str or "1001" in error_str:
                log.info("gemini_session_closed_by_server", reason=error_str)
                yield {"type": "session_closed", "reason": error_str}
                return
            log.error("gemini_receive_error", error=error_str)
            yield {"type": "error", "message": error_str}
            return
        

                     

if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
import time
print(time.time())

In [ ]:
l = [1,2,3,4,5]
print(l[:2])

In [ ]:
from __future__ import annotations
from  collections.abc import AsyncGenerator
from typing import Any


import structlog
from google import genai
from google.genai import types

from voice_commerce.core.tools import tool_registry   # ← Phase 3: import the tool declarations
from voice_commerce.core.voice import audio_processor  # ← Phase 5: import audio config for browser
from voice_commerce.config.settings import settings


log = structlog.get_logger(__name__)


class GeminiLiveHandler:

    def __init__(self) -> None:
        self._client = genai.Client(api_key=settings.gemini_api_key) 
        self._session: Any = None
        self._session_ctx: Any = None   
        self._config = self._build_session_config()

        log.debug(
            "gemini_live_handler_created",
            model=settings.gemini_model,
            voice=settings.gemini_voice_name,
        )

    def _build_session_config(self) -> types.LiveConnectConfig:
        """Builds the configuration for the Gemini speech session."""
        

        return types.LiveConnectConfig(

            response_modalities=[types.Modality.AUDIO],
            system_instruction=types.Content(
                parts=[
                    types.Part(
                        text=self._build_system_prompt()
                    )
                ]
            ),
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name=settings.gemini_voice_name
                        # Available: "Aoede", "Charon", "Fenrir", "Kore", "Puck"
                        # Charon: clear, neutral — good for shopping assistant
                    )
                )
            ),
            input_audio_transcription=types.AudioTranscriptionConfig(),   
            output_audio_transcription=types.AudioTranscriptionConfig(),
            tools=tool_registry.get_all_tools(),   # ← Phase 3: the one new line

            )
            

    
    def _build_system_prompt(self) -> str:
        """The 'Rules' the AI must follow."""
        return (
            "You are a friendly Voice Shopping Assistant. "
            "ALWAYS respond in English only, regardless of the language the user speaks. "
            "Keep answers short and conversational — 1 to 3 sentences maximum. "
            "If you don't understand the user, ask one short clarifying question."
        )
    

    async def send_text(self, text: str) -> None:
        """Sends text to Gemini and yields response text as it arrives."""
        if self._session is None:
            raise RuntimeError(
                "Cannot send text: Gemini session is not connected. "
                "Did you use 'async with handler.connect():'?"
            )
        
        log.debug("gemini_sending_text", text=text)

        await self._session.send_client_content(
            turns=types.Content(
                role="user",
                parts=[types.Part(text=text)]
            ),
            turn_complete=True,
            # turn_complete=True is the key flag that tells Gemini:
            # "The user is done speaking, please respond now."
        )



    async def send_audio_chunk(self, pcm_bytes: bytes) -> None:
        """
        Stream mic audio from the browser to Gemini.
        Added in Phase 5 (microphone input). Kept here ready for that phase.
 
        Format required: PCM s16le at 16kHz mono.
        Uses send_realtime_input() not send_client_content() because audio is
        a continuous stream — Gemini's VAD detects speech pauses automatically.
        """
        log.debug("gemini_sending_audio_chunk", bytes=len(pcm_bytes))
        if self._session is None:
            raise RuntimeError("Cannot send audio: session not connected.")
        if not pcm_bytes:
            return  # skip empty chunks (can happen with some mic implementations)

        await self._session.send_realtime_input(
            audio=types.Blob(
                data=pcm_bytes,
                mime_type=f"audio/pcm;rate={audio_processor.MIC_SAMPLE_RATE}",
            )
        )


    async def send_tool_result(self, call_id:   str | None, tool_name: str,result:    str) -> None:

        if self._session is None:
            raise RuntimeError("Cannot send tool result: not connected.")
        log.debug("gemini_sending_tool_result",tool=tool_name, preview=result[:80])

        await self._session.send_tool_response(
            function_responses=[
                types.FunctionResponse(
                    id=call_id,  # correlate with the tool call
                    name=tool_name,
                    response={"result": result},
                )
            ],
        )

            # """
            # Send a tool's return value back to Gemini after executing it.  ← new Phase 3
 
            # FULL FLOW:
            # 1. User:  "show me running shoes"
            # 2. Gemini yields tool_call → name="search_products", args={"query":"running shoes"}
            # 3. Handler dispatches → search_products() runs → returns result string
            # 4. Handler calls this method → result delivered to Gemini
            # 5. Gemini reads result → speaks it naturally to the user
    
            # WHY send_tool_response() NOT send_client_content():
            # send_client_content() is for USER messages (text or audio the user sent).
            # send_tool_response() is for FUNCTION RESULTS.
            # The Gemini API routes them to completely different internal processors.
            # Using send_client_content() here would make Gemini treat your search
            # results as if the USER said them — broken conversation structure.
    
            # WHY result is a string (not JSON or a dict):
            # Gemini reads the result as text context, then paraphrases it into
            # natural speech. A structured readable string like:
            #     "• Nike Air Zoom — $129 — ID:1\n  Lightweight running shoe."
            # is ideal. Gemini turns this into:
            #     "I found the Nike Air Zoom for a hundred and twenty-nine dollars.
            #     It's a lightweight running shoe. Want to add it to your cart?"
    
            # ABOUT call_id:
            # Gemini assigns a unique ID to each tool call for correlation.
            # It can be None for some model versions — send_tool_response handles None.
            # """

    async def receive_events(self) -> AsyncGenerator[dict[str, Any], None]:
        if self._session is None:
            raise RuntimeError("Cannot receive: Gemini session is not connected.")
        while True:
            try:
                async for response  in self._session.receive():
                    # Gemini sends text in chunks (like a typewriter)
                    response: types.LiveServerMessage
                    if response.text is not None:
                        log.debug("gemini_text_chunk", length=len(response.text))
                        yield {"type": "text", "text": response.text}

                    if response.data is not None:
                        log.debug("gemini_audio_chunk", bytes=len(response.data))
                        yield {"type": "audio", "data": response.data}
                    
                    # ── Tool call ← Phase 3 ─────────────────────────────────
                    # When Gemini decides to call a function, it sends the call
                    # in model_turn.parts (not in response.text or response.data).
                    # We detect it here and yield it so the handler can dispatch it.
                    #
                    # WHY check model_turn.parts (not just response.text):
                    #   Function calls are a separate part type in the Gemini API.
                    #   They arrive as Part objects with a function_call field,
                    #   not as text. You cannot detect them from response.text.

                    if (
                        response.server_content
                        and response.server_content.model_turn
                        and response.server_content.model_turn.parts
                    ):
                        for part in response.server_content.model_turn.parts:
                            if hasattr(part, "function_call") and part.function_call:
                                log.info("gemini_tool_call_detected",tool_name=part.function_call.name)
                                yield {
                                    "type": "tool_call",
                                    "name": part.function_call.name,
                                    "args": dict(part.function_call.args or {}),
                                    "call_id": getattr(part.function_call, "id", None),
                                }

                    # INPUT TRANSCRIPT (Phase 5+)
                    # Text version of what the USER said (from their audio)
                    if (
                        response.server_content
                        and hasattr(response.server_content, "input_transcription")
                        and response.server_content.input_transcription
                    ):
                        transcript = response.server_content.input_transcription
                        if hasattr(transcript, "text") and transcript.text:
                            log.debug("gemini_input_transcript", text=transcript.text[:80])
                            yield {"type": "input_transcript", "text": transcript.text}
    
                    # OUTPUT TRANSCRIPT (Phase 4+)
                    # Text version of what GEMINI said (from its audio response)
                    if (
                        response.server_content
                        and hasattr(response.server_content, "output_transcription")
                        and response.server_content.output_transcription
                    ):
                        transcript = response.server_content.output_transcription
                        if hasattr(transcript, "text") and transcript.text:
                            log.debug("gemini_output_transcript", text=transcript.text[:80])
                            yield {"type": "output_transcript", "text": transcript.text}

                    # Check if Gemini is finished with its thought
                    if response.server_content and response.server_content.turn_complete:
                        yield {"type": "turn_complete"}
                        break  # exit inner loop → outer loop restarts immediately
            except Exception as exc:
                error_str = str(exc)
                if "1000" in error_str or error_str.strip() in ("1000 None.", "1001 None."):
                    log.info("gemini_session_closed_by_server", reason=error_str)
                    yield {"type": "session_closed", "reason": error_str}
                    return  # Stop — session is over, browser will auto-reconnect
                # Real error — log, notify browser, stop
                log.error("gemini_receive_error", error=error_str)
                yield {"type": "error", "message": error_str}
                return


    async def __aenter__(self) -> "GeminiLiveHandler":
        log.info("gemini_session_connecting", model=settings.gemini_model   )

        self._session_ctx = self._client.aio.live.connect(
            model=settings.gemini_model,
            config=self._config,
        )

        # Enter the context manager → opens the WebSocket to Gemini's servers
        self._session = await self._session_ctx.__aenter__()
        log.info("gemini_session_connected")
        return self
    

    async def __aexit__(self, exc_type: Any, exc_val: Any, exc_tb: Any) -> None:
        """ 
        Close the Gemini Live session.
        Called automatically when exiting the `async with` block.
 
        exc_type, exc_val, exc_tb: exception info if an exception occurred
        (None, None, None if the block exited normally)
        """
        log.info("gemini_session_closing")
 
        if self._session_ctx is not None:
            try:
                # Close the google-genai context manager (closes the WebSocket)
                await self._session_ctx.__aexit__(exc_type, exc_val, exc_tb)
            except Exception as e:
                # Don't let cleanup errors mask the original exception
                log.warning("gemini_session_close_error", error=str(e))
            finally:
                self._session = None                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          
                self._session_ctx = None
 
        log.info("gemini_session_closed")

2026-03-24 07:20:12 [info     ] browser_message_received       message={'type': 'websocket.receive', 'text': '{"type":"text","text":"Need summar teshirt"}'} message_type=websocket.receive session_id=v5_mph3j0z2

2026-03-24 07:18:12 [info     ] browser_message_received       message={'type': 'websocket.receive', 'bytes': b'\x9c\xff\x8d\xff\x81\xffi\xffX\xff<\xff\x19\xff\x02\xff\xf0\xfe\xe3\xfe\xe3\xfe\xf0\xfe\x13\xff=\xff`\xff\x9a\xff\xcd\xff\x08\x00E\x00p\x00\x99\x00\xb5\x00\xc8\x00\xd7\x00\xef\x00\x12\x015\x01c\x01\x91\x01\xb1\x01\xc4\x01\xc4\x01\xac\x01\x7f\x01K\x01\x05\x01\xca\x00\x9a\x00n\x00P\x00-\x00\x0f\x00\xfd\xff\xf0\xff\xe6\xff\xcf\xff\xa7\xffm\xff,\xff\xe6\xfe\xb9\xfe\xa4\xfe\xa8\xfe\xbe\xfe\xca\xfe\xdc\xfe\xf7\xfe\x11\xff/\xffS\xff{\xff\xa1\xff\xc7\xff\xea\xff\x11\x00D\x00s\x00\xa0\x00\xd4\x00\x06\x012\x01J\x01L\x01@\x01/\x01\x0b\x01\xe7\x00\xcb\x00\xa7\x00\x8f\x00w\x00g\x00Z\x00E\x00.\x00\x02\x00\xd5\xff\xa7\xff\x88\xffm\xffS\xffC\xff-\xff\x10\xff\xf4\xfe\xe9\xfe\xe2\xfe\xe1\xfe\xde\xfe\xd8\xfe\xdc\xfe\xed\xfe\x13\xff9\xffa\xff\x90\xff\xb8\xff\xdc\xff\x07\x007\x00t\x00\xb0\x00\xdd\x00\x15\x01E\x01X\x01k\x01q\x01^\x01J\x01C\x011\x01#\x01\x1e\x01\x0c\x01\xeb\x00\xc4\x00\x9c\x00u\x00M\x00:\x00\x18\x00\xf1\xff\xd0\xff\x8e\xffT\xff"\xff\xdf\xfe\xab\xfe\x83\xfeY\xfe-\xfe\r\xfe\x05\xfe\x03\xfe\x1d\xfe0\xfe>\xfeZ\xfeq\xfe\x91\xfe\xb7\xfe\xde\xfe\x1b\xffo\xff\xc3\xff\x1f\x00u\x00\xbe\x00\xf4\x00\x15\x013\x01M\x01c\x01z\x01\x84\x01\x84\x01\x9e\x01\xae\x01\xb2\x01\xb1\x01\x92\x01f\x018\x01\xf7\x00\xb1\x00v\x00*\x00\xde\xff\xa2\xffn\xffM\xffC\xff.\xff\x11\xff\xf1\xfe\xd9\xfe\xc2\xfe\xa8\xfe\x89\xfek\xfeN\xfe7\xfe&\xfe2\xfeW\xfe\x80\xfe\xad\xfe\xe1\xfe\xfe\xfe\x0e\xff\x1e\xff%\xff>\xffp\xff\xb6\xff\n\x00q\x00\xd5\x00+\x01o\x01\x92\x01\x9b\x01\x92\x01{\x01M\x011\x01\x14\x01\n\x01\x13\x01+\x01K\x01W\x01M\x01+\x01\xf5\x00\xb0\x00k\x00>\x00\x17\x00\xed\xff\xd4\xff\xbe\xff\x91\xffb\xff/\xff\xe9\xfe\xab\xfe\x84\xfel\xfec\xfeo\xfe\x86\xfe\xa3\xfe\xd3\xfe\x10\xffK\xff\x93\xff\xce\xff\xf8\xff\x19\x00/\x00C\x00W\x00v\x00\x88\x00\xa6\x00\xd2\x00\xe5\x00\xfd\x00\x16\x01*\x01C\x01C\x01;\x01+\x01\x0b\x01\xe6\x00\xc8\x00\xb1\x00\x9c\x00\x8c\x00\x7f\x00o\x00X\x00G\x00\x14\x00\xd7\xff\xa3\xffj\xff?\xff$\xff\x13\xff\x03\xff\x04\xff\t\xff\x11\xff(\xff8\xff2\xff7\xff;\xffQ\xfft\xff\x9c\xff\xc3\xff\xe7\xff\x16\x00E\x00k\x00\x95\x00\xb7\x00\xc5\x00\xc3\x00\xba\x00\xaf\x00\xaf\x00\xb5\x00\xc5\x00\xe2\x00\xfa\x00\x08\x01\x13\x01\x12\x01\xfa\x00\xe6\x00\xc7\x00\x98\x00{\x00Y\x001\x00\x17\x00\xf5\xff\xe6\xff\xd7\xff\xb6\xff\x96\xfff\xff/\xff\xfa\xfe\xd3\xfe\xbc\xfe\xba\xfe\xd9\xfe\x04\xff5\xff}\xff\xc1\xff\xeb\xff\xf8\xff\xf9\xff\xec\xff\xdb\xff\xda\xff\xe3\xff\xf0\xff\x0e\x00;\x00]\x00\x8b\x00\xad\x00\xc3\x00\xd0\x00\xc3\x00\xb1\x00\x97\x00\x82\x00\x80\x00\x83\x00\x93\x00\xaa\x00\xb8\x00\xb9\x00\xb2\x00\x9a\x00w\x00E\x00\x12\x00\xd2\xff\x8e\xff^\xff6\xff#\xff!\xff+\xff1\xff$\xff\x13\xff\xf8\xfe\xde\xfe\xcb\xfe\xd2\xfe\xed\xfe\xfe\xfe\x1f\xffJ\xffx\xff\xb2\xff\xfb\xff)\x00P\x00q\x00x\x00o\x00s\x00\x8b\x00\x91\x00\xa4\x00\xbc\x00\xcf\x00\xe9\x00\xff\x00\x0c\x01\x06\x01\xf9\x00\xe1\x00\xb2\x00\x8d\x00g\x00=\x00\x19\x00\xf8\xff\xe3\xff\xbd\xff\x9f\xff}\xffH\xff\x10\xff\xe6\xfe\xc7\xfe\xa5\xfe\x91\xfe\x8e\xfe\x8c\xfe\x94\xfe\xad\xfe\xce\xfe\xf2\xfe\x19\xff9\xffR\xffg\xff}\xff\x99\xff\xb9\xff\xe6\xff\x10\x008\x00_\x00\x89\x00\xa9\x00\xc0\x00\xd6\x00\xe3\x00\xe9\x00\xe7\x00\xe8\x00\xef\x00\xef\x00\xed\x00\xec\x00\xde\x00\xd3\x00\xc7\x00\xa8\x00\x8d\x00h\x006\x00\x02\x00\xd7\xff\xb9\xff\x9c\xff\x86\xff\x83\xffp\xff_\xffQ\xff<\xff!\xff\x04\xff\xe6\xfe\xc9\xfe\xbb\xfe\xc4\xfe\xd0\xfe\xfa\xfe+\xffS\xffs\xff\x9e\xff\xcd\xff\xe5\xff\xfb\xff\x13\x00+\x00L\x00w\x00\xad\x00\xea\x00\x1d\x01L\x01n\x01z\x01\x83\x01\x86\x01s\x01]\x01G\x015\x012\x01*\x01\x19\x01\x03\x01\xe4\x00\xb6\x00\x85\x00A\x00\x05\x00\xc6\xff\x80\xffV\xff4\xff\x1a\xff\x15\xff\n\xff\xf5\xfe\xe1\xfe\xcc\xfe\xbf\xfe\xb5\xfe\xb1\xfe\xb6\xfe\xc2\xfe\xd6\xfe\xf3\xfe\x16\xff?\xffb\xff\x8f\xff\xc2\xff\xed\xff\x1f\x00N\x00o\x00\x8d\x00\xb8\x00\xea\x00(\x01\\\x01z\x01\x85\x01\x7f\x01l\x01\\\x01W\x01P\x01C\x011\x01\x1e\x01\x06\x01\xe8\x00\xc2\x00\x98\x00a\x00 \x00\xea\xff\xb3\xff\x84\xffd\xffD\xff*\xff\x19\xff\x03\xff\xf1\xfe\xd5\xfe\xb6\xfe\xae\xfe\xb2\xfe\xb7\xfe\xcd\xfe\xdf\xfe\xff\xfe#\xffO\xff\x80\xff\xa9\xff\xd5\xff\xf2\xff\x07\x00\x19\x00?\x00i\x00\x89\x00\xb7\x00\xda\x00\xf4\x00\r\x01%\x011\x01;\x01H\x01B\x01-\x01\x11\x01\xf6\x00\xd3\x00\xb8\x00\xa4\x00\x84\x00_\x00;\x00\t\x00\xd1\xff\xab\xff\x84\xffP\xff\x1a\xff\xda\xfe\xb0\xfe\x8d\xfez\xfe}\xfe\x7f\xfe\x83\xfe\x83\xfez\xfe~\xfe\x8b\xfe\xa0\xfe\xcc\xfe\xfc\xfe&\xff\\\xff\x8a\xff\xc7\xff\xfa\xff,\x00b\x00\x91\x00\xc4\x00\xe0\x00\xff\x00\x19\x01 \x01\x1f\x01\x1d\x01%\x01\'\x01,\x01\x1e\x01\x11\x01\x03\x01\xd8\x00\xa9\x00e\x000\x00\x01\x00\xd2\xff\xaa\xffu\xffL\xff\x1e\xff\x05\xff\xeb\xfe\xd8\xfe\xd3\xfe\xb9\xfe\xa5\xfe\x91\xfe\x92\xfe\xb6\xfe\xfc\xfeU\xff\xaf\xff\xf6\xff!\x005\x004\x00(\x00 \x00\r\x00\x02\x00\x0b\x00$\x00L\x00\x86\x00\xc6\x00\xf9\x00\x1c\x013\x01=\x01>\x01;\x01-\x01\x1b\x01\n\x01\xeb\x00\xd0\x00\xbe\x00\xa8\x00\x89\x00d\x00&\x00\xdb\xff\x98\xff_\xff7\xff\x1f\xff \xff+\xff4\xff>\xffB\xffB\xffA\xff>\xff1\xff1\xff5\xff<\xffV\xff\x85\xff\xc0\xff\x02\x00I\x00\x84\x00\xbf\x00\xde\x00\xed\x00\xf0\x00\xeb\x00\xeb\x00\xe7\x00\xe1\x00\xd2\x00\xc3\x00\xb8\x00\xad\x00\x9f\x00\x86\x00m\x00T\x00C\x005\x00,\x00(\x00.\x005\x000\x00\x1d\x00\xff\xff\xda\xff\xa3\xffY\xff\x1a\xff\xcf\xfe\x97\xfeu\xfeb\xfee\xfe{\xfe\xa7\xfe\xde\xfe\x1f\xffZ\xff\x97\xff\xc9\xff\xf8\xff"\x00J\x00\x7f\x00\xa8\x00\xc0\x00\xda\x00\xe9\x00\xf1\x00\x01\x01\x07\x01\x08\x01\x08\x01\xf8\x00\xe6\x00\xc7\x00\xac\x00\x9b\x00\x90\x00\x91\x00\x9f\x00\xb5\x00\xbb\x00\xbf\x00\xad\x00{\x00D\x00\x02\x00\xc0\xff\x80\xffL\xff\x1b\xff\xf6\xfe\xd5\xfe\xb4\xfe\x98\xfe\x81\xfey\xfet\xfep\xfei\xfeb\xfeq\xfe\x8b\xfe\xc4\xfe\x08\xffM\xff\x9d\xff\xc9\xff\xe3\xff\xf2\xff\xe8\xff\xf0\xff\xf4\xff\x01\x00(\x00U\x00\x81\x00\xb5\x00\xe3\x00\x0e\x01$\x01)\x01\'\x01\x19\x01\x08\x01\xfa\x00\xed\x00\xe8\x00\xef\x00\x02\x01\x13\x01\x1e\x01\x16\x01\xf4\x00\xb8\x00e\x00\x12\x00\xb4\xffg\xff2\xff\x0b\xff\x00\xff\xf8\xfe\xec\xfe\xe2\xfe\xe8\xfe\xe1\xfe\xd7\xfe\xdd\xfe\xe1\xfe\xf6\xfe\xfd\xfe\x0b\xff)\xffW\xff\x88\xff\xc3\xff\x00\x00\'\x00A\x00T\x00f\x00\x85\x00\xa9\x00\xdc\x00\x13\x01O\x01\x84\x01\xae\x01\xc4\x01\xbe\x01\xb0\x01\x8d\x01d\x014\x01\xf6\x00\xb9\x00\x83\x00W\x00\'\x00\x02\x00\xe3\xff\xcc\xff\xb6\xff\x95\xffi\xff0\xff\x04\xff\xe2\xfe\xca\xfe\xb7\xfe\xbf\xfe\xd0\xfe\xda\xfe\xe4\xfe\xf2\xfe\xf7\xfe\x03\xff\r\xff\x19\xff4\xffW\xff\x81\xff\xa8\xff\xd8\xff\x02\x00:\x00y\x00\xad\x00\xe1\x00\x03\x01\x07\x01\xf1\x00\xdc\x00\xd0\x00\xdb\x00\xf8\x00\x0f\x01\x1e\x01\x17\x01\x00\x01\xe0\x00\xba\x00\x9b\x00\x80\x00U\x00\x1b\x00\xde\xff\xa4\xff\x85\xffv\xffr\xff\x81\xff\x82\xff|\xffm\xff]\xffK\xff)\xff\x01\xff\xec\xfe\xd2\xfe\xc8\xfe\xdb\xfe\xe6\xfe\x04\xff&\xff2\xffC\xffW\xffg\xff\x89\xff\xaf\xff\xda\xff\x01\x00#\x00J\x00h\x00\x8e\x00\xbb\x00\xdc\x00\xf5\x00\x04\x01\x0c\x01\x0e\x01\t\x01\x07\x01\x0f\x01\x1c\x01\x1c\x01\t\x01\xe0\x00\xb1\x00~\x00E\x00\x15\x00\xe6\xff\xc4\xff\xb4\xff\xb6\xff\xae\xff\xbe\xff\xd2\xff\xc8\xff\xca\xff\xbd\xff\xaf\xff\xa8\xff\x96\xff\x7f\xff^\xffD\xff6\xff5\xffH\xff\x81\xff\xad\xff\xcd\xff\xe1\xff\xe3\xff\xe9\xff\xfa\xff\x12\x008\x00b\x00\x8d\x00\x9c\x00\xa5\x00\xb8\x00\xba\x00\xb8\x00\xac\x00\x93\x00t\x00c\x00M\x00=\x007\x00+\x00+\x00\x1e\x00\x0f\x00\xfa\xff\xe0\xff\xbf\xff\x92\xff}\xffq\xffd\xffU\xffW\xff[\xff]\xffq\xff\x84\xff\x92\xff\xa4\xff\xaa\xff\xab\xff\xac\xff\xbe\xff\xd6\xff\xf2\xff\t\x00*\x00D\x00S\x00k\x00\x7f\x00\x93\x00\x99\x00\x9a\x00\x92\x00\x93\x00\x97\x00\x94\x00\x9f\x00\xaa\x00\xb0\x00\xb4\x00\xb7\x00\xab\x00\x9f\x00\x80\x00H\x00\x0c\x00\xcf\xff\x8d\xffa\xff?\xff!\xff\x04\xff\xe2\xfe\xc4\xfe\xa7\xfe\x98\xfe\x94\xfe\x9a\xfe\xac\xfe\xc9\xfe\xec\xfe\xfe\xfe\x1e\xff1\xffI\xffo\xff\x91\xff\x9e\xff\xaf\xff\xc0\xff\xc0\xff\xcf\xff\xef\xff&\x00f\x00\xa8\x00\xf0\x001\x01n\x01\x97\x01\xb5\x01\xd4\x01\xda\x01\xdc\x01\xcc\x01\xb0\x01\x8d\x01Z\x01\x1c\x01\xe1\x00\xaf\x00w\x00G\x00\x19\x00\xe9\xff\xb9\xff\x8e\xffd\xffQ\xff?\xff8\xff<\xff0\xff&\xff\x18\xff\xfe\xfe\xf2\xfe\xe9\xfe\xe3\xfe\xe8\xfe\xf1\xfe\x00\xff\x1f\xffE\xffv\xff\xae\xff\xed\xff3\x00m\x00\x99\x00\xc7\x00\xf2\x00\x11\x01*\x01>\x01H\x01M\x01R\x01S\x01F\x014\x01\x1e\x01\xfe\x00\xe0\x00\xba\x00\x93\x00s\x00]\x00I\x003\x00\x16\x00\xf2\xff\xd1\xff\xb0\xff\x98\xffv\xffG\xff\x18\xff\xee\xfe\xc7\xfe\xa3\xfe\x94\xfe\x97\xfe\x9d\xfe\x9d\xfe\xa5\xfe\x9d\xfe\xa2\xfe\xb7\xfe\xd1\xfe\x02\xff3\xfff\xff\x95\xff\xce\xff\t\x00B\x00z\x00\xa9\x00\xd3\x00\xf3\x00\x14\x014\x01K\x01\\\x01b\x01s\x01p\x01i\x01^\x01B\x01"\x01\x02\x01\xe2\x00\xb6\x00\x84\x00_\x003\x00\n\x00\xe8\xff\xc1\xff\x8c\xffN\xff\x1d\xff\xf0\xfe\xd2\xfe\xbb\xfe\xa5\xfe\x93\xfe\x86\xfex\xfen\xfen\xfey\xfe\x9e\xfe\xc0\xfe\xe4\xfe\x11\xff1\xff`\xff\x99\xff\xd4\xff\x07\x00D\x00x\x00\x9f\x00\xcb\x00\xfe\x00,\x01P\x01o\x01x\x01x\x01|\x01p\x01b\x01\\\x01J\x01#\x01\xf2\x00\xba\x00\x86\x00X\x00)\x00\x00\x00\xd2\xff\xa7\xff\x81\xff^\xffK\xff0\xff\n\xff\xf7\xfe\xdf\xfe\xc2\xfe\xb3\xfe\x9f\xfe\x90\xfe\x93\xfe\x9f\xfe\xb0\xfe\xc5\xfe\xd9\xfe\xf0\xfe\x14\xffC\xffl\xff\x9b\xff\xca\xff\xf1\xff\x14\x00=\x00m\x00\x98\x00\xc0\x00\xda\x00\xe8\x00\xec\x00\xe3\x00\xd3\x00\xb5\x00\xa4\x00\x9a\x00\x8b\x00|\x00j\x00Z\x00@\x00,\x00\t\x00\xea\xff\xd6\xff\xb9\xff\x90\xffs\xffe\xffR\xffE\xff4\xff%\xff\x1c\xff\x16\xff\n\xff\xff\xfe\x00\xff\xfd\xfe\x06\xff\x18\xff,\xffM\xffo\xff\x91\xff\xb6\xff\xd4\xff\xfd\xff\'\x00L\x00u\x00\xa0\x00\xb7\x00\xd5\x00\xf2\x00\x01\x01\x13\x01&\x018\x016\x01A\x01F\x01=\x01/\x01\x15\x01\xf7\x00\xd3\x00\xac\x00\x86\x00d\x009\x00\x0c\x00\xe5\xff\xc4\xff\xa5\xff\x7f\xff_\xff>\xff\x1a\xff\x05\xff\xf3\xfe\xe9\xfe\xf5\xfe\x0f\xff-\xffR\xff\x7f\xff\xa4\xff\xc0\xff\xde\xff\x01\x00!\x00F\x00l\x00\x93\x00\xc2\x00\xea\x00\x13\x017\x01O\x01Y\x01X\x01L\x01@\x01,\x01\x16\x01\xfe\x00\xef\x00\xec\x00\xde\x00\xcd\x00\xb3\x00\x88\x00W\x00$\x00\xf9\xff\xd9\xff\xb8\xff\xa2\xff\x8b\xffi\xffP\xff8\xff,\xff&\xff\x1f\xff\x1a\xff\x0f\xff\x01\xff\xfe\xfe\x10\xff%\xffI\xffk\xff\x85\xff\xa9\xff\xc4\xff\xdd\xff\xf5\xff\x15\x00<\x00\\\x00\x81\x00\xa9\x00\xd0\x00\xf6\x00\x16\x013\x01F\x01O\x01P\x01N\x01H\x016\x01)\x01\x14\x01\xf5\x00\xd2\x00\xa9\x00w\x00Q\x001\x00\x08\x00\xdc\xff\xa8\xffp\xff<\xff\x0c\xff\xdf\xfe\xba\xfe\xa3\xfe\x8b\xfev\xfer\xfep\xfe{\xfe\x9f\xfe\xba\xfe\xcd\xfe\xe5\xfe\xf7\xfe\x0e\xff:\xffr\xff\xb2\xff\xfb\xff7\x00n\x00\x99\x00\xc5\x00\xe6\x00\xfa\x00\x0e\x01\x18\x01\x1f\x01\'\x012\x01C\x01\\\x01m\x01f\x01G\x01\x1a\x01\xda\x00\x9c\x00d\x001\x00\x06\x00\xe4\xff\xbe\xff\x92\xfff\xff;\xff\x16\xff\xf1\xfe\xda\xfe\xb3\xfe\x97\xfe~\xfee\xfeZ\xfeY\xfer\xfe\x80\xfe\xa4\xfe\xd1\xfe\xfb\xfe-\xffO\xffo\xff\x84\xff\xb1\xff\xd7\xff\xfb\xff9\x00j\x00\x9c\x00\xd3\x00\xfc\x00\x1e\x01B\x01N\x01N\x01K\x01:\x01\x1d\x01\x03\x01\xef\x00\xe0\x00\xda\x00\xde\x00\xc8\x00\x9c\x00f\x00,\x00\x00\x00\xe6\xff\xe0\xff\xdb\xff\xd0\xff\xc9\xff\xad\xff\x88\xffl\xffD\xff\x11\xff\xd5\xfe\x9c\xfer\xfeU\xfeU\xfe`\xfe\x81\xfe\xb2\xfe\xd9\xfe\x00\xff\'\xffZ\xff\x84\xff\xa9\xff\xcc\xff\xe4\xff\x03\x00\'\x00S\x00\x83\x00\xbe\x00\xe3\x00\xf7\x00\x10\x01\x0e\x01\x06\x01\x08\x01\x00\x01\x01\x01\x04\x01\x03\x01\xfd\x00\xe9\x00\xd1\x00\xba\x00\x9d\x00\x80\x00Y\x00,\x00\r\x00\xe2\xff\xc2\xff\xab\xff\x9c\xff\x84\xffr\xffh\xffO\xff=\xff(\xff\t\xff\xf7\xfe\xf3\xfe\xe8\xfe\xe2\xfe\xe5\xfe\xf2\xfe\n\xff1\xffj\xff\xa5\xff\xe1\xff\x1d\x00Q\x00\x8c\x00\xc7\x00\xf5\x00"\x01A\x01O\x01V\x01R\x01I\x01C\x01#\x01\x0c\x01\xf4\x00\xc8\x00\xab\x00\x8b\x00\x83\x00\x7f\x00}\x00\x90\x00\x94\x00\x9b\x00\xa8\x00\xa6\x00\x9f\x00\x93\x00u\x00R\x00\'\x00\x00\x00\xd3\xff\xac\xff\x8e\xff_\xffB\xff4\xff*\xff\x1e\xff\x0f\xff\r\xff\x14\xff%\xffM\xff\x85\xff\xc5\xff\x16\x00W\x00\x8f\x00\xba\x00\xd4\x00\xe3\x00\xdc\x00\xd9\x00\xbf\x00\xa7\x00\x96\x00\x88\x00\x82\x00|\x00w\x00o\x00g\x00\\\x00U\x00:\x00%\x00\x0c\x00\xf3\xff\xf3\xff\xe9\xff\xde\xff\xc3\xff\x96\xffj\xff)\xff\xe5\xfe\xb0\xfe}\xfe\\\xfeM\xfeT\xfe\x7f\xfe\xb0\xfe\xf2\xfe.\xff]\xff\x9c\xff\xc4\xff\xef\xff\x1c\x00E\x00w\x00\xb4\x00\xf2\x00*\x01V\x01z\x01\x89\x01|\x01m\x01K\x01\x1a\x01\xf4\x00\xc9\x00\xb2\x00\x9e\x00\x80\x00s\x00W\x00B\x00+\x00\x10\x00\xf9\xff\xda\xff\xb5\xff\x8b\xffc\xff6\xff\x1b\xff\t\xff\xfc\xfe\xf3\xfe\xdc\xfe\xd1\xfe\xc6\xfe\xb6\xfe\xac\xfe\xb1\xfe\xc4\xfe\xdb\xfe\xff\xfe/\xffp\xff\xab\xff\xe8\xff\x1e\x00Q\x00|\x00\x93\x00\xb1\x00\xc5\x00\xdc\x00\xe9\x00\xf4\x00\t\x01\x15\x01\x1e\x01\'\x010\x01)\x01\x12\x01\xf7\x00\xd4\x00\x9f\x00p\x00=\x00\x11\x00\xeb\xff\xc0\xff\xae\xff\x8a\xffp\xffW\xff:\xff\'\xff\x13\xff\x08\xff\xf6\xfe\xdc\xfe\xc5\xfe\xbb\xfe\xb3\xfe\xc2\xfe\xe8\xfe\x02\xff\x16\xff(\xff3\xffB\xffg\xff\x94\xff\xcc\xff\x04\x000\x00b\x00\x8d\x00\xc1\x00\xf7\x00\x1d\x018\x01?\x01?\x01;\x01-\x014\x018\x012\x018\x01=\x019\x01\x19\x01\xf5\x00\xc2\x00z\x008\x00\xf9\xff\xb8\xff\x80\xffR\xff*\xff\x0b\xff\xf0\xfe\xc6\xfe\x92\xfec\xfe?\xfe&\xfe\x1f\xfe!\xfe\x1e\xfe9\xfeY\xfe\x89\xfe\xcc\xfe\x12\xffa\xff\x9b\xff\xcd\xff\xf6\xff\x1a\x00H\x00~\x00\xad\x00\xe0\x00\x10\x014\x01U\x01l\x01\x83\x01\x98\x01\xa0\x01\x9c\x01\x97\x01\x99\x01\x8e\x01\x80\x01l\x01I\x01.\x01\x04\x01\xd3\x00\x96\x00Z\x00\x08\x00\xb1\xffd\xff \xff\x00\xff\xf0\xfe\xed\xfe\xe3\xfe\xd4\xfe\xb5\xfe\x97\xfe\x83\xfe\x81\xfe\x8f\xfe\xa0\xfe\xbe\xfe\xd3\xfe\xe5\xfe\x01\xff*\xffP\xff|\xff\xa7\xff\xbf\xff\xe4\xff\x03\x00&\x00X\x00|\x00\xa6\x00\xd3\x00\xf8\x00\x1d\x01>\x01N\x01d\x01\x87\x01\x9e\x01\xb2\x01\xa4\x01\x90\x01r\x01J\x01+\x01\xff\x00\xd9\x00\xb2\x00\x8a\x00\\\x00)\x00\xfb\xff\xbc\xff\x7f\xffB\xff\x08\xff\xdc\xfe\xc1\xfe\xb4\xfe\xb1\xfe\xbb\xfe\xc3\xfe\xd3\xfe\xef\xfe\r\xff\x1f\xffA\xffi\xff\x91\xff\xcb\xff\x0b\x00F\x00\x7f\x00\xb8\x00\xe0\x00\xf7\x00\x0b\x01\xfc\x00\xdc\x00\xbf\x00\xb0\x00\xa5\x00\x93\x00\x92\x00\x8e\x00~\x00n\x00k\x00m\x00v\x00u\x00f\x00P\x007\x00\x16\x00\xf4\xff\xd9\xff\xb1\xff\x8d\xff_\xff2\xff\x08\xff\xdc\xfe\xc5\xfe\xbe\xfe\xc9\xfe\xec\xfe\x15\xff7\xffZ\xff{\xff\xae\xff\xea\xff\x0f\x009\x00J\x00T\x00W\x00P\x00[\x00S\x00R\x00h\x00\x83\x00\xa6\x00\xc9\x00\xe5\x00\xed\x00\xf0\x00\xf1\x00\xf3\x00\xfd\x00\xff\x00\xed\x00\xd3\x00\xac\x00y\x00]\x004\x00\xfd\xff\xc5\xff\x88\xffM\xff\x1e\xff\x08\xff\xf3\xfe\xe0\xfe\xd7\xfe\xda\xfe\xd6\xfe\xd9\xfe\xe6\xfe\xee\xfe\xf3\xfe\xef\xfe\xf5\xfe\x04\xff\x0f\xff(\xffF\xffl\xff\x95\xff\xc4\xff\xf2\xff\x11\x004\x00Q\x00r\x00\x93\x00\xbd\x00\xd2\x00\xdf\x00\xee\x00\xff\x00\x10\x01\x1f\x01.\x01\x18\x01\xff\x00\xd8\x00\xa8\x00\x81\x00^\x00>\x00\x15\x00\xf9\xff\xdc\xff\xb3\xff\x8e\xff[\xff+\xff\xfd\xfe\xda\xfe\xbc\xfe\xaa\xfe\x9e\xfe\x99\xfe\xa3\xfe\xb3\xfe\xda\xfe\xf3\xfe\x16\xff9\xffL\xff]\xffh\xff\x7f\xff\x9d\xff\xbf\xff\xd9\xff\x01\x00-\x00Z\x00\x8d\x00\xc1\x00\xed\x00\x06\x01+\x018\x019\x01R\x01m\x01v\x01o\x01a\x01?\x01!\x01\xfd\x00\xd3\x00\xa7\x00c\x00\x1a\x00\xb6\xffl\xff:\xff\r\xff\xf7\xfe\xe3\xfe\xc5\xfe\xa4\xfe\x90\xfe\x7f\xfe\x8a\xfe\x94\xfe\x9d\xfe\xa6\xfe\xb8\xfe\xee\xfe\'\xffc\xff\x91\xff\xb6\xff\xd5\xff\xe1\xff\x05\x00\x1c\x000\x00=\x00X\x00x\x00\x8d\x00\xbe\x00\xee\x00\x1c\x013\x01J\x01`\x01q\x01o\x01m\x01t\x01^\x01W\x01N\x01<\x01\x17\x01\xe8\x00\xb3\x00y\x00L\x00\x14\x00\xda\xff\x9d\xff]\xff-\xff\t\xff\xf0\xfe\xd5\xfe\xcb\xfe\xbc\xfe\xaf\xfe\xaf\xfe\xab\xfe\xb5\xfe\xc8\xfe\xe1\xfe\xf9\xfe\x1c\xffE\xffq\xff\x98\xff\xb0\xff\xde\xff\x0b\x00=\x00_\x00}\x00\x9e\x00\xb3\x00\xe3\x00\x1d\x01H\x01k\x01\x89\x01\x91\x01\x8c\x01\x94\x01\x8c\x01y\x01_\x01?\x01\x17\x01\xee\x00\xc8\x00\x9b\x00p\x003\x00\xfb\xff\xc0\xff\x8d\xff\\\xff!\xff\xf9\xfe\xd8\xfe\xc2\xfe\xad\xfe\xaf\xfe\xad\xfe\xb9\xfe\xd2\xfe\xda\xfe\xe7\xfe\xe7\xfe\xf1\xfe\x02\xff\x11\xff8\xffb\xff\x8a\xff\xb0\xff\xe3\xff\x14\x009\x00]\x00z\x00\x95\x00\xad\x00\xcf\x00\xf0\x00\t\x01\x1f\x01/\x01@\x01?\x01<\x01.\x01\r\x01\xe0\x00\xb1\x00\x84\x00Q\x001\x00\x19\x00\x01\x00\xe7\xff\xd0\xff\xb3\xff\xac\xff\x8e\xfft\xff[\xff2\xff\x13\xff\xff\xfe\xfa\xfe\xfb\xfe\xfd\xfe\x19\xff2\xff3\xffD\xffO\xffW\xffp\xff\x91\xff\xbb\xff\xe7\xff\x1c\x00F\x00b\x00|\x00\x95\x00\xba\x00\xd1\x00\xe0\x00\xe9\x00\xe0\x00\xd3\x00\xd0\x00\xd6\x00\xce\x00\xd0\x00\xce\x00\xc0\x00\xb1\x00\xa2\x00\x8d\x00g\x00:\x00\x0f\x00\xed\xff\xda\xff\xc2\xff\xa8\xff\x81\xffM\xff\'\xff\t\xff\xfe\xfe\xf2\xfe\xf7\xfe\xee\xfe\xd6\xfe\xdd\xfe\xe0\xfe\xf2\xfe\x0e\xff\'\xffE\xffi\xff\x94\xff\xc8\xff\xf7\xff\x19\x00B\x00o\x00\x95\x00\xc0\x00\xdd\x00\xfc\x00\x12\x01%\x019\x01<\x018\x013\x01\x14\x01\xf5\x00\xde\x00\xbd\x00\x99\x00m\x00?\x00\x12\x00\xf3\xff\xdb\xff\xbd\xff\xa5\xffx\xffT\xff9\xff\x1c\xff\x10\xff\r\xff\x0b\xff\xff\xfe\x04\xff\x0c\xff\x13\xff$\xff<\xffN\xff`\xff}\xff\x93\xff\xb7\xff\xd9\xff\xf8\xff"\x00@\x00]\x00u\x00\x8f\x00\xa4\x00\xbc\x00\xe0\x00\xed\x00\x07\x01\x10\x01\xf9\x00\xe4\x00\xde\x00\xd2\x00\xc4\x00\xc5\x00\xba\x00\xaa\x00\x83\x00e\x00>\x00\x11\x00\xf4\xff\xc4\xff\xa3\xff}\xff`\xff=\xff\x1a\xff\x00\xff\xd0\xfe\xb5\xfe\x9e\xfe\x99\xfe\x9e\xfe\xa0\xfe\xbf\xfe\xcf\xfe\xf4\xfe"\xffG\xffm\xff\x8c\xff\xa8\xff\xbe\xff\xdb\xff\xf5\xff\x15\x00=\x00i\x00\xa1\x00\xd5\x00\x01\x01!\x017\x01[\x01u\x01\xa8\x01\xdb\x01\xe7\x01\xd4\x01\xb0\x01z\x01G\x01\x0c\x01\xc8\x00~\x00(\x00\xda\xff\x99\xffk\xffE\xff\'\xff\x04\xff\xe0\xfe\xca\xfe\xbf\xfe\xca\xfe\xcb\xfe\xd0\xfe\xcb\xfe\xc2\xfe\xbf\xfe\xc5\xfe\xdf\xfe\xec\xfe\xfe\xfe\x16\xff*\xffK\xffx\xff\xae\xff\xe5\xff\x1f\x00e\x00\x99\x00\xcb\x00\xf2\x00\x1c\x01B\x01`\x01{\x01\x95\x01\xa8\x01\xb1\x01\xab\x01\xab\x01\xaa\x01\xa9\x01\xa2\x01\x94\x01\x81\x01H\x01\x10\x01\xce\x00\x84\x00-\x00\xdf\xff\x8b\xff,\xff\xdf\xfe\x89\xfeA\xfe\x0f\xfe\xec\xfd\xdc\xfd\xed\xfd\x0f\xfe,\xfeY\xfe\x8c\xfe\xc9\xfe\x16\xffX\xff\x94\xff\xad\xff\xc8\xff\xdd\xff\xe3\xff\xf7\xff\x0c\x00,\x00L\x00r\x00\xa6\x00\xdb\x00\x10\x019\x01]\x01w\x01\x99\x01\xc7\x01\xe1\x01\xea\x01\xed\x01\xe4\x01\xc8\x01\x9c\x01j\x01)\x01\xe1\x00\x92\x00C\x00\xf6\xff\xb0\xff\x80\xff@\xff\x02\xff\xd7\xfe\xa2\xfe\x7f\xfed\xfeV\xfeR\xfeU\xfew\xfe\x95\xfe\xad\xfe\xc4\xfe\xd2\xfe\xdc\xfe\xe3\xfe\xfd\xfe\x1c\xff9\xffR\xffs\xff\x8c\xff\xb3\xff\xf0\xff$\x00a\x00\x9a\x00\xd2\x00\xff\x00/\x01d\x01s\x01\x83\x01\x8a\x01z\x01l\x01H\x01\x17\x01\xd7\x00\x8c\x00O\x00\x1b\x00\xf1\xff\xcd\xff\x9e\xffr\xffV\xff<\xff,\xff\x19\xff\x08\xff\xf9\xfe\xda\xfe\xda\xfe\xef\xfe\xff\xfe\x16\xff2\xffH\xff_\xff\x80\xff\x99\xff\xa3\xff\xb2\xff\xb4\xff\xc5\xff\xe3\xff\x07\x004\x00O\x00j\x00\x80\x00\xa2\x00\xbb\x00\xd4\x00\xe8\x00\xff\x00\x18\x01,\x01:\x012\x01"\x01\r\x01\xef\x00\xd9\x00\xc3\x00\x9a\x00f\x004\x00\x05\x00\xe1\xff\xcc\xff\xb7\xff\x98\xff|\xffq\xff`\xff\\\xffW\xffA\xff)\xff\x0f\xff\x04\xff\xf5\xfe\xeb\xfe\xfa\xfe\xf0\xfe\xe8\xfe\xf3\xfe\x10\xff7\xff[\xff\x8b\xff\xb1\xff\xd0\xff\x04\x00,\x00U\x00\x8a\x00\xa4\x00\xc4\x00\xed\x00\xff\x00\r\x01\x14\x01\x1a\x01\x18\x01\x1c\x01"\x01\x1a\x01\x16\x01\xfa\x00\xc7\x00\x97\x00b\x00-\x00\xff\xff\xda\xff\xac\xff|\xff_\xffJ\xff=\xff0\xff\x1f\xff\x15\xff\x04\xff\xf2\xfe\xe7\xfe\xdc\xfe\xdd\xfe\xe5\xfe\xe9\xfe\x02\xff \xff6\xffP\xffw\xff\xa8\xff\xd6\xff\x15\x00P\x00\x8a\x00\xcd\x00\xf6\x00\x1b\x01?\x01E\x017\x01+\x01$\x01\x12\x01\x04\x01\x05\x01\xe1\x00\xc5\x00\xab\x00\x88\x00m\x00N\x00$\x00\x04\x00\xde\xff\xbd\xff\x98\xffy\xff^\xffR\xff?\xff>\xffC\xff \xff\x05\xff\xe6\xfe\xdb\xfe\xe2\xfe\xe9\xfe\x07\xff\x1a\xff>\xff_\xff\x8e\xff\xce\xff\x05\x00B\x00u\x00\xb0\x00\xe6\x00\x1b\x01M\x01x\x01\xa3\x01\xbf\x01\xcc\x01\xd4\x01\xd6\x01\xbc\x01\x92\x01m\x01?\x01#\x01\x0c\x01\xf8\x00\xe1\x00\xad\x00b\x00\x17\x00\xdf\xff\xaa\xff\x87\xffr\xffN\xff$\xff\xfc\xfe\xe0\xfe\xb9\xfe\x9d\xfex\xfeU\xfeF\xfe<\xfe=\xfeG\xfeN\xfed\xfe\x87\xfe\xa6\xfe\xc6\xfe\xed\xfe\x18\xffC\xfft\xff\xb2\xff\xf7\xffB\x00\x8a\x00\xc9\x00\xeb\x00\x10\x01A\x01b\x01\x82\x01\x9c\x01\xb2\x01\xb6\x01\xb2\x01\xc0\x01\xb2\x01\x9c\x01t\x01@\x01\x10\x01\xde\x00\xa8\x00q\x00:\x00\x03\x00\xc9\xff\x92\xffd\xff0\xff\x03\xff\xe4\xfe\xc7\xfe\xad\xfe\x93\xfe\x8b\xfe\x86\xfe|\xfe\x81\xfe\x80\xfe\x8b\xfe\x94\xfe\x97\xfe\xa8\xfe\xab\xfe\xc7\xfe\xe7\xfe\x0b\xff?\xffc\xff\x94\xff\xc0\xff\x02\x00L\x00\x84\x00\xc4\x00\xf4\x00#\x01;\x01C\x01F\x019\x01#\x01\x0c\x01\xf4\x00\xdc\x00\xc2\x00\xa2\x00\x8f\x00\x7f\x00p\x00b\x00Z\x00P\x003\x00\x1e\x00\x11\x00\xf6\xff\xe1\xff\xbc\xff\x9c\xffx\xffE\xff&\xff\xfe\xfe\xd7\xfe\xce\xfe\xc1\xfe\xd4\xfe\xf6\xfe\x11\xff:\xff`\xff\x8e\xff\xa8\xff\xd6\xff\x15\x00D\x00|\x00\xa5\x00\xcc\x00\xec\x00\xfc\x00\x13\x01"\x01,\x01#\x01\x17\x01\t\x01\xef\x00\xd6\x00\xbb\x00\xa5\x00\x98\x00\x89\x00i\x00A\x00\r\x00\xd7\xff\xa2\xffm\xffL\xff \xff\x06\xff\xf1\xfe\xd7\xfe\xca\xfe\xb8\xfe\xad\xfe\xa9\xfe\x9d\xfe\x9b\xfe\x99\xfe\xac\xfe\xcc\xfe\xde\xfe\xfb\xfe\x1b\xffG\xff\x7f\xff\xbd\xff\x0c\x00?\x00c\x00\x82\x00\x9b\x00\xba\x00\xe7\x00\n\x01#\x017\x01<\x015\x01-\x01%\x01\x12\x01\xfc\x00\xec\x00\xe3\x00\xd3\x00\xb8\x00\x9b\x00w\x00\\\x00@\x00\x16\x00\xf9\xff\xdc\xff\xb8\xff\x8d\xffj\xff;\xff$\xff\x03\xff\xe1\xfe\xd4\xfe\xc2\xfe\xbc\xfe\xbe\xfe\xd7\xfe\xf3\xfe\x16\xffA\xffb\xff\x94\xff\xc9\xff\xf8\xff\x19\x004\x00V\x00f\x00\x84\x00\xa8\x00\xc9\x00\xe3\x00\xe7\x00\xdb\x00\xc7\x00\xc4\x00\xc8\x00\xc7\x00\xb0\x00\x9e\x00\x80\x00j\x00b\x00R\x00H\x00:\x00(\x00\x0c\x00\xef\xff\xde\xff\xce\xff\xb2\xff\x9d\xff\x8f\xff}\xffl\xff[\xffO\xff?\xff:\xffB\xff0\xff8\xffN\xffX\xff\x7f\xff\xa6\xff\xd2\xff\xf9\xff\'\x00N\x00u\x00\xaa\x00\xdb\x00\x0c\x01#\x01@\x01Z\x01]\x01e\x01f\x01d\x01`\x01U\x016\x01\x18\x01\xfb\x00\xda\x00\xc0\x00\xa4\x00\x86\x00W\x00,\x00\x03\x00\xc9\xff\x94\xff^\xff4\xff\x04\xff\xdc\xfe\xb4\xfe\x87\xfek\xfeW\xfeK\xfeN\xfeN\xfeI\xfeY\xfe|\xfe\xa7\xfe\xe1\xfe\x1c\xffT\xff\x8f\xff\xc4\xff\xf7\xff+\x00^\x00\x8e\x00\xbd\x00\xeb\x00\x0b\x01 \x010\x016\x01-\x01%\x01\x1e\x01\x19\x01\x11\x01\xfc\x00\xee\x00\xd7\x00\xb4\x00\x90\x00k\x00D\x00\x1c\x00\x00\x00\xde\xff\xc4\xff\xa6\xff\x85\xffc\xff;\xff"\xff\x07\xff\xee\xfe\xd8\xfe\xcd\xfe\xbc\xfe\xba\xfe\xc5\xfe\xcc\xfe\xdd\xfe\xeb\xfe\xfd\xfe\x10\xff)\xffD\xffc\xff\x8b\xff\xb0\xff\xe7\xff\x1e\x00J\x00s\x00\x94\x00\xb6\x00\xd8\x00\xfb\x00\x1a\x01=\x01b\x01|\x01\x90\x01\x98\x01\x88\x01\x85\x01{\x01R\x01/\x01\x05\x01\xd6\x00\xad\x00\x8d\x00^\x00+\x00\xfb\xff\xc2\xff\x8e\xff^\xff"\xff\xea\xfe\xc7\xfe\xa7\xfe\x93\xfe\x92\xfe\x96\xfe\x94\xfe\xa3\xfe\xa4\xfe\xa6\xfe\xb6\xfe\xd2\xfe\xf4\xfe\x1a\xffD\xffv\xff\xa5\xff\xd3\xff\x07\x00?\x00\x7f\x00\xb6\x00\xdf\x00\x00\x01\x19\x01\'\x018\x01R\x01h\x01r\x01w\x01c\x01J\x014\x01\x17\x01\xff\x00\xe8\x00\xd3\x00\xbb\x00\x95\x00g\x000\x00\xfd\xff\xd7\xff\xb5\xff\x93\xffh\xffI\xff-\xff\n\xff\xfb\xfe\xed\xfe\xde\xfe\xd4\xfe\xb9\xfe\xa5\xfe\xb0\xfe\xbf\xfe\xdb\xfe\xeb\xfe\x07\xff \xffE\xffx\xff\x99\xff\xc0\xff\xdb\xff\xe7\xff\x0e\x008\x00[\x00\x85\x00\xb8\x00\xdc\x00\xee\x00\x08\x01\x18\x01\x18\x01!\x01\x19\x01\x19\x01\x16\x01\x07\x01\xe4\x00\xce\x00\xb2\x00\x84\x00U\x002\x00\x02\x00\xd3\xff\xac\xff\x7f\xffg\xffR\xff=\xff4\xff\x1b\xff\xfd\xfe\xea\xfe\xe4\xfe\xf4\xfe\xf8\xfe\x04\xff\x14\xff \xff(\xff3\xffK\xffk\xff\x8a\xff\xb7\xff\xe0\xff\x06\x00<\x00R\x00q\x00\xa5\x00\xce\x00\xee\x00\x07\x01%\x01&\x01*\x013\x01\x1e\x01\x16\x01\x06\x01\xef\x00\xd8\x00\xb9\x00\xb1\x00\xa0\x00\x8b\x00g\x00:\x00\x16\x00\xf2\xff\xc2\xff\x9c\xffs\xffH\xff\x1b\xff\xf8\xfe\xde\xfe\xd8\xfe\xda\xfe\xd8\xfe\xee\xfe\x11\xff=\xffn\xff\xa2\xff\xd0\xff\x05\x009\x00c\x00{\x00\x8d\x00\x90\x00\x92\x00\x9a\x00\xa7\x00\xb4\x00\xb6\x00\xac\x00\xaf\x00\xb5\x00\xb8\x00\xc3\x00\xca\x00\xce\x00\xce\x00\xc6\x00\xb7\x00\x9e\x00|\x00I\x00\x10\x00\xe6\xff\xb8\xff\x8a\xfff\xff@\xff*\xff\x16\xff\x0b\xff\x05\xff\xf9\xfe\xfa\xfe\xfa\xfe\xfa\xfe\x10\xff \xff.\xff9\xffJ\xff[\xffx\xff\x99\xff\xb0\xff\xd6\xff\xf6\xff\x13\x00C\x00r\x00\x9d\x00\xb7\x00\xb9\x00\xc5\x00\xcf\x00\xe1\x00\xe6\x00\xde\x00\xd2\x00\xad\x00\x9a\x00\x82\x00n\x00S\x003\x00\x1b\x00\x10\x00\x10\x00\x12\x00\x0e\x00\x08\x00\x00\x00\xed\xff\xde\xff\xc7\xff\xa8\xffz\xff=\xff\x14\xff\xf6\xfe\xce\xfe\xbd\xfe\xbc\xfe\xb7\xfe\xc1\xfe\xde\xfe\n\xff8\xffo\xff\x9b\xff\xcf\xff\x18\x00c\x00\xb1\x00\xf5\x007\x01f\x01\x87\x01\xa0\x01\x9d\x01\x8e\x01\x8e\x01t\x01Y\x01G\x01\x1b\x01\xf7\x00\xca\x00\x9b\x00\x87\x00n\x00K\x00.\x00 \x00\x12\x00\x06\x00\xfa\xff\xe5\xff\xc7\xff\xa2\xff\x83\xff`\xff,\xff\xf9\xfe\xbb\xfe{\xfeK\xfe \xfe\x07\xfe\x08\xfe\x1d\xfe6\xfeP\xfe\x80\xfe\xc0\xfe\xf8\xfe?\xff\x88\xff\xc5\xff\xfc\xff%\x00F\x00]\x00f\x00c\x00f\x00\x83\x00\x9a\x00\xb8\x00\xce\x00\xde\x00\xf4\x00\x06\x01\x16\x01\x1c\x01\x1e\x01\x1e\x01\x12\x01\x08\x01\xf7\x00\xcf\x00\xb5\x00\x97\x00z\x00l\x00N\x00 \x00\xf0\xff\xbd\xff\x86\xffP\xff)\xff\x10\xff\xf6\xfe\xe5\xfe\xd5\xfe\xcc\xfe\xb5\xfe\xb4\xfe\xbd\xfe\xc1\xfe\xea\xfe\x10\xff4\xff\\\xff\x87\xff\xa6\xff\xd1\xff\n\x00<\x00y\x00\xb3\x00\xd5\x00\xf4\x00!\x01:\x01L\x01b\x01b\x01n\x01o\x01i\x01]\x01D\x01/\x01\x07\x01\xe3\x00\xcb\x00\x9f\x00v\x006\x00\xf5\xff\xc7\xff\x8e\xff[\xff3\xff\x0c\xff\xeb\xfe\xd0\xfe\xbe\xfe\xa9\xfe\x9f\xfe\x98\xfe\x91\xfe\xa0\xfe\xaa\xfe\xc5\xfe\xf3\xfe\x0f\xff6\xffX\xff\x80\xff\xbc\xff\xf2\xff,\x00h\x00\x95\x00\xc5\x00\xf3\x00*\x01R\x01g\x01\x83\x01\x8b\x01\x89\x01\x81\x01d\x01;\x01$\x01\x15\x01\xfa\x00\xe0\x00\xa9\x00k\x00>\x00)\x00\x13\x00\x00\x00\xe4\xff\xa9\xffm\xff>\xff\x1f\xff\x19\xff\r\xff\x07\xff\x05\xff\xfe\xfe\xfa\xfe\x00\xff\xff\xfe\x06\xff\x1b\xff\x1d\xff/\xffM\xffa\xffw\xff\x8f\xff\xa9\xff\xc5\xff\xdc\xff\xef\xff\x06\x00%\x00A\x00h\x00\x87\x00\x97\x00\x9c\x00\x98\x00\x97\x00\xa7\x00\xc8\x00\xe5\x00\xf1\x00\xf6\x00\xe3\x00\xca\x00\xbd\x00\xa6\x00\x89\x00y\x00@\x00\x00\x00\xe4\xff\xc2\xff\x9e\xff\x92\xff}\xffg\xffk\xffk\xffi\xfff\xff\\\xffM\xffF\xffa\xff\x80\xff\x9a\xff\xb6\xff\xc5\xff\xbb\xff\xb4\xff\xb0\xff\xad\xff\xc7\xff\xf1\xff\x19\x00<\x00X\x00`\x00h\x00u\x00}\x00\x95\x00\xaf\x00\xc7\x00\xc8\x00\xae\x00\x9e\x00\x84\x00g\x00W\x006\x00\r\x00\xf1\xff\xcb\xff\xad\xff\xa4\xff\x92\xff\x7f\xffy\xffi\xffY\xffT\xffJ\xffD\xffN\xff[\xffe\xffs\xff{\xff\x89\xff\x97\xff\xa5\xff\xb1\xff\xbb\xff\xd0\xff\xe6\xff\xfc\xff\x12\x009\x00k\x00\x98\x00\xb5\x00\xce\x00\xde\x00\xf2\x00\x04\x01\x06\x01\x00\x01\x02\x01\x06\x01\xfd\x00\x03\x01\xf8\x00\xde\x00\xc7\x00\xa5\x00\x90\x00\x83\x00u\x00`\x00A\x00\x1c\x00\xe5\xff\xa7\xffa\xff \xff\xea\xfe\xb2\xfe~\xfef\xfeH\xfe(\xfe/\xfe>\xfeV\xfe\x92\xfe\xcb\xfe\xfa\xfe8\xffg\xff\x83\xff\xa7\xff\xd0\xff\xeb\xff\x03\x00\x1b\x00%\x00!\x00/\x00R\x00t\x00\xad\x00\xe9\x00\x1e\x01O\x01\x81\x01\xa7\x01\xbe\x01\xda\x01\xf0\x01\xf8\x01\xe8\x01\xb2\x01f\x01\x0f\x01\xb6\x00X\x00\xfb\xff\xb3\xffs\xff@\xff \xff\x06\xff\xfb\xfe\xf6\xfe\xf3\xfe\xf7\xfe\xff\xfe\xf9\xfe\xf2\xfe\xe7\xfe\xd8\xfe\xcd\xfe\xcd\xfe\xd4\xfe\xda\xfe\xed\xfe\n\xff\x1c\xff,\xffZ\xff\x87\xff\xb5\xff\xff\xffC\x00o\x00\xac\x00\xe2\x00\xf1\x00\x11\x014\x01C\x01N\x01^\x01X\x01L\x01T\x01A\x01$\x01\x18\x01\xff\x00\xd3\x00\xa9\x00|\x00G\x00\r\x00\xea\xff\xb6\xff\x84\xff`\xff4\xff\x1c\xff\x17\xff\x19\xff\x1a\xff\x16\xff\x0c\xff\xfc\xfe\xe9\xfe\xf1\xfe\xfd\xfe\r\xff+\xff2\xff4\xff9\xff:\xffG\xffi\xff\x99\xff\xd1\xff\x10\x00M\x00s\x00\xa4\x00\xdf\x00\x11\x018\x01g\x01\x80\x01\x89\x01\x8d\x01x\x01[\x01=\x01\x14\x01\xe4\x00\xc9\x00\xb0\x00\x8f\x00\x82\x00s\x00Q\x00,\x00\x08\x00\xee\xff\xd2\xff\xc8\xff\xb3\xff~\xffK\xff\x17\xff\xe7\xfe\xcd\xfe\xcc\xfe\xcd\xfe\xd6\xfe\xeb\xfe\xf7\xfe\xf6\xfe\x01\xff"\xff;\xffa\xff\x9e\xff\xcb\xff\xf1\xff!\x00@\x00r\x00\xa8\x00\xc7\x00\xeb\x00\xf9\x00\x05\x01\x1a\x01.\x01:\x01@\x01B\x012\x01!\x01\x0f\x01\xed\x00\xce\x00\xad\x00s\x00/\x00\xf0\xff\xb7\xff\x93\xffu\xffa\xffT\xffI\xffA\xff0\xff4\xff1\xff\x1d\xff\x14\xff\x1f\xff\x15\xff\x15\xff\x10\xff\xfe\xfe\r\xff\x15\xff$\xff1\xff:\xffP\xffl\xff\x9a\xff\xc8\xff\xee\xff\x0c\x00-\x00Z\x00y\x00\x9c\x00\xc3\x00\xd0\x00\xe0\x00\xec\x00\xe5\x00\xd2\x00\xb5\x00\x95\x00~\x00m\x00c\x00U\x00;\x00%\x00\x07\x00\xe7\xff\xce\xff\xbf\xff\xb6\xff\x9b\xffz\xff\\\xffA\xff4\xff4\xff0\xff#\xff/\xffA\xffH\xffV\xffe\xffm\xffx\xff\x8e\xff\xb0\xff\xd3\xff\xf1\xff\x0c\x00&\x00A\x00`\x00u\x00\x94\x00\xba\x00\xde\x00\xfc\x00\r\x01\x19\x01\x0e\x01\xfd\x00\xe7\x00\xce\x00\xbd\x00\xaa\x00\x8c\x00p\x00P\x00+\x00\n\x00\xeb\xff\xcb\xff\xad\xff\x97\xff\x80\xff\\\xffM\xff9\xff\x17\xff\x05\xff\xef\xfe\xd4\xfe\xd7\xfe\xd5\xfe\xda\xfe\xe8\xfe\x05\xff(\xffC\xff{\xff\xaf\xff\xe2\xff\x1b\x00J\x00\x80\x00\xaf\x00\xcf\x00\xeb\x00\x04\x01\x19\x010\x01S\x01c\x01p\x01y\x01i\x01[\x01F\x011\x01 \x01\x06\x01\xeb\x00\xd1\x00\xaf\x00v\x007\x00\xfe\xff\xc0\xff\x88\xffV\xff*\xff\x07\xff\xf1\xfe\xdf\xfe\xd0\xfe\xcd\xfe\xcf\xfe\xe0\xfe\xef\xfe\xfa\xfe\x15\xff)\xff1\xff3\xffM\xffb\xff\x87\xff\xb9\xff\xde\xff\x0f\x00;\x00X\x00y\x00\xaa\x00\xd2\x00\xf3\x00+\x01P\x01^\x01z\x01\x81\x01u\x01w\x01g\x01K\x013\x01\x1a\x01\xff\x00\xeb\x00\xc4\x00\x98\x00s\x00>\x00\x05\x00\xce\xff\x98\xffe\xff9\xff\x18\xff\xed\xfe\xc0\xfe\x9c\xfe\x84\xfeo\xfeY\xfeO\xfeH\xfe@\xfeW\xfe\x88\xfe\xb2\xfe\xef\xfe+\xffX\xff\x89\xff\xbc\xff\xe7\xff\x10\x00D\x00~\x00\xaf\x00\xe4\x00\x17\x01A\x01e\x01x\x01u\x01x\x01v\x01c\x01W\x01?\x01%\x01\r\x01\xea\x00\xbb\x00\x83\x00H\x00\x02\x00\xce\xff\xa9\xffs\xffL\xff\x12\xff'} message_type=websocket.receive session_id=v5_mph3j0z2

(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent> uv run uvicorn src.voice_commerce.main:app --reload
INFO:     Will watch for changes in these directories: ['P:\\AI_Empire\\Projects\\voice-commerce-agent']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [5316] using WatchFiles
INFO:     Started server process [11064]
INFO:     Waiting for application startup.
2026-03-24 06:28:59 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
INFO:     127.0.0.1:4568 - "GET /static/test_client.html HTTP/1.1" 200 OK
2026-03-24 06:29:00 [info     ] voice_handler_created          session_id=v5_wtjkhgz9
INFO:     127.0.0.1:1969 - "WebSocket /ws/voice?session_id=v5_wtjkhgz9" [accepted]
2026-03-24 06:29:00 [info     ] ws_connected                   client="Address(host='127.0.0.1', port=1969)" session_id=v5_wtjkhgz9
INFO:     connection open
2026-03-24 06:29:00 [debug    ] gemini_live_handler_created    model=gemini-2.5-flash-native-audio-preview-12-2025 voice=Charon
2026-03-24 06:29:00 [info     ] gemini_session_connecting      model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-24 06:29:01 [info     ] gemini_session_connected      
2026-03-24 06:29:03 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:03 [info     ] input_mode_switched_to_audio   session_id=v5_wtjkhgz9
2026-03-24 06:29:03 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:03 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:03 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:03 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:03 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:04 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:04 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:04 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:04 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:05 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:05 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:05 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:05 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:06 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:06 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:06 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:06 [debug    ] gemini_input_transcript        text=' <noise> ഹായ് <noise>'
2026-03-24 06:29:06 [info     ] user_speech_transcribed        session_id=v5_wtjkhgz9 text=' <noise> ഹായ് <noise>'
Warning: there are non-data parts in the response: ['text', 'thought'], returning concatenated data result from data parts, check out the non data parts for full response from model.
2026-03-24 06:29:07 [debug    ] gemini_output_transcript       text='Hello! How'
2026-03-24 06:29:07 [debug    ] gemini_output_transcript       text=' can I'
2026-03-24 06:29:08 [debug    ] gemini_output_transcript       text=' help'
Warning: there are non-text parts in the response: ['inline_data'], returning concatenated text result from text parts, check out the non text parts for full response from model.
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=46080
2026-03-24 06:29:08 [debug    ] gemini_output_transcript       text=' you today?'
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
INFO:     127.0.0.1:14239 - "GET /favicon.ico HTTP/1.1" 404 Not Found
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:09 [debug    ] turn_saved                     length=32 session=v5_wtjkhgz9
2026-03-24 06:29:14 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:14 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:14 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:14 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:14 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:14 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:20 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:20 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:20 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:20 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:20 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:20 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:20 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:20 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:20 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:21 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:21 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:21 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:21 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:22 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:22 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:22 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:22 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:23 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:23 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:23 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:23 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:23 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:23 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:23 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:23 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:23 [debug    ] gemini_sending_audio_chunk     bytes=8192
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 INFO:     Shutting down
INFO:     connection closed
2026-03-24 06:34:07 [info     ] gemini_session_closing        
2026-03-24 06:34:07 [info     ] gemini_session_closed_by_server reason='1000 None. '
2026-03-24 06:34:07 [info     ] gemini_session_closed         
2026-03-24 06:34:07 [info     ] ws_disconnected_normally       session_id=v5_wtjkhgz9 turns=2
2026-03-24 06:34:07 [info     ] ws_handler_finished            session_id=v5_wtjkhgz9 total_turns=2
INFO:     Waiting for application shutdown.
2026-03-24 06:34:07 [info     ] Application shutting down     
INFO:     Application shutdown complete.
INFO:     Finished server process [11064]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 INFO:     Started server process [1544]
INFO:     Waiting for application startup.
2026-03-24 06:34:11 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
2026-03-24 06:34:11 [info     ] Application shutting down     
INFO:     Application shutdown complete.
INFO:     Finished server process [1544]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "C:\Users\mahah\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\multiprocessing\spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mahah\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\multiprocessing\spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\__init__.py", line 2, in <module>
    from uvicorn.main import Server, main, run
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\main.py", line 32, in <module>
    from uvicorn.supervisors import ChangeReload, Multiprocess
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\supervisors\__init__.py", line 12, in <module>
    from uvicorn.supervisors.watchfilesreload import WatchFilesReload as ChangeReload
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\supervisors\watchfilesreload.py", line 7, in <module>
    from watchfiles import watch
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\watchfiles\__init__.py", line 2, in <module>
    from .main import Change, awatch, watch
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\watchfiles\main.py", line 9, in <module>
    import anyio
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\__init__.py", line 52, in <module>
    from ._core._streams import create_memory_object_stream as create_memory_object_stream
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\_core\_streams.py", line 7, in <module>
    from ..streams.memory import (
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1091, in get_code
  File "<frozen importlib._bootstrap_external>", line 1190, in get_data
KeyboardInterrupt
INFO:     Started server process [14720]
INFO:     Waiting for application startup.
2026-03-24 06:34:14 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
2026-03-24 06:34:14 [info     ] voice_handler_created          session_id=v5_wtjkhgz9
INFO:     127.0.0.1:6550 - "WebSocket /ws/voice?session_id=v5_wtjkhgz9" [accepted]
2026-03-24 06:34:14 [info     ] ws_connected                   client="Address(host='127.0.0.1', port=6550)" session_id=v5_wtjkhgz9
INFO:     connection open
2026-03-24 06:34:14 [debug    ] gemini_live_handler_created    model=gemini-2.5-flash-native-audio-preview-12-2025 voice=Charon
2026-03-24 06:34:14 [info     ] gemini_session_connecting      model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-24 06:34:15 [info     ] gemini_session_connected      
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 INFO:     Shutting down
INFO:     connection closed
2026-03-24 06:34:17 [info     ] gemini_session_closing        
2026-03-24 06:34:17 [info     ] gemini_session_closed_by_server reason='1000 None. '
2026-03-24 06:34:17 [info     ] gemini_session_closed_gracefully reason='1000 None. ' session=v5_wtjkhgz9
2026-03-24 06:34:17 [info     ] gemini_session_closed         
2026-03-24 06:34:17 [info     ] ws_disconnected_normally       session_id=v5_wtjkhgz9 turns=0
2026-03-24 06:34:17 [info     ] ws_handler_finished            session_id=v5_wtjkhgz9 total_turns=0
INFO:     Waiting for application shutdown.
2026-03-24 06:34:17 [info     ] Application shutting down     
INFO:     Application shutdown complete.
INFO:     Finished server process [14720]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "C:\Users\mahah\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\multiprocessing\spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mahah\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\multiprocessing\spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\__init__.py", line 2, in <module>
    from uvicorn.main import Server, main, run
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\main.py", line 32, in <module>
    from uvicorn.supervisors import ChangeReload, Multiprocess
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\supervisors\__init__.py", line 12, in <module>
    from uvicorn.supervisors.watchfilesreload import WatchFilesReload as ChangeReload
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\supervisors\watchfilesreload.py", line 7, in <module>
    from watchfiles import watch
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\watchfiles\__init__.py", line 2, in <module>
    from .main import Change, awatch, watch
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\watchfiles\main.py", line 9, in <module>
    import anyio
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\__init__.py", line 32, in <module>
    from ._core._sockets import TCPConnectable as TCPConnectable
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1091, in get_code
  File "<frozen importlib._bootstrap_external>", line 1190, in get_data
KeyboardInterrupt
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 INFO:     Started server process [18616]
INFO:     Waiting for application startup.
2026-03-24 06:34:21 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
2026-03-24 06:34:21 [info     ] Application shutting down     
INFO:     Application shutdown complete.
INFO:     Finished server process [18616]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "C:\Users\mahah\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\multiprocessing\spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mahah\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\multiprocessing\spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\__init__.py", line 2, in <module>
    from uvicorn.main import Server, main, run
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\main.py", line 32, in <module>
    from uvicorn.supervisors import ChangeReload, Multiprocess
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\supervisors\__init__.py", line 12, in <module>
    from uvicorn.supervisors.watchfilesreload import WatchFilesReload as ChangeReload
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\supervisors\watchfilesreload.py", line 7, in <module>
    from watchfiles import watch
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\watchfiles\__init__.py", line 2, in <module>
    from .main import Change, awatch, watch
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\watchfiles\main.py", line 9, in <module>
    import anyio
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\__init__.py", line 26, in <module>
    from ._core._fileio import AsyncFile as AsyncFile
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\_core\_fileio.py", line 27, in <module>
    from .. import to_thread
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\to_thread.py", line 14, in <module>
    from .abc import CapacityLimiter
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\abc\__init__.py", line 43, in <module>
    from .._core._synchronization import (
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\_core\_synchronization.py", line 13, in <module>
    from ._tasks import CancelScope
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\_core\_tasks.py", line 12, in <module>
    class _IgnoredTaskStatus(TaskStatus[object]):
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\_core\_tasks.py", line 12, in _IgnoredTaskStatus
    class _IgnoredTaskStatus(TaskStatus[object]):

KeyboardInterrupt
INFO:     Started server process [7424]
INFO:     Waiting for application startup.
2026-03-24 06:34:24 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
2026-03-24 06:34:24 [info     ] voice_handler_created          session_id=v5_wtjkhgz9
INFO:     127.0.0.1:13768 - "WebSocket /ws/voice?session_id=v5_wtjkhgz9" [accepted]
2026-03-24 06:34:24 [info     ] ws_connected                   client="Address(host='127.0.0.1', port=13768)" session_id=v5_wtjkhgz9
INFO:     connection open
2026-03-24 06:34:24 [debug    ] gemini_live_handler_created    model=gemini-2.5-flash-native-audio-preview-12-2025 voice=Charon
2026-03-24 06:34:24 [info     ] gemini_session_connecting      model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-24 06:34:25 [info     ] gemini_session_connected      
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 INFO:     Shutting down
INFO:     connection closed
2026-03-24 06:35:36 [info     ] gemini_session_closing        
2026-03-24 06:35:36 [info     ] gemini_session_closed_by_server reason='1000 None. '
2026-03-24 06:35:36 [info     ] gemini_session_closed_gracefully reason='1000 None. ' session=v5_wtjkhgz9
2026-03-24 06:35:36 [info     ] gemini_session_closed         
2026-03-24 06:35:36 [info     ] ws_disconnected_normally       session_id=v5_wtjkhgz9 turns=0
2026-03-24 06:35:36 [info     ] ws_handler_finished            session_id=v5_wtjkhgz9 total_turns=0
INFO:     Waiting for application shutdown.
2026-03-24 06:35:36 [info     ] Application shutting down     
INFO:     Application shutdown complete.
INFO:     Finished server process [7424]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 INFO:     Started server process [19220]
INFO:     Waiting for application startup.
2026-03-24 06:35:39 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
2026-03-24 06:35:39 [info     ] Application shutting down     
INFO:     Application shutdown complete.
INFO:     Finished server process [19220]
INFO:     Started server process [7820]
INFO:     Waiting for application startup.
2026-03-24 06:35:42 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
2026-03-24 06:35:42 [info     ] voice_handler_created          session_id=v5_wtjkhgz9
INFO:     127.0.0.1:9405 - "WebSocket /ws/voice?session_id=v5_wtjkhgz9" [accepted]
2026-03-24 06:35:42 [info     ] ws_connected                   client="Address(host='127.0.0.1', port=9405)" session_id=v5_wtjkhgz9
INFO:     connection open
2026-03-24 06:35:42 [debug    ] gemini_live_handler_created    model=gemini-2.5-flash-native-audio-preview-12-2025 voice=Charon
2026-03-24 06:35:42 [info     ] gemini_session_connecting      model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-24 06:35:42 [info     ] gemini_session_connected      


(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent> uv run uvicorn src.voice_commerce.main:app --reload
INFO:     Will watch for changes in these directories: ['P:\\AI_Empire\\Projects\\voice-commerce-agent']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [8608] using WatchFiles
INFO:     Started server process [14120]
INFO:     Waiting for application startup.
2026-03-24 06:55:06 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
2026-03-24 06:55:06 [info     ] voice_handler_created          session_id=v5_wtjkhgz9
INFO:     127.0.0.1:10217 - "WebSocket /ws/voice?session_id=v5_wtjkhgz9" [accepted]
2026-03-24 06:55:06 [info     ] ws_connected                   client="Address(host='127.0.0.1', port=10217)" session_id=v5_wtjkhgz9
INFO:     connection open
2026-03-24 06:55:06 [debug    ] gemini_live_handler_created    model=gemini-2.5-flash-native-audio-preview-12-2025 voice=Charon
2026-03-24 06:55:06 [info     ] gemini_session_connecting      model=gemini-2.5-flash-native-audio-preview-12-2025
INFO:     127.0.0.1:1358 - "GET /static/test_client.html HTTP/1.1" 304 Not Modified
INFO:     connection closed
2026-03-24 06:55:07 [info     ] voice_handler_created          session_id=v5_mph3j0z2
INFO:     127.0.0.1:9769 - "WebSocket /ws/voice?session_id=v5_mph3j0z2" [accepted]
2026-03-24 06:55:07 [info     ] ws_connected                   client="Address(host='127.0.0.1', port=9769)" session_id=v5_mph3j0z2
INFO:     connection open
2026-03-24 06:55:07 [debug    ] gemini_live_handler_created    model=gemini-2.5-flash-native-audio-preview-12-2025 voice=Charon
2026-03-24 06:55:07 [info     ] gemini_session_connecting      model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-24 06:55:07 [info     ] gemini_session_connected      
2026-03-24 06:55:07 [info     ] gemini_session_closing        
2026-03-24 06:55:07 [info     ] gemini_session_closed         
2026-03-24 06:55:07 [info     ] ws_disconnected_normally       session_id=v5_wtjkhgz9 turns=0
2026-03-24 06:55:07 [info     ] ws_handler_finished            session_id=v5_wtjkhgz9 total_turns=0
2026-03-24 06:55:07 [info     ] gemini_session_connected      
2026-03-24 06:55:16 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:16 [info     ] input_mode_switched_to_audio   session_id=v5_mph3j0z2
2026-03-24 06:55:16 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:16 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:16 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:16 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:16 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:17 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:17 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:17 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:17 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:18 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:18 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:18 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:18 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:18 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:18 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:21 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:21 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:21 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:21 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:21 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:21 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:21 [debug    ] gemini_input_transcript        text=' Hi, how are you?'
2026-03-24 06:55:21 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' Hi, how are you?'
2026-03-24 06:55:22 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:22 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:22 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:22 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:22 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:22 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:22 [debug    ] gemini_output_transcript       text="Hello! I'm"
2026-03-24 06:55:22 [debug    ] gemini_output_transcript       text=' good,'
2026-03-24 06:55:22 [debug    ] gemini_output_transcript       text=' thank'
Warning: there are non-text parts in the response: ['inline_data'], returning concatenated text result from text parts, check out the non text parts for full response from model.
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=46080
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_output_transcript       text=' you.'
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
INFO:     127.0.0.1:5328 - "GET /favicon.ico HTTP/1.1" 404 Not Found
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_output_transcript       text=' How can'
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_output_transcript       text=' I help'
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_output_transcript       text=' you today?'
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:25 [debug    ] turn_saved                     length=53 session=v5_mph3j0z2
2026-03-24 06:55:32 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:32 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:32 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:32 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:32 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:32 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:32 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:32 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:32 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:33 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:33 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:33 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:33 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:34 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:34 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:34 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:34 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:35 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:35 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:35 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:35 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:35 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:35 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:35 [debug    ] gemini_input_transcript        text=' find what about you.'
2026-03-24 06:55:35 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' find what about you.'
Warning: there are non-data parts in the response: ['text', 'thought'], returning concatenated data result from data parts, check out the non data parts for full response from model.
2026-03-24 06:55:37 [debug    ] gemini_output_transcript       text="I'm here"
2026-03-24 06:55:37 [debug    ] gemini_output_transcript       text=' to help'
2026-03-24 06:55:37 [debug    ] gemini_output_transcript       text=' you find'
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=46080
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_output_transcript       text=' everything'
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_output_transcript       text=' you need!'
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_output_transcript       text=' What are'
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_output_transcript       text=' you looking'
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_output_transcript       text=' for?'
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:40 [debug    ] turn_saved                     length=72 session=v5_mph3j0z2
2026-03-24 06:55:42 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:42 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:42 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:42 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:43 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:43 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:43 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:43 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:47 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:47 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:47 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:47 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:48 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:48 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:48 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:48 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:48 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:48 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:48 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:48 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:48 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:36 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:36 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:36 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:36 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:36 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:36 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:36 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:36 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:36 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:37 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:37 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:37 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:37 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:38 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:38 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:38 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:38 [debug    ] gemini_input_transcript        text=' <noise> Oi, oi.'
2026-03-24 06:58:38 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' <noise> Oi, oi.'
2026-03-24 06:58:38 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:38 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:38 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:38 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:38 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:38 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:39 [debug    ] gemini_output_transcript       text="I'm actually"
2026-03-24 06:58:39 [debug    ] gemini_output_transcript       text=' a voice'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=46080
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_output_transcript       text=' shopping'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_output_transcript       text=' assistant.'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_output_transcript       text=' How can'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_output_transcript       text=' I help'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_output_transcript       text=' you shop'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_output_transcript       text=' today?'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:43 [debug    ] turn_saved                     length=71 session=v5_mph3j0z2
2026-03-24 06:58:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:08 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:08 [debug    ] text_message_parsed_as_json    text='What are today'
2026-03-24 07:01:08 [info     ] text_message_received          preview='What are today' session_id=v5_mph3j0z2
2026-03-24 07:01:08 [debug    ] gemini_sending_text            text='What are today'
2026-03-24 07:01:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:44 [info     ] input_mode_switched_to_audio   session_id=v5_mph3j0z2
2026-03-24 07:01:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:49 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:49 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:49 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:49 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:50 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:50 [debug    ] gemini_input_transcript        text=' So'
2026-03-24 07:01:50 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' So'
2026-03-24 07:01:50 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:50 [debug    ] gemini_input_transcript        text=' are'
2026-03-24 07:01:50 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' are'
2026-03-24 07:01:50 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:50 [debug    ] gemini_input_transcript        text=' you'
2026-03-24 07:01:50 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' you'
2026-03-24 07:01:50 [debug    ] gemini_input_transcript        text=' li'
2026-03-24 07:01:50 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' li'
2026-03-24 07:01:50 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:51 [debug    ] gemini_input_transcript        text=stening
2026-03-24 07:01:51 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=stening
2026-03-24 07:01:51 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:51 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:51 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:51 [debug    ] gemini_input_transcript        text=?
2026-03-24 07:01:51 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=?
2026-03-24 07:01:51 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:51 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:51 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:51 [debug    ] gemini_input_transcript        text=' So'
2026-03-24 07:01:51 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' So'
2026-03-24 07:02:11 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:11 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:11 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:11 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:12 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:12 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:12 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:12 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:13 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:13 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:13 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:13 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:14 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:14 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:14 [debug    ] gemini_input_transcript        text=' Ste'
2026-03-24 07:02:14 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' Ste'
2026-03-24 07:02:14 [debug    ] gemini_input_transcript        text=el
2026-03-24 07:02:14 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=el
2026-03-24 07:02:14 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:14 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:15 [debug    ] gemini_input_transcript        text=' '
2026-03-24 07:02:15 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' '
2026-03-24 07:02:15 [debug    ] gemini_input_transcript        text=ste
2026-03-24 07:02:15 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=ste
2026-03-24 07:02:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:15 [debug    ] gemini_input_transcript        text=el
2026-03-24 07:02:15 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=el
2026-03-24 07:02:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:15 [debug    ] gemini_input_transcript        text=' recording.'
2026-03-24 07:02:15 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' recording.'
2026-03-24 07:02:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:16 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:16 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:16 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:16 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:16 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:16 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:32 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:32 [debug    ] text_message_parsed_as_json    text='This is a problem'
2026-03-24 07:02:32 [info     ] text_message_received          preview='This is a problem' session_id=v5_mph3j0z2
2026-03-24 07:02:32 [debug    ] gemini_sending_text            text='This is a problem'